# Failure-Aware Environmental Forecasting

Repository copy of the supplied Kaggle notebook. Saved execution outputs and Kaggle attachment metadata have been cleared. `RESUME_FROM=None` supports a fresh run; scientific modules and experiment parameters are unchanged. For a staged Kaggle run, set `END_PHASE` as described in `docs/REPRODUCIBILITY.md`.


# Failure-Aware Environmental Forecasting Under Missing, Faulty, and Shifted Sensor Data

**Target journal:** *Environment Systems and Decisions* (Springer Nature)  
**Notebook version:** ESD-v4 multi-site external-replication design  
**Compute:** CPU only; on Kaggle select **Accelerator: None**.

This notebook audits when one-step-ahead environmental forecasts, explanations, alarms, adaptation, uncertainty intervals, and monitoring decisions fail. It does **not** presume that a machine-learning system is robust. Failed baselines, unsupported reliability targets, and negative external results remain visible.

## Research questions

1. How do mechanism-specific missing-data processes change forecast error and model ranking?
2. How do sensor faults change average and tail error?
3. Does explanation stability deteriorate beyond clean-repeat permutation noise?
4. What false-alarm performance is statistically estimable, and does alarm adaptation add value beyond scheduled and budget-matched updating?
5. How do coverage, error triage, review capacity, and decision-cost assumptions interact under degradation?
6. What training, inference, memory, and qualified energy costs accompany the evidence?

Each RQ creates three vector-PDF figures and one table. The external phase evaluates all twelve Beijing stations with local and leave-one-station-out training and creates four additional artifacts.

## Phase 4 execution repair — 17 September 2026
Scientific configuration unchanged. Phase 6 scientific source is untouched. 
Phase 4 uses per-model prediction checkpoints, statistical checkpoints, visible IO/rendering progress, 
and immediate tracebacks without an automatic archive in error handlers. 
Use `CTX.snapshot(False)` to export a partial archive before ending a session.
Restart the kernel and Run All. Use `RESUME_FROM=None` initially because changed code invalidates old cache identities. 
Keep the working directory to reuse checkpoints on subsequent runs of this revision.
The uploaded saved run shows Phase 4 complete at 17:22:14 on September 14; the last saved operation is in Phase 5. 
These historical outputs were cleared so they cannot be confused with a new execution.
Validation: the reused Phase 4 repair previously passed full-count synthetic integration and numerical comparison tests; 
this notebook has been structurally checked. Full real-data execution remains unverified.


## Evidential boundaries

- Core Italian air-quality and Belgian appliance-energy periods are retrospective fixed-history audits.
- Public external datasets are reported as independent replication, not prospectively confirmed evidence. A Boolean setting cannot substitute for external preregistration.
- Measurements at origin time *t* predict *t+1*. Targets are never imputed. Preprocessing and model selection use development data only.
- Corruption replicas are repeated interventions on fixed histories, not independent sites. Beijing stations are the external generalisation unit.
- A zero alarm count is accompanied by an exact one-sided event-rate upper bound. It is never treated as proof that a rare false-alarm target is met.
- Adaptation is prediction-first and label-later. Alarm timing is compared with periodic and random budget-matched controls.
- Explanation degradation is paired against clean-repeat explainer noise.
- Decision costs and review capacities are sensitivity assumptions, not stakeholder preferences.
- Energy is reported only with its measurement provenance; estimated energy is not converted into carbon claims.
- Software QA confirms execution integrity, not publication readiness or journal acceptance.

## Kaggle execution plan

Use a standard Kaggle CPU notebook with Internet enabled, or attach the official UCI files. GPU acceleration does not benefit the scikit-learn models used here.

| Session | `END_PHASE` | Work |
|---|---:|---|
| 1 | 2 | Inputs, causal tests, temporal CV, clean models |
| 2 | 5 | Missingness, sensor faults, explanation contrasts |
| 3 | 8 | Detection/adaptation, uncertainty/decisions, compute |
| 4 | 9 | Cross-domain replication and all-station Beijing evaluation |
| 5 | 10 | Regenerate artifacts, evidence gates, manifest, final ZIP |

After a staged session, download the partial ZIP. In the next Kaggle session attach it as an input and set `RESUME_FROM` to that file. Run all cells from the top: compatible checkpoint tasks load; incomplete tasks recompute.

In [ ]:
!find /kaggle/input -name "*.zip"

In [ ]:
from pathlib import Path

root = Path('/kaggle/input')
manifests = list(root.rglob('run_identity.json'))

if manifests:
    for p in manifests:
        print("Checkpoint setting:")
        print(f"RESUME_FROM = {str(p.parent.parent)!r}")
else:
    print("No checkpoint manifest found. Available files:")
    for p in list(root.rglob('*'))[:80]:
        print(p)

In [ ]:
# USER SETTINGS — edit this cell only.
from pathlib import Path
import os,sys,json,subprocess,importlib,importlib.metadata,hashlib,zipfile

RUN_PROFILE='paper'                 # use 'pilot' only for a quick engineering test
START_PHASE=1
END_PHASE= 10                          # STAGED RUN: session 1 -> 5, then 8, 9, 10 (never all 10 in one 12h Kaggle session)
ACKNOWLEDGE_RETROSPECTIVE=True
RUN_EXTERNAL_CONFIRMATION=True      # retained name for cache compatibility; analysis is labelled replication
EXTERNAL_OUTCOMES_PREVIOUSLY_INSPECTED=True
ALLOW_DOWNLOAD=True
INSTALL_PDF_VALIDATOR=True
TRY_CODECARBON=False                # enable only if already available; avoids mutating Kaggle dependencies
THREADS=4
RESUME_FROM = None          # find one checkpoint .keep file under /kaggle/input; or supply its exact path

DATA_PATHS={
 # 'Air Quality':'/kaggle/input/airqualityuci/AirQualityUCI.csv',
 # 'Appliances Energy':'/kaggle/input/energydata-complete/energydata_complete.csv',
}
EXTERNAL_DATA_PATHS={
 # 'Beijing Aotizhongxin':'/kaggle/input/.../PRSA_Data_Aotizhongxin_20130301-20170228.csv',
 # 'Household Power':'/kaggle/input/.../household_power_consumption.txt',
}
BEIJING_MULTISITE_ROOT=None         # directory containing all 12 PRSA_Data_*.csv files

WORK_ROOT=Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()/'primna_esd_v4_work'
WORK_ROOT.mkdir(parents=True,exist_ok=True)
CODE_DIR=WORK_ROOT/'primna_esd_v4_source';CODE_DIR.mkdir(exist_ok=True)
for variable in ['OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS']:
    os.environ[variable]=str(THREADS)
print('Working directory:',WORK_ROOT,'| profile:',RUN_PROFILE,'| accelerator: CPU only')

In [ ]:
# Validate the scientific environment. Only PyMuPDF may be installed because it
# is used for final PDF inspection; CodeCarbon is never installed into Kaggle's base environment.
required={'numpy':'numpy','pandas':'pandas','scipy':'scipy','sklearn':'scikit-learn','matplotlib':'matplotlib',
          'joblib':'joblib','psutil':'psutil','threadpoolctl':'threadpoolctl'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing:raise RuntimeError('Missing core packages: '+', '.join(missing))
if INSTALL_PDF_VALIDATOR and importlib.util.find_spec('fitz') is None:
    subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','PyMuPDF>=1.24,<2'],check=True,timeout=300)
    importlib.invalidate_caches()
if importlib.util.find_spec('fitz') is None:raise RuntimeError('PyMuPDF is required for final PDF validation.')
from threadpoolctl import threadpool_limits
THREAD_LIMITER=threadpool_limits(limits=THREADS)
for module,package in required.items():print(package,importlib.metadata.version(package))
def write_module(name,source):
    compile(source,name,'exec');(CODE_DIR/name).write_text(source,encoding='utf-8');print('Prepared',name)

In [ ]:
def configuration(profile='paper'):
    c=dict(profile=profile,work_root=str(WORK_ROOT),data_paths=DATA_PATHS,external_data_paths=EXTERNAL_DATA_PATHS,
      beijing_multisite_root=BEIJING_MULTISITE_ROOT,allow_download=ALLOW_DOWNLOAD,allow_install=INSTALL_PDF_VALIDATOR,
      resume_root=RESUME_FROM,seed=20260911,threads=THREADS,coverage=.90,acceptance_target=.80,target_far=1.0,
      acknowledge_retrospective=ACKNOWLEDGE_RETROSPECTIVE,run_external_confirmation=RUN_EXTERNAL_CONFIRMATION,
      external_outcomes_previously_inspected=EXTERNAL_OUTCOMES_PREVIOUSLY_INSPECTED,
      heartbeat_seconds=60,snapshot_each_phase=False,start_phase=START_PHASE,end_phase=END_PHASE,
      cv_folds=4,bootstrap=2000,randomization_draws=20000,
      missing_levels=[0,.05,.10,.20,.30],fault_levels=[0,.025,.05,.10,.20],perturb_reps=40,
      explanation_reps=20,importance_repeats=9,explanation_rows=1200,
      detector_sims=500,drift_reps=60,adaptation_reps=30,
      adapt_window=768,adapt_cooldown=256,max_refits=4,min_update_labels=96,
      adapt_validation_labels=128,adapt_min_relative_gain=.02,adapt_tail_tolerance=.05,
      uncertainty_reps=40,decision_event_quantiles=[.80,.90,.95],decision_fn_cost_ratios=[1.,2.,5.,10.],
      decision_abstain_cost=.25,decision_abstain_costs=[.10,.25,.50,1.,2.],
      decision_review_capacities=[.05,.10,.20,.40],natural_missing_min_n=100,
      external_cv_folds=4,external_reps=30,multisite_train_cap=150000,multisite_stress_reps=20,
      cross_site_success_fraction=.75,compute_reps=15,benchmark_seconds=5.,
      try_codecarbon=TRY_CODECARBON,png_dpi=600)
    if profile=='pilot':
        c.update(cv_folds=2,bootstrap=120,randomization_draws=500,perturb_reps=2,
          explanation_reps=2,importance_repeats=2,explanation_rows=200,detector_sims=20,
          drift_reps=2,adaptation_reps=2,max_refits=2,uncertainty_reps=2,
          external_cv_folds=2,external_reps=2,multisite_train_cap=12000,multisite_stress_reps=2,
          compute_reps=2,benchmark_seconds=.25,heartbeat_seconds=20,png_dpi=180,threads=min(2,THREADS))
    return c
CFG=configuration(RUN_PROFILE)
assert 1<=START_PHASE<=END_PHASE<=10
print(json.dumps(CFG,indent=2))

### Bundled source: Runtime

In [ ]:
write_module('primna_esd_runtime.py','"""Auditable task cache, logging, statistics and artifact utilities."""\nfrom __future__ import annotations\nimport contextlib, datetime as dt, gc, hashlib, importlib.metadata, json, logging, os\nimport platform, shutil, sys, threading, time, traceback, uuid, zipfile\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport psutil\nfrom scipy import stats\n\nSCHEMA_VERSION = \'primna-esd-audit-v4.0\'\n\n\ndef jsonable(x):\n    if isinstance(x, dict): return {str(k): jsonable(v) for k,v in x.items()}\n    if isinstance(x, (list,tuple)): return [jsonable(v) for v in x]\n    if isinstance(x, (Path,pd.Timestamp,dt.datetime)): return str(x)\n    if isinstance(x,np.ndarray): return x.tolist()\n    if isinstance(x,np.generic): return x.item()\n    return x\n\n\ndef digest(x):\n    return hashlib.sha256(json.dumps(jsonable(x), sort_keys=True, default=str).encode()).hexdigest()\n\n\ndef filehash(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(2**20),b\'\'): h.update(b)\n    return h.hexdigest()\n\n\ndef atomic_json(path, value):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.\'+uuid.uuid4().hex+\'.tmp\')\n    tmp.write_text(json.dumps(jsonable(value),indent=2,default=str),encoding=\'utf-8\')\n    os.replace(tmp,path)\n\n\ndef seed(*parts): return int(digest(parts)[:8],16)\n\n\ndef environment():\n    result={\'python\':platform.python_version(),\'platform\':platform.platform()}\n    for name in [\'numpy\',\'pandas\',\'scipy\',\'scikit-learn\',\'joblib\',\'matplotlib\',\'psutil\',\'threadpoolctl\',\'codecarbon\']:\n        try: result[name]=importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError: result[name]=\'unavailable\'\n    return result\n\n\ndef resource_state(path=None):\n    """Small, failure-tolerant resource snapshot for progress and diagnostics."""\n    proc=psutil.Process();vm=psutil.virtual_memory()\n    state={\'rss_mb\':round(proc.memory_info().rss/2**20,1),\n           \'available_ram_mb\':round(vm.available/2**20,1),\n           \'ram_percent\':float(vm.percent),\'process_threads\':proc.num_threads(),\n           \'cpu_count\':psutil.cpu_count(logical=True),\'gpu\':\'not requested (CPU-only design)\'}\n    if path is not None:\n        disk=psutil.disk_usage(str(Path(path)))\n        state.update(disk_free_mb=round(disk.free/2**20,1),disk_percent=float(disk.percent))\n    return state\n\n\nclass Context:\n    def __init__(self, config, source_dir):\n        self.cfg=dict(config); self.source_dir=Path(source_dir)\n        sources={p.name:p.read_text() for p in self.source_dir.glob(\'primna_esd_*.py\')}\n        core={k:v for k,v in sources.items() if not k.endswith(\'_figures.py\')}\n        runtime_keys={\'start_phase\',\'end_phase\',\'resume_root\',\'work_root\',\'data_paths\',\'external_data_paths\',\'allow_download\',\n                      \'allow_install\',\'heartbeat_seconds\',\'snapshot_each_phase\',\'overwrite_failed_tasks\',\'png_dpi\'}\n        scientific={k:v for k,v in self.cfg.items() if k not in runtime_keys}\n        self.env=environment()\n        compute_env={k:v for k,v in self.env.items() if k!=\'platform\'}\n        self.key=digest({\'schema\':SCHEMA_VERSION,\'sources\':core,\'config\':scientific,\'environment\':compute_env})\n        self.root=Path(config[\'work_root\'])/(\'primna_esd_\'+config[\'profile\']+\'_\'+self.key[:12])\n        self.root.mkdir(parents=True,exist_ok=True)\n        self.cache=self.root/\'checkpoints\'; self.cache.mkdir(exist_ok=True)\n        for n in [\'logs\',\'source\',\'manifests\',\'data\',\'results\',\'figures_pdf\',\'figures_png\',\'tables\',\'plot_data\']:\n            (self.root/n).mkdir(exist_ok=True)\n        self.logger=logging.getLogger(\'primna_esd_\'+self.key[:12]); self.logger.setLevel(logging.INFO)\n        self.logger.handlers.clear()\n        fmt=logging.Formatter(\'%(asctime)s | %(message)s\')\n        sh=logging.StreamHandler(sys.stdout); sh.setFormatter(fmt); self.logger.addHandler(sh)\n        fh=logging.FileHandler(self.root/\'logs\'/\'execution.log\'); fh.setFormatter(fmt); self.logger.addHandler(fh)\n        self.results={}; self.task_events=[]; self.phase_status={};self._last_event_flush=0.\n        evaluation_status=\'retrospective core audit plus independent external replication; no software-generated confirmatory claim\'\n        atomic_json(self.root/\'manifests\'/\'run_identity.json\',{\'schema\':SCHEMA_VERSION,\'key\':self.key,\n            \'config\':self.cfg,\'environment\':self.env,\'source_sha256\':{k:hashlib.sha256(v.encode()).hexdigest() for k,v in sources.items()},\n            \'target_journal\':\'Environment Systems and Decisions (Springer Nature)\',\n            \'evaluation_status\':evaluation_status})\n        for name,s in sources.items(): (self.root/\'source\'/name).write_text(s)\n        atomic_json(self.root/\'source\'/\'config.json\',self.cfg)\n        self.restore(config.get(\'resume_root\'))\n        self.log(\'Run directory: %s | accelerator: CPU only\',self.root)\n\n    def log(self,msg,*args):\n        self.logger.info(msg,*args)\n        for handler in self.logger.handlers:\n            try:handler.flush()\n            except Exception:pass\n\n    def resources(self): return resource_state(self.root)\n\n    def record_task_event(self,event):\n        self.task_events.append(event)\n        # Append one small record instead of rewriting a growing manifest after\n        # each model. The conventional JSON manifest is flushed periodically\n        # and before snapshots; checkpoint metadata remains the commit marker.\n        with (self.root/\'logs\'/\'task_events.jsonl\').open(\'a\',encoding=\'utf-8\') as f:\n            f.write(json.dumps(jsonable(event),default=str)+\'\\n\')\n        if time.monotonic()-self._last_event_flush>=10:\n            atomic_json(self.root/\'manifests\'/\'task_events.json\',self.task_events)\n            self._last_event_flush=time.monotonic()\n\n    def failure(self,label):\n        # Preserve the active traceback immediately. Do not start a potentially\n        # multi-GB archive job while handling cancellation or another exception.\n        self.log(\'FAILED %s\\n%s\',label,traceback.format_exc())\n        try:\n            atomic_json(self.root/\'manifests\'/\'phase_status.json\',self.phase_status)\n            atomic_json(self.root/\'manifests\'/\'task_events.json\',self.task_events)\n        except Exception:\n            self.log(\'Failed to save failure status\\n%s\',traceback.format_exc())\n\n    @contextlib.contextmanager\n    def substage(self,label,**details):\n        started=time.perf_counter()\n        self.log(\'SUBSTAGE START %s | details=%s | resources=%s\',label,jsonable(details),self.resources())\n        try:\n            yield\n        except BaseException:\n            self.log(\'SUBSTAGE FAILED %s | %.1f s | details=%s\\n%s\',label,time.perf_counter()-started,\n                     jsonable(details),traceback.format_exc())\n            raise\n        else:\n            self.log(\'SUBSTAGE COMPLETE %s | %.1f s | resources=%s\',label,time.perf_counter()-started,self.resources())\n\n    def restore(self,root):\n        if not root: return\n        root=Path(root)\n        candidates=[root] if (root/\'manifests\'/\'run_identity.json\').exists() else list(root.rglob(\'run_identity.json\'))\n        candidates=[p.parent.parent if p.is_file() else p for p in candidates]\n        matched=[]\n        for p in candidates:\n            ident=json.loads((p/\'manifests\'/\'run_identity.json\').read_text())\n            if ident.get(\'key\')==self.key: matched.append(p)\n        if len(matched)!=1:\n            raise RuntimeError(\'Resume requires exactly one compatible ESD-v4 run. Legacy runs and changed code/config/environment are rejected. Found \'+str(len(matched)))\n        src=matched[0]\n        if src.resolve()==self.root.resolve(): return\n        manifest=src/\'manifests\'/\'output_checksums.csv\'\n        if not manifest.exists():raise RuntimeError(\'Resume archive lacks checksum manifest. Preserve it; do not treat as a trusted compatible snapshot.\')\n        entries=pd.read_csv(manifest)\n        for row in entries.itertuples():\n            p=src/row.path\n            if not p.is_file() or filehash(p)!=row.sha256:raise RuntimeError(\'Resume checksum mismatch: \'+str(p))\n            rel=Path(row.path)\n            if rel.parts[0] in {\'source\',\'logs\'} or rel.name in {\'run_identity.json\',\'task_events.json\'}:continue\n            dst=self.root/rel\n            if not dst.exists():dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst)\n        self.log(\'Restored verified compatible snapshot: %s\',src)\n\n    @contextlib.contextmanager\n    def heartbeat(self,label):\n        start=time.monotonic(); stop=threading.Event()\n        def pulse():\n            while not stop.wait(self.cfg.get(\'heartbeat_seconds\',60)):\n                self.log(\'RUNNING %s | elapsed %.1f min | resources=%s\',label,(time.monotonic()-start)/60,self.resources())\n        t=threading.Thread(target=pulse,daemon=True);t.start()\n        try: yield\n        finally: stop.set();t.join(timeout=2)\n\n    def task(self,stage,label,inputs,fn):\n        # Include cache validation, loading, serialization and hashing, not only fn().\n        with self.heartbeat(\'task lifecycle | \'+label):\n            return self._task_impl(stage,label,inputs,fn)\n\n    def _task_impl(self,stage,label,inputs,fn):\n        key=digest({\'run\':self.key,\'stage\':stage,\'label\':label,\'inputs\':inputs})\n        stem=stage+\'_\'+key[:24]; path=self.cache/(stem+\'.joblib\'); meta=self.cache/(stem+\'.json\')\n        expected={\'task_key\':key,\'run_key\':self.key,\'label\':label,\'stage\':stage}\n        if path.exists() or meta.exists():\n            reason=None\n            try:\n                if not (path.exists() and meta.exists()):reason=\'incomplete checkpoint pair\'\n                else:\n                    m=json.loads(meta.read_text())\n                    if not all(m.get(k)==v for k,v in expected.items()):reason=\'identity mismatch\'\n                    elif m.get(\'sha256\')!=filehash(path):reason=\'checksum mismatch\'\n                    else:\n                        self.log(\'LOAD %s\',label)\n                        # Load only checkpoints created under this exact run identity.\n                        obj=joblib.load(path); self.record_task_event(dict(expected,status=\'loaded\'))\n                        return obj\n            except Exception as exc:\n                reason=\'metadata/load failure: \'+type(exc).__name__+\': \'+str(exc)\n            quarantine=self.cache/\'quarantine\';quarantine.mkdir(exist_ok=True)\n            stamp=dt.datetime.now(dt.timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n            moved=[]\n            for bad in (path,meta):\n                if bad.exists():\n                    target=quarantine/(bad.name+\'.\'+stamp+\'.invalid\');os.replace(bad,target);moved.append(str(target.name))\n            self.log(\'CACHE REBUILD %s | reason=%s | quarantined=%s\',label,reason,moved)\n        # Metadata is a commit marker: an interrupted half-write is never loaded.\n        self.log(\'START %s\',label)\n        started=time.perf_counter(); temp=path.with_suffix(\'.\'+uuid.uuid4().hex+\'.tmp\')\n        try:\n            obj=fn()\n            self.log(\'SERIALIZE START %s\',label)\n            joblib.dump(obj,temp,compress=3)\n            os.replace(temp,path)\n            atomic_json(meta,dict(expected,sha256=filehash(path),seconds=time.perf_counter()-started,completed_utc=dt.datetime.now(dt.timezone.utc).isoformat()))\n            self.log(\'SAVED %s | %.1f s\',label,time.perf_counter()-started)\n            self.record_task_event(dict(expected,status=\'computed\'))\n            return obj\n        except BaseException:\n            self.log(\'FAILED/INTERRUPTED %s\\n%s\',label,traceback.format_exc())\n            if temp.exists(): temp.unlink()\n            raise\n\n    def table(self,name,df):\n        if not isinstance(df,pd.DataFrame): df=pd.DataFrame(df)\n        p=self.root/\'results\'/(name+\'.csv\');p.parent.mkdir(parents=True,exist_ok=True)\n        tmp=p.with_name(p.name+\'.\'+uuid.uuid4().hex+\'.tmp\')\n        df.to_csv(tmp,index=False,float_format=\'%.12g\');os.replace(tmp,p)\n        return df\n\n    def snapshot(self,complete=False):\n        # Print only: writing to execution.log while checksumming it invalidates\n        # the archive. This heartbeat deliberately never uses self.log().\n        stop=threading.Event();started=time.monotonic()\n        def pulse():\n            while not stop.wait(self.cfg.get(\'heartbeat_seconds\',60)):\n                print(f\'{dt.datetime.now(dt.timezone.utc).isoformat()} SNAPSHOT RUNNING | \'\n                      f\'{time.monotonic()-started:.1f} s\',flush=True)\n        thread=threading.Thread(target=pulse,daemon=True);thread.start()\n        print(\'SNAPSHOT START: inventory, hashing, compression, verification\',flush=True)\n        try:return self._snapshot_impl(complete)\n        finally:stop.set();thread.join(timeout=2)\n\n    def _snapshot_impl(self,complete=False):\n        atomic_json(self.root/\'manifests\'/\'task_events.json\',self.task_events)\n        atomic_json(self.root/\'manifests\'/\'phase_status.json\',self.phase_status)\n        status={\'complete\':bool(complete),\'run_key\':self.key,\'profile\':self.cfg[\'profile\'],\n                \'evaluation_status\':json.loads((self.root/\'manifests\'/\'run_identity.json\').read_text())[\'evaluation_status\'],\n                \'phases\':self.phase_status}\n        atomic_json(self.root/\'RUN_STATUS.json\',status)\n        publication_dirs={\'source\',\'results\',\'figures_pdf\',\'figures_png\',\'plot_data\',\'tables\',\'manifests\'}\n        top_files={\'README_RESULTS.md\',\'FIGURE_TABLE_CAPTIONS.md\',\'RUN_STATUS.json\'}\n        def included(p):\n            rel=p.relative_to(self.root)\n            if not p.is_file() or p.name==\'output_checksums.csv\' or p.name.endswith(\'.tmp\') or p.name in {\'.DS_Store\'}:return False\n            if not complete:return True\n            return (rel.parts[0] in publication_dirs or str(rel) in top_files)\n        payloads=[p for p in sorted(self.root.rglob(\'*\')) if included(p)]\n        records=[]\n        for i,p in enumerate(payloads):\n            if i%100==0:print(f\'SNAPSHOT HASH {i+1}/{len(payloads)} | {p.name}\',flush=True)\n            records.append({\'path\':str(p.relative_to(self.root)),\'bytes\':p.stat().st_size,\'sha256\':filehash(p)})\n        pd.DataFrame(records).to_csv(self.root/\'manifests\'/\'output_checksums.csv\',index=False)\n        suffix=\'Outputs\' if complete else \'Partial\'\n        dest=self.root.parent/f\'Environment_Systems_and_Decisions_Failure_Aware_Audit_v4_{suffix}.zip\'\n        tmp=dest.with_suffix(\'.zip.tmp\')\n        with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=3) as z:\n            for i,p in enumerate(payloads+[self.root/\'manifests\'/\'output_checksums.csv\']):\n                if i%100==0:print(f\'SNAPSHOT ZIP {i+1}/{len(payloads)+1} | {p.name}\',flush=True)\n                z.write(p,str(Path(self.root.name)/p.relative_to(self.root)))\n        os.replace(tmp,dest)\n        # Validate the delivered archive, not merely the source directory. The\n        # checksum manifest intentionally excludes itself to avoid recursion.\n        expected={str(Path(self.root.name)/r[\'path\']):r for r in records}\n        with zipfile.ZipFile(dest) as z:\n            names=set(z.namelist())\n            required=set(expected)|{str(Path(self.root.name)/\'manifests\'/\'output_checksums.csv\')}\n            missing=required-names\n            if missing:raise RuntimeError(\'ZIP validation failed; missing entries: \'+str(sorted(missing)[:10]))\n            for i,(name,row) in enumerate(expected.items()):\n                if i%100==0:print(f\'SNAPSHOT VERIFY {i+1}/{len(expected)}\',flush=True)\n                h=hashlib.sha256()\n                with z.open(name) as stream:\n                    for chunk in iter(lambda:stream.read(2**20),b\'\'):h.update(chunk)\n                if h.hexdigest()!=row[\'sha256\']:\n                    raise RuntimeError(\'ZIP checksum mismatch: \'+name)\n        print(f\'{suffix} ZIP: {dest}\', flush=True) # Do not mutate the just-hashed execution log.\n        return dest\n\n\ndef scored(y,p):\n    y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float);mask=np.isfinite(y)\n    if not np.all(np.isfinite(p[mask])): raise ValueError(\'Nonfinite predictions on observed targets; no silent denominator reduction.\')\n    return y[mask],p[mask],mask\n\n\ndef metrics(y,p):\n    y,p,_=scored(y,p);e=y-p\n    den=np.sum((y-y.mean())**2)\n    return {\'n\':len(y),\'RMSE\':float(np.sqrt(np.mean(e*e))),\'MAE\':float(np.mean(abs(e))),\n            \'R2\':float(1-np.sum(e*e)/den) if den>0 else np.nan,\'AE_q95\':float(np.quantile(abs(e),.95))}\n\n\ndef block_sums(values,valid,block):\n    a=np.asarray(values); a=np.where(valid,a,0)\n    starts=np.arange(0,len(a),block)\n    return np.add.reduceat(a,starts),np.add.reduceat(np.asarray(valid,dtype=int),starts)\n\n\ndef paired_block_summary(y,predictions,clean,block,B,rng):\n    """Two-level resampling: corruption replicates and nonoverlapping clock-time blocks.\n    Estimand is sqrt(mean squared error over replicas and observed timestamps), not\n    a sample of independent datasets. All models share the same bootstrap draws.\n    predictions: (replicate, time, model); clean: (time, model).\n    """\n    y=np.asarray(y);P=np.asarray(predictions);C=np.asarray(clean)\n    valid=np.isfinite(y);counts=block_sums(np.zeros(len(y)),valid,block)[1]\n    sums=np.array([[block_sums((p[:,j]-y)**2,valid,block)[0] for j in range(P.shape[2])] for p in P])\n    cs=np.array([block_sums((C[:,j]-y)**2,valid,block)[0] for j in range(C.shape[1])])\n    point=np.sqrt(sums.sum(axis=2).mean(axis=0)/counts.sum());cpoint=np.sqrt(cs.sum(axis=1)/counts.sum())\n    boot=np.empty((B,P.shape[2]));rel=np.empty_like(boot)\n    for b in range(B):\n        ri=rng.integers(0,len(P),len(P));bi=rng.integers(0,len(counts),len(counts));den=counts[bi].sum()\n        if den==0: boot[b]=np.nan;rel[b]=np.nan;continue\n        boot[b]=np.sqrt(sums[ri][:,:,bi].sum(axis=2).mean(axis=0)/den)\n        cr=np.sqrt(cs[:,bi].sum(axis=1)/den)\n        rel[b]=boot[b]/np.maximum(cr,1e-12)-1\n    return point,cpoint,boot,rel,sums,counts\n\n\ndef scalar_block_ci(values,block,B,rng):\n    a=np.asarray(values,dtype=float);v=np.isfinite(a);ss,nn=block_sums(a,v,block)\n    draws=[]\n    for _ in range(B):\n        i=rng.integers(0,len(ss),len(ss));d=nn[i].sum()\n        draws.append(ss[i].sum()/d if d else np.nan)\n    return float(np.nanmean(a)),*np.nanquantile(draws,[.025,.975])\n\n\ndef ci(x):\n    x=np.asarray(x,dtype=float);x=x[np.isfinite(x)]\n    return tuple(np.quantile(x,[.025,.975])) if len(x) else (np.nan,np.nan)\n\n\ndef holm(p):\n    p=np.asarray(p,dtype=float);out=np.full(len(p),np.nan);valid=np.flatnonzero(np.isfinite(p))\n    order=valid[np.argsort(p[valid])];m=len(order)\n    if m:out[order]=np.minimum(1,np.maximum.accumulate(p[order]*(m-np.arange(m))))\n    return out\n\n\ndef paired_sign_test(block_differences,B,rng):\n    d=np.asarray(block_differences,dtype=float);d=d[np.isfinite(d)]\n    if not len(d):return np.nan\n    observed=abs(d.sum());null=np.abs((rng.choice([-1.,1.],size=(B,len(d)))*d).sum(axis=1))\n    return float((1+(null>=observed-1e-12).sum())/(B+1))\n')


### Bundled source: Data

In [ ]:
write_module('primna_esd_data.py','"""Raw inputs, frozen historical date boundaries, causal features and fault generators."""\nfrom __future__ import annotations\nimport io, json, math, urllib.request, zipfile\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom primna_esd_runtime import digest,filehash,atomic_json,seed\n\nDATASETS=[\'Air Quality\',\'Appliances Energy\']\nSPECS={\n \'Air Quality\':{\'target\':\'CO(GT)\',\'frequency\':\'1h\',\'period\':24,\'lags\':[1,2,3,6,12,23,24],\n   \'windows\':[3,6,24],\'exclude\':[\'NMHC(GT)\'],\'filename\':\'AirQualityUCI.csv\',\'unit\':\'mg/m3\',\n   \'url\':\'https://archive.ics.uci.edu/static/public/360/air+quality.zip\',\n   \'ranges\':{\'development\':[\'2004-03-11 18:00\',\'2004-11-17 09:00\'],\n             \'calibration\':[\'2004-11-18 12:00\',\'2004-12-20 14:00\'],\n             \'policy\':[\'2004-12-21 16:00\',\'2005-01-28 23:00\'],\n             \'test\':[\'2005-01-30 01:00\',\'2005-04-04 13:00\']}},\n \'Appliances Energy\':{\'target\':\'Appliances\',\'frequency\':\'10min\',\'period\':144,\n   \'lags\':[1,2,3,6,12,18,36,143,144],\'windows\':[6,18,144],\'exclude\':[\'rv1\',\'rv2\'],\n   \'filename\':\'energydata_complete.csv\',\'unit\':\'Wh per 10 min\',\n   \'url\':\'https://archive.ics.uci.edu/static/public/374/appliances+energy+prediction.zip\',\n   \'ranges\':{\'development\':[\'2016-01-12 17:00\',\'2016-04-01 12:20\'],\n             \'calibration\':[\'2016-04-02 12:40\',\'2016-04-15 19:40\'],\n             \'policy\':[\'2016-04-16 20:00\',\'2016-04-30 03:00\'],\n             \'test\':[\'2016-05-01 03:20\',\'2016-05-27 17:50\']}}\n}\nMISSING=[\'MCAR\',\'Block outage\',\'Channel dropout\',\'MAR\',\'MNAR sensitivity\']\nFAULTS=[\'Gaussian\',\'Multiplicative\',\'Calibration bias\',\'Spikes\',\'Clipping\',\'Stuck-at\']\nDRIFTS=[\'Abrupt mean\',\'Gradual mean\',\'Variance\',\'Recurring\']\n\n\ndef input_path(ctx,ds):\n    spec=SPECS[ds];explicit=ctx.cfg.get(\'data_paths\',{}).get(ds)\n    if explicit:\n        p=Path(explicit)\n        if not p.exists():raise FileNotFoundError(p)\n        return p\n    roots=[Path(\'/kaggle/input\'),ctx.root/\'data\']\n    matches=[]\n    for root in roots:\n        if root.exists(): matches+=sorted(root.rglob(spec[\'filename\']))\n    if matches:\n        hashes={filehash(p) for p in matches}\n        if len(hashes)>1:raise RuntimeError(f\'Multiple differing {spec["filename"]} files: set data_paths explicitly.\')\n        return matches[0]\n    if not ctx.cfg[\'allow_download\']:raise FileNotFoundError(\'Attach \'+spec[\'filename\']+\' or enable Internet.\')\n    ctx.log(\'Downloading official UCI archive: %s\',spec[\'url\'])\n    archive=ctx.root/\'data\'/(\'uci_\'+str(spec[\'period\'])+\'.zip\')\n    try:\n        with urllib.request.urlopen(spec[\'url\'],timeout=120) as r: archive.write_bytes(r.read())\n        with zipfile.ZipFile(archive) as z:\n            wanted=[n for n in z.namelist() if Path(n).name==spec[\'filename\']]\n            if len(wanted)!=1:raise ValueError(\'Expected one \'+spec[\'filename\'])\n            dest=ctx.root/\'data\'/spec[\'filename\'];dest.write_bytes(z.read(wanted[0]));return dest\n    except Exception as exc:\n        raise RuntimeError(\'Official download unavailable. Attach the UCI CSV using Add Input; no synthetic fallback.\') from exc\n\n\ndef parse_raw(path,ds):\n    spec=SPECS[ds]\n    if ds==\'Air Quality\':\n        df=pd.read_csv(path,sep=\';\',decimal=\',\').dropna(how=\'all\')\n        df=df.loc[:,~df.columns.str.startswith(\'Unnamed\')]\n        t=pd.to_datetime(df.pop(\'Date\').astype(str)+\' \'+df.pop(\'Time\').astype(str),format=\'%d/%m/%Y %H.%M.%S\',errors=\'coerce\')\n        df=df.apply(pd.to_numeric,errors=\'raise\').replace(-200,np.nan)\n    else:\n        df=pd.read_csv(path);t=pd.to_datetime(df.pop(\'date\'),errors=\'coerce\')\n        df=df.apply(pd.to_numeric,errors=\'raise\')\n    if t.isna().any():raise ValueError(\'Invalid timestamps: no silent row removal.\')\n    df.index=pd.DatetimeIndex(t);df=df.sort_index()\n    # Official files have no duplicate clock origins; reject conflicts rather than keep-last.\n    if df.index.duplicated().any():raise ValueError(\'Duplicate timestamps require explicit adjudication before forecasting.\')\n    grid=pd.date_range(df.index[0],df.index[-1],freq=spec[\'frequency\'])\n    if not df.index.isin(grid).all():raise ValueError(\'Timestamps do not align with declared acquisition grid.\')\n    inserted=len(grid)-len(df);df=df.reindex(grid)\n    if np.isinf(df.to_numpy()).any():raise ValueError(\'Infinite input values.\')\n    return df,inserted\n\n\ndef features(raw,spec,columns):\n    """Features at origin t use only raw measurements at or before t; target is t+1."""\n    b=raw[columns];parts=[b.add_prefix(\'cur::\')]\n    for lag in spec[\'lags\']:parts.append(b.shift(lag).add_prefix(f\'lag{lag}::\'))\n    dc=[c for c in columns if c==spec[\'target\'] or c.startswith(\'PT08\') or c in {\'T\',\'RH\',\'lights\',\'T_out\',\'RH_out\'}]\n    parts.append(b[dc].diff().add_prefix(\'diff1::\'))\n    for w in spec[\'windows\']:\n        r=b[spec[\'target\']].rolling(w,min_periods=max(2,w//3))\n        parts.append(pd.DataFrame({f\'roll{w}_{op}::{spec["target"]}\':getattr(r,op)() for op in [\'mean\',\'std\',\'min\',\'max\']}))\n    ts=raw.index;tm={};h=ts.hour+ts.minute/60\n    for k,v,p in [(\'hour\',h,24),(\'dow\',ts.dayofweek,7),(\'doy\',ts.dayofyear,365.25)]:\n        tm[\'time::\'+k+\'_sin\']=np.sin(2*np.pi*v/p);tm[\'time::\'+k+\'_cos\']=np.cos(2*np.pi*v/p)\n    parts.append(pd.DataFrame(tm,index=ts));X=pd.concat(parts,axis=1)\n    assert X.columns.is_unique\n    assert sum(c.startswith(\'lag\') for c in X)==len(columns)*len(spec[\'lags\'])\n    return X\n\n\ndef make_bundle(ctx,ds,path):\n    raw,inserted=parse_raw(path,ds);spec=SPECS[ds]\n    columns=[c for c in raw if c not in spec[\'exclude\']]\n    X=features(raw,spec,columns);y=raw[spec[\'target\']].shift(-1)\n    naive=pd.DataFrame({\'Persistence\':raw[spec[\'target\']],\n                       \'Seasonal naive\':raw[spec[\'target\']].shift(spec[\'period\']-1)},index=raw.index)\n    ranges={k:(pd.Timestamp(v[0]),pd.Timestamp(v[1])) for k,v in spec[\'ranges\'].items()}\n    splits={k:raw.index[(raw.index>=start)&(raw.index<=end)] for k,(start,end) in ranges.items()}\n    if any(len(i)<50 for i in splits.values()):raise ValueError(\'Input does not cover frozen historical intervals.\')\n    delta=pd.Timedelta(spec[\'frequency\']);history=max(spec[\'lags\']+spec[\'windows\'])\n    for a,b in zip(list(splits)[:-1],list(splits)[1:]):\n        assert splits[a][-1]+delta < splits[b][0]-history*delta\n    dev=raw.loc[:ranges[\'development\'][1],columns]\n    med=dev.median();std=dev.std().replace(0,np.nan);iqr=(dev.quantile(.75)-dev.quantile(.25)).replace(0,np.nan)\n    st=pd.DataFrame({\'median\':med.fillna(0),\'std\':std.fillna(iqr).fillna(1),\n                     \'iqr\':iqr.fillna(std).fillna(1)})\n    groups={}\n    for col in X:\n        group=\'Calendar\' if col.startswith(\'time::\') else col.split(\'::\')[-1]\n        groups.setdefault(group,[]).append(col)\n    b={\'dataset\':ds,\'raw\':raw,\'X\':X,\'y\':y,\'naive\':naive,\'spec\':spec,\'columns\':columns,\'stats\':st,\n       \'groups\':groups,\'splits\':splits,\'input_sha256\':filehash(path),\'inserted_grid_rows\':inserted}\n    b[\'data_key\']=digest({\'file\':b[\'input_sha256\'],\'ranges\':spec[\'ranges\'],\'features\':list(X),\'columns\':columns})\n    return b\n\n\ndef acquire(ctx):\n    bundles={};audit=[];splits=[]\n    for ds in DATASETS:\n        path=input_path(ctx,ds);h=filehash(path)\n        bundles[ds]=ctx.task(\'P1\',ds+\' raw/features\',{\'input\':h,\'spec\':SPECS[ds]},lambda ds=ds,path=path:make_bundle(ctx,ds,path))\n        b=bundles[ds]\n        # Cache full raw inputs as CSV alongside the canonical parsed time series.\n        dest=ctx.root/\'data\'/SPECS[ds][\'filename\']\n        if path.resolve()!=dest.resolve():\n            import shutil;shutil.copy2(path,dest)\n        for split,idx in b[\'splits\'].items():\n            splits.append({\'dataset\':ds,\'split\':split,\'start\':idx[0],\'end\':idx[-1],\n                           \'clock_origins\':len(idx),\'observed_future_targets\':int(b[\'y\'].loc[idx].notna().sum()),\n                           \'evaluation_status\':\'historical inspected cohort; retrospective core audit\'})\n        for col in b[\'raw\']:\n            audit.append({\'dataset\':ds,\'variable\':col,\'n_rows\':len(b[\'raw\']),\n              \'missing_fraction\':b[\'raw\'][col].isna().mean(),\'included\':col in b[\'columns\'],\n              \'input_sha256\':h,\'inserted_grid_rows\':b[\'inserted_grid_rows\']})\n    ctx.table(\'P1_data_audit\',audit);ctx.table(\'P1_frozen_split_manifest\',splits)\n    return bundles\n\n\ndef cv_folds(bundle,n_splits):\n    """Equally spaced CLOCK folds before dropping unavailable labels; full temporal gap."""\n    idx=bundle[\'splits\'][\'development\'];gap=max(bundle[\'spec\'][\'lags\']+bundle[\'spec\'][\'windows\'])+1\n    fold_n=len(idx)//(n_splits+1)\n    for f in range(n_splits):\n        vstart=(f+1)*fold_n\n        tr=idx[:vstart-gap];va=idx[vstart:vstart+fold_n if f<n_splits-1 else len(idx)]\n        tr=tr[bundle[\'y\'].loc[tr].notna()];va=va[bundle[\'y\'].loc[va].notna()]\n        if len(tr)<50 or len(va)<20:raise ValueError(\'Insufficient train/validation labels.\')\n        assert tr[-1]+pd.Timedelta(bundle[\'spec\'][\'frequency\']) < va[0]\n        yield f,tr,va\n\n\ndef fit_risk_intercept(z,rate):\n    z=np.clip(np.nan_to_num(z),-8,8);lo=-30.;hi=30.\n    for _ in range(70):\n        mid=(lo+hi)/2;p=1/(1+np.exp(-np.clip(z+mid,-40,40)))\n        if p.mean()>rate:hi=mid\n        else:lo=mid\n    return (lo+hi)/2\n\n\ndef corrupt(bundle,kind,mechanism,severity,rep,scope=\'test\'):\n    """Perturb every clock time in the evaluation interval, including unscored origins.\n    Existing missing measurements stay missing. Ground-truth y is never modified.\n    Nominal severity is mechanism-specific, not a universal physical error percentage.\n    """\n    raw=bundle[\'raw\'].copy();idx=bundle[\'splits\'][scope];cols=bundle[\'columns\'];st=bundle[\'stats\']\n    block=raw.loc[idx,cols];arr=block.to_numpy().copy();original=arr.copy();observed=np.isfinite(arr)\n    n,p=arr.shape;mask=np.zeros_like(observed);s=float(severity)\n    rng=np.random.default_rng(seed(bundle[\'dataset\'],kind,mechanism,rep)) # severity omitted: paired/nested paths\n    med=st[\'median\'].to_numpy()[None,:];scale=st[\'iqr\'].to_numpy()[None,:]\n    onset=None;context=None\n    if s>0 and kind==\'missing\':\n        if mechanism==\'MCAR\':mask=rng.random((n,p))<s\n        elif mechanism==\'Block outage\':\n            # Every channel loses a contiguous interval of the requested clock fraction.\n            length=min(n,max(1,int(round(s*n))))\n            anchors=rng.uniform(0,1,p)\n            for j in range(p):\n                start=int(anchors[j]*max(1,n-length+1));mask[start:start+length,j]=True\n        elif mechanism==\'Channel dropout\':\n            order=rng.permutation(p);mask[:,order[:min(p,max(1,int(round(s*p))))]]=True\n        elif mechanism in {\'MAR\',\'MNAR sensitivity\'}:\n            dev=bundle[\'raw\'].loc[:bundle[\'splits\'][\'development\'][-1],cols].to_numpy()\n            if mechanism==\'MAR\':\n                context=next((c for c in [\'RH\',\'RH_out\',\'T\',\'T_out\'] if c in cols),cols[0]);j=cols.index(context)\n                z=(arr[:,j]-med[0,j])/scale[0,j];zd=(dev[:,j]-med[0,j])/scale[0,j]\n                a=fit_risk_intercept(zd,s)\n                prob=1/(1+np.exp(-np.clip(np.nan_to_num(z)+a,-40,40)))\n                mask=rng.random((n,p))<prob[:,None];mask[:,j]=False\n            else:\n                z=np.abs((arr-med)/scale);zd=np.abs((dev-med)/scale)\n                a=fit_risk_intercept(zd.ravel(),s)\n                prob=1/(1+np.exp(-np.clip(np.nan_to_num(z)+a,-40,40)))\n                mask=rng.random((n,p))<prob\n        else:raise KeyError(mechanism)\n        arr[mask & observed]=np.nan\n    elif s>0 and kind==\'fault\':\n        order=rng.permutation(p)\n        if mechanism==\'Gaussian\':arr+=s*rng.normal(size=(n,p))*scale\n        elif mechanism==\'Multiplicative\':arr*=1+s/2*rng.normal(size=(n,p))\n        elif mechanism==\'Calibration bias\':\n            chosen=order[:max(1,int(np.ceil(.35*p)))];arr[:,chosen]+=2*s*scale[:,chosen]\n        elif mechanism==\'Spikes\':\n            hit=rng.random((n,p))<s/2;arr+=hit*rng.choice([-1,1],(n,p))*(2.5+5*s)*scale\n        elif mechanism==\'Clipping\':arr=np.clip(arr,med-(1.5-s)*scale,med+(1.5-s)*scale)\n        elif mechanism==\'Stuck-at\':\n            chosen=order[:max(1,int(round(s*p)))];starts=rng.integers(0,max(1,n//2),p)\n            for j in chosen:arr[starts[j]:,j]=med[0,j]\n        else:raise KeyError(mechanism)\n        # Signed electronic measurement errors are permitted; no undocumented clipping to test bounds.\n    elif s>0 and kind==\'drift\':\n        onset=max(1,int(.40*n));chosen=rng.permutation(p)[:max(1,int(np.ceil(.35*p)))];t=np.arange(n-onset)\n        if mechanism==\'Abrupt mean\':add=np.ones((len(t),len(chosen)))*s\n        elif mechanism==\'Gradual mean\':add=np.minimum(1,t/max(1,n//4))[:,None]*s*np.ones((1,len(chosen)))\n        elif mechanism==\'Variance\':add=rng.normal(size=(len(t),len(chosen)))*s\n        elif mechanism==\'Recurring\':add=(((t//max(1,n//6))%2)==0)[:,None]*s*np.ones((1,len(chosen)))\n        else:raise KeyError(mechanism)\n        arr[onset:,chosen]+=add*scale[:,chosen]\n    arr[~observed]=np.nan # never turn a naturally missing value into an observed one\n    raw.loc[idx,cols]=arr\n    X=features(raw,bundle[\'spec\'],cols).loc[idx]\n    target=bundle[\'spec\'][\'target\'];P=bundle[\'spec\'][\'period\']\n    naive=pd.DataFrame({\'Persistence\':raw[target], \'Seasonal naive\':raw[target].shift(P-1)},index=raw.index).loc[idx]\n    changed=np.isfinite(original)&(np.isnan(arr)|(~np.isclose(original,arr,equal_nan=True)))\n    changed_abs=np.where(np.isfinite(arr)&observed,abs(arr-original)/scale,np.nan)\n    meta={\'nominal_severity\':s,\'achieved_new_missing_fraction\':float((np.isnan(arr)&observed).sum()/max(1,observed.sum())),\n          \'changed_observed_fraction\':float(changed.sum()/max(1,observed.sum())),\n          \'mean_absolute_change_IQR\':float(np.nanmean(changed_abs)),\'MAR_context_protected\':context,\n          \'injection_onset\':onset,\'grid_rows\':n,\'scored_rows\':int(bundle[\'y\'].loc[idx].notna().sum())}\n    return X,naive,meta\n\n\ndef scenario(bundle,name,rep,scope=\'test\'):\n    lookup={\'Clean\':(\'none\',\'Clean\',0),\'MCAR 20%\':(\'missing\',\'MCAR\',.2),\n      \'Block 20%\':(\'missing\',\'Block outage\',.2),\'Gaussian 10%\':(\'fault\',\'Gaussian\',.1),\n      \'Bias 10%\':(\'fault\',\'Calibration bias\',.1),\'Abrupt drift\':(\'drift\',\'Abrupt mean\',.75)}\n    return corrupt(bundle,*lookup[name],rep,scope=scope)\n')

### Bundled source: Models

In [ ]:
write_module('primna_esd_models.py','"""Six fixed model families, temporal inner validation, per-fit checkpoints."""\nfrom __future__ import annotations\nimport json,time,warnings\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import BaseEstimator,TransformerMixin\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.compose import TransformedTargetRegressor\nfrom sklearn.linear_model import Ridge\nfrom sklearn.ensemble import RandomForestRegressor,HistGradientBoostingRegressor\nfrom sklearn.neural_network import MLPRegressor\nfrom threadpoolctl import threadpool_limits\nfrom primna_esd_runtime import seed,metrics,paired_block_summary,ci\nfrom primna_esd_data import DATASETS,cv_folds\n\nMODELS=[\'Persistence\',\'Seasonal naive\',\'Ridge\',\'Random Forest\',\'HistGradientBoosting\',\'MLP\']\nML=MODELS[2:]\nSHORT={\'Persistence\':\'Persist.\',\'Seasonal naive\':\'Seasonal\',\'Ridge\':\'Ridge\',\'Random Forest\':\'RF\',\'HistGradientBoosting\':\'HGB\',\'MLP\':\'MLP\'}\n\n\nclass MissingAware(TransformerMixin,BaseEstimator):\n    """Training-only median imputation and fixed indicators for ALL input columns."""\n    def fit(self,X,y=None):\n        a=np.asarray(X,dtype=float);self.n_features_in_=a.shape[1]\n        self.medians_=pd.DataFrame(a).median().fillna(0).to_numpy()\n        return self\n    def transform(self,X):\n        a=np.asarray(X,dtype=float)\n        if a.shape[1]!=self.n_features_in_:raise ValueError(\'Feature schema mismatch\')\n        m=np.isnan(a);return np.concatenate([np.where(m,self.medians_,a),m.astype(float)],axis=1)\n\n\ndef grid(name,profile):\n    if profile==\'pilot\':\n        return {\'Ridge\':[{\'alpha\':10.}],\n                \'Random Forest\':[{\'n_estimators\':35,\'max_depth\':10,\'min_samples_leaf\':2,\'max_features\':.7}],\n                \'HistGradientBoosting\':[{\'max_iter\':70,\'max_leaf_nodes\':15,\'learning_rate\':.05,\'l2_regularization\':.1}],\n                \'MLP\':[{\'hidden_layer_sizes\':(32,16),\'alpha\':.001,\'learning_rate_init\':.001,\'max_iter\':120}]}[name]\n    return {\n      \'Ridge\':[{\'alpha\':a} for a in [.1,1.,10.,100.]],\n      \'Random Forest\':[{\'n_estimators\':300,\'max_depth\':d,\'min_samples_leaf\':l,\'max_features\':f} for d,l,f in [(12,1,.7),(18,2,.7),(None,4,1.)]],\n      \'HistGradientBoosting\':[{\'max_iter\':i,\'max_leaf_nodes\':l,\'learning_rate\':lr,\'l2_regularization\':reg} for i,l,lr,reg in [(350,15,.05,.1),(350,31,.05,.5),(450,63,.03,1.)]],\n      \'MLP\':[{\'hidden_layer_sizes\':h,\'alpha\':a,\'learning_rate_init\':lr,\'max_iter\':i} for h,a,lr,i in [((64,32),.001,.001,700),((128,64),.001,.0005,700),((128,64,32),.01,.0005,800)]]\n    }[name]\n\n\ndef factory(name,params,random_seed,n_threads=4):\n    p=dict(params)\n    if name==\'Ridge\':return Pipeline([(\'impute\',MissingAware()),(\'scale\',StandardScaler()),(\'model\',Ridge(**p))])\n    if name==\'Random Forest\':return Pipeline([(\'impute\',MissingAware()),(\'model\',RandomForestRegressor(**p,random_state=random_seed,n_jobs=n_threads))])\n    if name==\'HistGradientBoosting\':return Pipeline([(\'impute\',MissingAware()),(\'model\',HistGradientBoostingRegressor(**p,early_stopping=False,random_state=random_seed))])\n    if name==\'MLP\':\n        reg=Pipeline([(\'impute\',MissingAware()),(\'scale\',StandardScaler()),(\'model\',MLPRegressor(**p,early_stopping=False,n_iter_no_change=40,random_state=random_seed))])\n        return TransformedTargetRegressor(regressor=reg,transformer=StandardScaler())\n    raise KeyError(name)\n\n\ndef fit_logged(ctx,model,X,y,label):\n    start=time.perf_counter()\n    with warnings.catch_warnings(record=True) as ws,threadpool_limits(limits=ctx.cfg[\'threads\']):\n        warnings.simplefilter(\'always\');model.fit(X,y)\n    messages=[str(w.category.__name__)+\': \'+str(w.message) for w in ws]\n    for w in messages:ctx.log(\'WARNING %s | %s\',label,w)\n    return model,messages,time.perf_counter()-start\n\n\ndef predict_all(bundle,fit,X,naive):\n    arr=[]\n    for model in MODELS:\n        if model not in ML:\n            p=naive[model].fillna(fit[\'fallbacks\'][model]).to_numpy(dtype=float)\n        else:p=fit[\'models\'][model].predict(X)\n        if not np.isfinite(p).all():raise ValueError(\'Nonfinite prediction: \'+model)\n        arr.append(p)\n    return np.column_stack(arr)\n\n\ndef train(ctx,bundles):\n    fitted={};cv_records=[];clean_records=[];clean_summary=[];selection=[]\n    for ds in DATASETS:\n        b=bundles[ds];dev=b[\'splits\'][\'development\'];observed=dev[b[\'y\'].loc[dev].notna()]\n        fit={\'models\':{},\'params\':{},\'cv_scores\':{},\'fallbacks\':{},\'clean\':{}}\n        for name in MODELS[:2]:\n            v=b[\'naive\'].loc[observed,name].median();fit[\'fallbacks\'][name]=float(v if np.isfinite(v) else b[\'y\'].loc[observed].median())\n        for name in ML:\n            configs=grid(name,ctx.cfg[\'profile\']);scores=[]\n            for config_id,params in enumerate(configs):\n                errors=[]\n                for fold,tr,va in cv_folds(b,ctx.cfg[\'cv_folds\']):\n                    label=f\'{ds} | {name} | config {config_id+1}/{len(configs)} | fold {fold+1}/{ctx.cfg["cv_folds"]}\'\n                    def work(name=name,params=params,tr=tr,va=va,fold=fold,label=label):\n                        est=factory(name,params,seed(ctx.cfg[\'seed\'],ds,name,fold),ctx.cfg[\'threads\'])\n                        est,ws,elapsed=fit_logged(ctx,est,b[\'X\'].loc[tr],b[\'y\'].loc[tr],label)\n                        pr=est.predict(b[\'X\'].loc[va]);return dict(metrics(b[\'y\'].loc[va],pr),warnings=ws,fit_seconds=elapsed,\n                            train_end=str(tr[-1]),validation_start=str(va[0]),n_train=len(tr),n_validation=len(va))\n                    out=ctx.task(\'P2CV\',label,{\'data\':b[\'data_key\'],\'params\':params,\'fold\':fold},work)\n                    errors.append(out[\'RMSE\']);cv_records.append(dict(dataset=ds,model=name,config=config_id,fold=fold,parameters=json.dumps(params),**out))\n                scores.append(float(np.mean(errors)))\n            best=int(np.argmin(scores));params=configs[best];fit[\'params\'][name]=params;fit[\'cv_scores\'][name]=scores[best]\n            label=f\'{ds} | {name} | final fit\'\n            def finalfit(name=name,params=params,label=label):\n                est=factory(name,params,seed(ctx.cfg[\'seed\'],ds,name,\'final\'),ctx.cfg[\'threads\'])\n                est,ws,elapsed=fit_logged(ctx,est,b[\'X\'].loc[observed],b[\'y\'].loc[observed],label)\n                return {\'model\':est,\'warnings\':ws,\'fit_seconds\':elapsed}\n            out=ctx.task(\'P2FIT\',label,{\'data\':b[\'data_key\'],\'params\':params},finalfit)\n            fit[\'models\'][name]=out[\'model\']\n            selection.append({\'dataset\':ds,\'model\':name,\'best_config\':best,\'cv_RMSE\':scores[best],\'params\':json.dumps(params),\'warnings\':\' | \'.join(out[\'warnings\'])})\n        fit[\'deployment_model\']=min(ML,key=lambda n:(fit[\'cv_scores\'][n],n))\n        ctx.log(\'%s deployment model frozen from development CV only: %s\',ds,fit[\'deployment_model\'])\n        for split in [\'calibration\',\'policy\',\'test\']:\n            idx=b[\'splits\'][split];y=b[\'y\'].loc[idx].to_numpy();P=predict_all(b,fit,b[\'X\'].loc[idx],b[\'naive\'].loc[idx]);fit[\'clean\'][split]=P\n            for j,name in enumerate(MODELS):\n                for i,t in enumerate(idx):clean_records.append({\'dataset\':ds,\'split\':split,\'origin\':t,\'target_time\':t+pd.Timedelta(b[\'spec\'][\'frequency\']),\n                    \'model\':name,\'y_true\':y[i],\'y_pred\':P[i,j],\'scored\':bool(np.isfinite(y[i]))})\n            point,_,boot,_,_,_=paired_block_summary(y,P[None,:,:],P,b[\'spec\'][\'period\'],ctx.cfg[\'bootstrap\'],np.random.default_rng(seed(ds,split,\'clean_ci\')))\n            for j,name in enumerate(MODELS):\n                lo,hi=ci(boot[:,j]);clean_summary.append(dict(dataset=ds,split=split,model=name,**metrics(y,P[:,j]),RMSE_low=lo,RMSE_high=hi))\n        fitted[ds]=fit\n    ctx.table(\'P2_rolling_origin_CV\',cv_records);ctx.table(\'P2_model_selection\',selection)\n    ctx.table(\'P2_clean_row_predictions\',clean_records);ctx.table(\'P2_clean_summary\',clean_summary)\n    return fitted\n')

### Bundled source: Stress

In [ ]:
write_module('primna_esd_stress.py','"""RQ1/RQ2 common-denominator stress tests and RQ3 explanation stability."""\nfrom __future__ import annotations\nimport math,time\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\nfrom scipy.integrate import trapezoid\nfrom primna_esd_runtime import seed,metrics,paired_block_summary,ci,holm,paired_sign_test\nfrom primna_esd_data import DATASETS,MISSING,FAULTS,corrupt,scenario\nfrom primna_esd_models import MODELS,ML,predict_all\n\n\ndef stress_suite(ctx,bundles,fitted,rq):\n    kind=\'missing\' if rq==1 else \'fault\';mechanisms=MISSING if rq==1 else FAULTS\n    levels=ctx.cfg[\'missing_levels\'] if rq==1 else ctx.cfg[\'fault_levels\']\n    records=[];summary=[];audcs=[];tests=[];tails=[];audit=[]\n    total=len(DATASETS)*len(mechanisms)*len(levels)*ctx.cfg[\'perturb_reps\']\n    completed=0;started=time.monotonic()\n    for ds in DATASETS:\n        b=bundles[ds];fit=fitted[ds];idx=b[\'splits\'][\'test\'];y=b[\'y\'].loc[idx].to_numpy();clean=fit[\'clean\'][\'test\']\n        R=ctx.cfg[\'perturb_reps\'];B=ctx.cfg[\'bootstrap\'];block=b[\'spec\'][\'period\'];all_mechanism_boot=[]\n        clean_rmse=[metrics(y,clean[:,j])[\'RMSE\'] for j in range(len(MODELS))]\n        ctx.log(\'RQ%d PLAN | dataset=%s | rows=%d | models=%d | mechanisms=%d | levels=%d | replicas=%d | bootstrap=%d\',\n                rq,ds,len(idx),len(MODELS),len(mechanisms),len(levels),R,B)\n        for mech in mechanisms:\n            relative_boot=[];relative_point=[]\n            for s in levels:\n                ctx.log(\'RQ%d LEVEL START | %s | %s | severity=%s\',rq,ds,mech,s)\n                PP=[]\n                for rep in range(R):\n                    label=f\'RQ{rq} | {ds} | {mech} s={s} | replicate {rep+1}/{R}\'\n                    if s==0:out={\'predictions\':clean,\'audit\':{\'achieved_new_missing_fraction\':0.,\'changed_observed_fraction\':0.,\'mean_absolute_change_IQR\':0.}}\n                    else:\n                        def work(mech=mech,s=s,rep=rep):\n                            with ctx.substage(label+\' corruption\'):\n                                X,N,meta=corrupt(b,kind,mech,s,rep)\n                            with ctx.substage(label+\' six-model prediction\',shape=X.shape):\n                                columns=[]\n                                for model_name in MODELS:\n                                    def predict_one(model_name=model_name):\n                                        if model_name not in ML:\n                                            p=N[model_name].fillna(fit[\'fallbacks\'][model_name]).to_numpy(dtype=float)\n                                        else:p=fit[\'models\'][model_name].predict(X)\n                                        p=np.asarray(p)\n                                        if p.shape!=(len(X),) or not np.isfinite(p).all():\n                                            raise ValueError(\'Invalid prediction shape/values: \'+model_name)\n                                        return p\n                                    columns.append(ctx.task(\'RQ\'+str(rq)+\'MODEL\',label+\' | model=\'+model_name,\n                                        {\'data\':b[\'data_key\'],\'mechanism\':mech,\'severity\':s,\'rep\':rep,\n                                         \'model\':model_name,\'params\':fit.get(\'params\',{}).get(model_name)},predict_one))\n                                predictions=np.column_stack(columns)\n                            return {\'predictions\':predictions,\'audit\':meta}\n                        out=ctx.task(\'RQ\'+str(rq),label,{\'data\':b[\'data_key\'],\'mechanism\':mech,\'severity\':s,\'rep\':rep},work)\n                    P=out[\'predictions\'];PP.append(P)\n                    completed+=1\n                    ctx.log(\'RQ%d PROGRESS | units=%d/%d | elapsed=%.1f min | current=%s\',\n                            rq,completed,total,(time.monotonic()-started)/60,label)\n                    audit.append(dict(dataset=ds,mechanism=mech,severity=s,replicate=rep,**out[\'audit\']))\n                    for j,m in enumerate(MODELS):\n                        met=metrics(y,P[:,j]);cr=clean_rmse[j]\n                        records.append(dict(dataset=ds,mechanism=mech,severity=s,replicate=rep,model=m,**met,clean_RMSE=cr,relative_change=met[\'RMSE\']/cr-1))\n                        if s==levels[-1]:tails.append(dict(dataset=ds,mechanism=mech,replicate=rep,model=m,tail95=met[\'AE_q95\'],MAE=met[\'MAE\']))\n                PP=np.asarray(PP)\n                with ctx.substage(f\'RQ{rq} bootstrap | {ds} | {mech} | {s}\',draws=B,shape=PP.shape):\n                    point,cp,bt,rb,ss,counts=ctx.task(\'RQ\'+str(rq)+\'BOOT\',f\'RQ{rq} bootstrap | {ds} | {mech} | {s}\',\n                        {\'data\':b[\'data_key\'],\'mechanism\':mech,\'severity\':s,\'replicas\':R,\'draws\':B,\'block\':block},\n                        lambda:paired_block_summary(y,PP,clean,block,B,np.random.default_rng(seed(ds,rq,\'shared_boot\'))))\n                relative_point.append(point/cp-1);relative_boot.append(rb)\n                for j,m in enumerate(MODELS):\n                    lo,hi=ci(bt[:,j]);rl,rh=ci(rb[:,j]);summary.append(dict(dataset=ds,mechanism=mech,severity=s,model=m,\n                      pooled_RMSE=point[j],RMSE_low=lo,RMSE_high=hi,relative_change=point[j]/cp[j]-1,relative_low=rl,relative_high=rh,\n                      interval=\'95% replicate + clock-day block bootstrap; fixed historical test period\'))\n                if s==levels[-1]:\n                    ref=MODELS.index(\'HistGradientBoosting\')\n                    for j,m in enumerate(MODELS):\n                        if j==ref:continue\n                        diff=(ss[:,j,:]-ss[:,ref,:]).mean(axis=0)\n                        p_sign=ctx.task(\'RQ\'+str(rq)+\'SIGN\',f\'RQ{rq} sign test | {ds} | {mech} | {m}\',\n                            {\'data\':b[\'data_key\'],\'severity\':s,\'draws\':ctx.cfg[\'randomization_draws\']},\n                            lambda:paired_sign_test(diff,ctx.cfg[\'randomization_draws\'],np.random.default_rng(seed(ds,rq,mech,m,\'test\'))))\n                        tests.append({\'dataset\':ds,\'mechanism\':mech,\'severity\':s,\'model\':m,\'reference\':\'HistGradientBoosting\',\n                         \'MSE_difference\':float(diff.sum()/counts.sum()),\n                         \'p_block_sign\':p_sign,\n                         \'n_clock_day_blocks\':len(counts),\'assumption\':\'secondary block-sign randomization; blocks sign-exchangeable; not independent datasets\'})\n            area=trapezoid(np.array(relative_point),levels,axis=0)/max(levels)\n            area_boot=trapezoid(np.array(relative_boot),levels,axis=0)/max(levels)\n            all_mechanism_boot.append((area,area_boot))\n            for j,m in enumerate(MODELS):\n                lo,hi=ci(area_boot[:,j]);audcs.append(dict(dataset=ds,mechanism=mech,model=m,AUDC=area[j],AUDC_low=lo,AUDC_high=hi))\n        av=np.mean([x[0] for x in all_mechanism_boot],axis=0);avb=np.mean([x[1] for x in all_mechanism_boot],axis=0)\n        for j,m in enumerate(MODELS):\n            lo,hi=ci(avb[:,j]);audcs.append(dict(dataset=ds,mechanism=\'Descriptive equal-weight scenario mean\',model=m,AUDC=av[j],AUDC_low=lo,AUDC_high=hi))\n    tests=pd.DataFrame(tests);tests[\'p_Holm\']=np.nan\n    for ds,g in tests.groupby(\'dataset\'):tests.loc[g.index,\'p_Holm\']=holm(g[\'p_block_sign\'])\n    out={\'replicates\':pd.DataFrame(records),\'summary\':pd.DataFrame(summary),\'audc\':pd.DataFrame(audcs),\n         \'comparisons\':tests,\'tails\':pd.DataFrame(tails),\'injection_audit\':pd.DataFrame(audit)}\n    for name,df in out.items():ctx.table(f\'RQ{rq}_{name}\',df)\n    return out\n\n\nEXPLANATION_SCENARIOS=[\'Clean repeat control\',\'MCAR 20%\',\'Block 20%\',\'Gaussian 10%\',\'Bias 10%\',\'Abrupt drift\']\n\n\ndef explanation_positions(n,budget):\n    # Two prespecified CONTIGUOUS windows, not evenly spaced isolated observations.\n    w=min(n//2,max(20,budget//2));starts=[max(0,n//4-w//2),min(n-w,3*n//4-w//2)]\n    a=np.arange(starts[0],starts[0]+w);b=np.arange(starts[1],starts[1]+w)\n    if np.intersect1d(a,b).size:return np.arange(min(n,budget)),[min(n,budget)]\n    return np.r_[a,b],[w,w]\n\n\ndef permutation_order(lengths,block,rng):\n    order=[];offset=0\n    for n in lengths:\n        parts=[np.arange(i,min(n,i+block))+offset for i in range(0,n,block)]\n        order.extend([parts[i] for i in rng.permutation(len(parts))]);offset+=n\n    return np.concatenate(order)\n\n\ndef group_importance(model,X,y,groups,repeats,lengths,block,random_seed):\n    baseline=metrics(y,model.predict(X))[\'RMSE\'];rows=[]\n    for group,cols in groups.items():\n        for r in range(repeats):\n            rng=np.random.default_rng(seed(random_seed,group,r))\n            order=permutation_order(lengths,block,rng);Xp=X.copy();Xp.loc[:,cols]=X.iloc[order][cols].to_numpy()\n            rows.append({\'group\':group,\'permutation\':r,\'importance\':metrics(y,model.predict(Xp))[\'RMSE\']-baseline})\n    return pd.DataFrame(rows),baseline\n\n\ndef stability(a,b):\n    a,b=a.align(b);a=a.astype(float);b=b.astype(float)\n    sp=float(stats.spearmanr(a,b).statistic) if a.nunique()>1 and b.nunique()>1 else np.nan\n    kt=float(stats.kendalltau(a,b).statistic) if a.nunique()>1 and b.nunique()>1 else np.nan\n    k=min(8,max(2,len(a)//3));topa=set(a.index[a>=a.nlargest(k).min()]);topb=set(b.index[b>=b.nlargest(k).min()])\n    active=np.abs(a)>max(1e-12,.1*np.median(abs(a)))\n    den=np.linalg.norm(a)*np.linalg.norm(b)\n    return {\'Spearman\':sp,\'Kendall\':kt,\'topk_Jaccard\':len(topa&topb)/max(1,len(topa|topb)),\n      \'signed_cosine\':float(np.dot(a,b)/den) if den else np.nan,\n      \'sign_agreement\':float(np.mean(np.sign(a[active])==np.sign(b[active]))) if active.any() else np.nan,\n      \'top_k_requested\':k,\'top_a_with_ties\':len(topa),\'top_b_with_ties\':len(topb)}\n\n\ndef explanation_suite(ctx,bundles,fitted):\n    profiles=[];out=[];windows=[]\n    R=ctx.cfg[\'explanation_reps\'];K=ctx.cfg[\'importance_repeats\']\n    for ds in DATASETS:\n        b=bundles[ds];idx=b[\'splits\'][\'test\'];pos,lengths=explanation_positions(len(idx),ctx.cfg[\'explanation_rows\'])\n        ti=idx[pos];y=b[\'y\'].loc[ti].to_numpy();clean=b[\'X\'].loc[ti];block=max(4,b[\'spec\'][\'period\']//6)\n        windows.extend([{\'dataset\':ds,\'origin\':t} for t in ti])\n        for model in ML:\n            est=fitted[ds][\'models\'][model]\n            for rep in range(R):\n                lab=f\'RQ3 | {ds} | {model} | paired clean profile {rep+1}/{R}\'\n                clean_res=ctx.task(\'RQ3\',lab,{\'data\':b[\'data_key\'],\'model\':model,\'rep\':rep,\'role\':\'clean\'},\n                    lambda rep=rep:group_importance(est,clean,y,b[\'groups\'],K,lengths,block,seed(ds,rep,\'paired_perms\')))\n                ip,clean_rmse=clean_res;a=ip.groupby(\'group\')[\'importance\'].mean()\n                tmp=ip.assign(dataset=ds,model=model,scenario=\'Clean\',replicate=rep);profiles.append(tmp)\n                for condition in EXPLANATION_SCENARIOS:\n                    lab=f\'RQ3 | {ds} | {model} | {condition} | {rep+1}/{R}\'\n                    def work(condition=condition,rep=rep):\n                        if condition==\'Clean repeat control\':Xp=clean;sr=seed(ds,rep,\'independent_clean_perms\')\n                        else:Xp=scenario(b,condition,rep)[0].loc[ti];sr=seed(ds,rep,\'paired_perms\')\n                        return group_importance(est,Xp,y,b[\'groups\'],K,lengths,block,sr)\n                    res=ctx.task(\'RQ3\',lab,{\'data\':b[\'data_key\'],\'model\':model,\'rep\':rep,\'condition\':condition},work)\n                    ip2,pert_rmse=res;bp=ip2.groupby(\'group\')[\'importance\'].mean();met=stability(a,bp)\n                    out.append(dict(dataset=ds,model=model,scenario=condition,replicate=rep,**met,\n                                    relative_RMSE_change=pert_rmse/clean_rmse-1,n_scored=int(np.isfinite(y).sum())))\n                    profiles.append(ip2.assign(dataset=ds,model=model,scenario=condition,replicate=rep))\n    stability_df=pd.DataFrame(out)\n    # RQ3 asks whether degradation exceeds the explainer\'s own Monte Carlo\n    # variability.  Therefore every degraded profile is paired with the clean\n    # repeat from the same dataset/model/replicate; raw stability alone is not\n    # an answer to that question.\n    metrics_to_compare=[\'Spearman\',\'Kendall\',\'topk_Jaccard\',\'signed_cosine\',\'sign_agreement\']\n    control=(stability_df[stability_df.scenario==\'Clean repeat control\']\n             [[\'dataset\',\'model\',\'replicate\']+metrics_to_compare]\n             .rename(columns={m:m+\'_control\' for m in metrics_to_compare}))\n    paired=(stability_df[stability_df.scenario!=\'Clean repeat control\']\n            .merge(control,on=[\'dataset\',\'model\',\'replicate\'],validate=\'many_to_one\'))\n    contrasts=[]\n    rng=np.random.default_rng(seed(ctx.cfg[\'seed\'],\'RQ3 paired contrasts\'))\n    for (ds,model,scenario_name),g in paired.groupby([\'dataset\',\'model\',\'scenario\'],sort=False):\n        for metric_name in metrics_to_compare:\n            delta=(g[metric_name]-g[metric_name+\'_control\']).dropna().to_numpy(float)\n            if not len(delta):continue\n            draws=np.array([np.mean(rng.choice(delta,len(delta),replace=True)) for _ in range(ctx.cfg[\'bootstrap\'])])\n            # Two-sided exact sign-flip randomization on paired replica contrasts.\n            B=ctx.cfg[\'randomization_draws\'];observed=abs(delta.sum())\n            null=np.abs((rng.choice([-1.,1.],size=(B,len(delta)))*delta).sum(axis=1))\n            contrasts.append({\'dataset\':ds,\'model\':model,\'scenario\':scenario_name,\'metric\':metric_name,\n              \'n_pairs\':len(delta),\'mean_degraded_minus_control\':float(delta.mean()),\n              \'CI_low\':float(np.quantile(draws,.025)),\'CI_high\':float(np.quantile(draws,.975)),\n              \'p_signflip\':float((1+(null>=observed-1e-12).sum())/(B+1))})\n    contrast_df=pd.DataFrame(contrasts);contrast_df[\'p_Holm\']=np.nan\n    for (ds,model,metric_name),g in contrast_df.groupby([\'dataset\',\'model\',\'metric\']):\n        contrast_df.loc[g.index,\'p_Holm\']=holm(g.p_signflip)\n    result={\'stability\':stability_df,\'contrasts\':contrast_df,\n      \'importance\':pd.concat(profiles,ignore_index=True),\'windows\':pd.DataFrame(windows)}\n    for k,v in result.items():ctx.table(\'RQ3_\'+k,v)\n    return result\n')


### Bundled source: Drift

In [ ]:
write_module('primna_esd_drift.py','"""RQ4: empirical calibration feasibility, matched controls, non-oracle adaptation."""\nfrom __future__ import annotations\nfrom collections import deque\nimport math,time\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import ks_2samp, chi2\nfrom primna_esd_runtime import seed,metrics,ci\nfrom primna_esd_data import DATASETS,DRIFTS,corrupt\nfrom primna_esd_models import ML,factory,fit_logged\n\nGRIDS={\'Page-Hinkley\':[8,18,35,75,150,400,1000],\n       \'CUSUM\':[8,18,40,80,160,400,1000],\n       \'EWMA\':[2,4,8,16,32,64],\n       \'Windowed KS\':[.02,.005,.001,.0001,.00001,.000001]}\n\n\nclass Detector:\n    """One-sided error-rise monitoring. KS is a two-adjacent-window KS detector,\n    NOT River KSWIN. Thresholds are calibrated empirically; iid p-values are not\n    claimed valid under serial dependence. reset/cooldown applies after all alarms.\n    """\n    def __init__(self,method,param,cooldown=30):\n        self.method=method;self.param=param;self.cooldown=cooldown;self.last=-10**9;self.reset()\n    def reset(self):self.n=0;self.mean=0.;self.s=0.;self.minimum=0.;self.z=0.;self.buf=deque(maxlen=120)\n    def update(self,value,index):\n        if not np.isfinite(value):return False\n        self.n+=1;alarm=False\n        if self.method==\'Page-Hinkley\':\n            self.mean+=(value-self.mean)/self.n;self.s+=value-self.mean-.02;self.minimum=min(self.minimum,self.s)\n            alarm=self.n>=30 and self.s-self.minimum>self.param\n        elif self.method==\'CUSUM\':\n            self.s=max(0,self.s+value-.5);alarm=self.n>=30 and self.s>self.param\n        elif self.method==\'EWMA\':\n            self.z=.12*value+.88*self.z;alarm=self.n>=30 and self.z>self.param\n        elif self.method==\'Windowed KS\':\n            self.buf.append(value)\n            if len(self.buf)==120 and self.n%10==0:\n                a=np.asarray(self.buf);alarm=ks_2samp(a[:80],a[80:],method=\'asymp\').pvalue<self.param and a[80:].mean()>a[:80].mean()\n        else:raise KeyError(self.method)\n        alarm=alarm and index-self.last>=self.cooldown\n        if alarm:self.last=index;self.reset()\n        return bool(alarm)\n\n\ndef alarms(z,method,param):\n    d=Detector(method,param);return [i for i,v in enumerate(z) if d.update(v,i)]\n\n\ndef normalize(errors,reference):\n    ref=np.asarray(reference);ref=ref[np.isfinite(ref)];mu=float(ref.mean());sd=max(float(ref.std()),1e-8)\n    return (np.asarray(errors)-mu)/sd,mu,sd\n\n\ndef circular_blocks(x,block,rng):\n    x=np.asarray(x);n=len(x);starts=rng.integers(0,n,math.ceil(n/block))\n    ii=np.concatenate([(s+np.arange(block))%n for s in starts])[:n];return x[ii]\n\n\ndef select_feasible(table,target):\n    # The old implementation treated a zero count as proof that FAR <= target.\n    # Here the exact one-sided Poisson upper bound is the evidential gate.  A\n    # provisional candidate can still be studied when the available exposure is\n    # too short, but it is never labelled as satisfying the operational claim.\n    t=table.copy();t[\'feasible\']=(t[\'null_far_q95\']<=target)&(t[\'observed_policy_far\']<=target)\n    t[\'far_claim_supported\']=t[\'observed_far_upper95\']<=target\n    t[\'selected\']=False;t[\'provisional_selected\']=False\n    ok=t[t.feasible & (t.policy_strict_detection>=.5)].sort_values(\n        [\'policy_strict_detection\',\'policy_median_delay\',\'observed_far_upper95\'],ascending=[False,True,True],na_position=\'last\')\n    if not ok.empty:\n        t.loc[ok.index[0],\'provisional_selected\']=True\n        if bool(ok.iloc[0].far_claim_supported):t.loc[ok.index[0],\'selected\']=True\n    return t\n\n\ndef calibrate(ctx,b,fit):\n    model_name=fit[\'deployment_model\'];model=fit[\'models\'][model_name]\n    ca=b[\'splits\'][\'calibration\'];po=b[\'splits\'][\'policy\']\n    calerr=abs(b[\'y\'].loc[ca].to_numpy()-model.predict(b[\'X\'].loc[ca]))\n    polerr=abs(b[\'y\'].loc[po].to_numpy()-model.predict(b[\'X\'].loc[po]))\n    z,mu,sd=normalize(polerr,calerr);S=ctx.cfg[\'detector_sims\'];tables=[]\n    for method,params in GRIDS.items():\n        rec=[]\n        for param in params:\n            label=f\'RQ4 calibration | {b["dataset"]} | {method} | {param}\'\n            def work(method=method,param=param):\n                nulls=[];strict=[];delay=[]\n                for r in range(S):\n                    rng=np.random.default_rng(seed(b[\'dataset\'],\'shared_policy_stream\',r))\n                    x=circular_blocks(z,b[\'spec\'][\'period\'],rng);nulls.append(len(alarms(x,method,param))*1000/len(x))\n                    point=int(.6*len(x));xd=x.copy();xd[point:]+=2.\n                    aa=alarms(xd,method,param);pre=[i for i in aa if i<point];post=[i for i in aa if i>=point]\n                    strict.append(int(not pre and bool(post)))\n                    if not pre and post:delay.append(post[0]-point)\n                k=len(alarms(z,method,param));exposure=len(z)\n                upper=.5*chi2.ppf(.95,2*(k+1))/exposure*1000\n                return {\'detector\':method,\'parameter\':param,\'null_mean_far\':np.mean(nulls),\n                   \'null_far_q95\':np.quantile(nulls,.95),\'observed_policy_far\':len(alarms(z,method,param))*1000/len(z),\n                   \'observed_alarm_count\':k,\'observed_exposure\':exposure,\'observed_far_upper95\':upper,\n                   \'policy_strict_detection\':np.mean(strict),\'policy_median_delay\':np.median(delay) if delay else np.nan,\'n_simulations\':S}\n            rec.append(ctx.task(\'RQ4CAL\',label,{\'data\':b[\'data_key\'],\'model\':model_name,\'parameter\':param,\'method\':method},work))\n        tables.append(select_feasible(pd.DataFrame(rec),ctx.cfg[\'target_far\']))\n    tab=pd.concat(tables,ignore_index=True).assign(dataset=b[\'dataset\'],model=model_name)\n    chosen=tab[tab.provisional_selected].sort_values([\'policy_strict_detection\',\'policy_median_delay\',\'observed_far_upper95\'],ascending=[False,True,True],na_position=\'last\')\n    primary=None if chosen.empty else (str(chosen.iloc[0].detector),float(chosen.iloc[0].parameter))\n    supported=bool(not chosen.empty and chosen.iloc[0].far_claim_supported)\n    return {\'table\':tab,\'mean\':mu,\'scale\':sd,\'primary\':primary,\'far_claim_supported\':supported}\n\n\ndef prequential_adapt(ctx,b,fit,X,y,policy,normalizer,detector_spec=None,scheduled_indices=None):\n    """Predict first, reveal y(t+1), update next prediction. No onset argument exists.\n    Batch predictions between updates are a computational optimization: estimates\n    are row-independent and only past labels enter detector/refitting decisions.\n    All alarms, including pre-injection alarms, are actionable. A fixed refit budget\n    and cooldown are configured before test evaluation.\n    """\n    n=len(X);y=np.asarray(y,dtype=float);model_name=fit[\'deployment_model\'];current=fit[\'models\'][model_name]\n    params=fit[\'params\'][model_name];out=np.empty(n);events=[]\n    mu,sd=normalizer;det=Detector(*detector_spec) if detector_spec else None\n    window=ctx.cfg[\'adapt_window\'];cooldown=ctx.cfg[\'adapt_cooldown\'];budget=ctx.cfg[\'max_refits\']\n    dev=b[\'splits\'][\'development\'];dv=dev[b[\'y\'].loc[dev].notna()][-window:]\n    baseX=b[\'X\'].loc[dv];basey=b[\'y\'].loc[dv]\n    cursor=0;last_update=-10**9;last_attempt=-10**9\n    while cursor<n:\n        forecasts=current.predict(X.iloc[cursor:]);did_update=False\n        for i in range(cursor,n):\n            out[i]=forecasts[i-cursor]\n            signal=False\n            if np.isfinite(y[i]) and det:signal=det.update((abs(y[i]-out[i])-mu)/sd,i)\n            scheduled=(policy==\'Periodic\' and (i+1)%cooldown==0) or (\n                policy==\'Random budget-matched\' and scheduled_indices is not None and i in scheduled_indices)\n            triggered=policy in {\'Alarm-triggered\',\'Guarded alarm\'} and signal\n            eligible=np.flatnonzero(np.isfinite(y[:i+1]))\n            if (scheduled or triggered) and i<n-1 and i-last_attempt>=cooldown and len(events)<budget and len(eligible)>=ctx.cfg[\'min_update_labels\']:\n                validation=np.array([],dtype=int)\n                if policy==\'Guarded alarm\':\n                    nv=min(ctx.cfg[\'adapt_validation_labels\'],max(10,len(eligible)//4))\n                    validation=eligible[-nv:];candidate_pool=eligible[:-nv]\n                else:candidate_pool=eligible\n                recent=candidate_pool[-window:]\n                if len(recent)<max(20,ctx.cfg[\'min_update_labels\']//2):continue\n                trainX=pd.concat([baseX,X.iloc[recent]]).tail(window)\n                trainy=pd.concat([basey,pd.Series(y[recent],index=X.index[recent])]).tail(window)\n                # Training includes label of origin i only after its target time arrives.\n                update_time=X.index[i]+pd.Timedelta(b[\'spec\'][\'frequency\'])\n                assert update_time<=X.index[i+1]\n                fresh=factory(model_name,params,seed(ctx.cfg[\'seed\'],b[\'dataset\'],policy,len(events),\'adapt\'),ctx.cfg[\'threads\'])\n                fresh,ws,secs=fit_logged(ctx,fresh,trainX,trainy,\'causal \'+policy+\' candidate\')\n                accepted=True;base_val=np.nan;candidate_val=np.nan;base_tail=np.nan;candidate_tail=np.nan\n                if policy==\'Guarded alarm\':\n                    vy=y[validation];bp=current.predict(X.iloc[validation]);cp=fresh.predict(X.iloc[validation])\n                    base_val=metrics(vy,bp)[\'RMSE\'];candidate_val=metrics(vy,cp)[\'RMSE\']\n                    base_tail=float(np.quantile(abs(vy-bp),.95));candidate_tail=float(np.quantile(abs(vy-cp),.95))\n                    accepted=(candidate_val<=base_val*(1-ctx.cfg[\'adapt_min_relative_gain\']) and\n                              candidate_tail<=base_tail*(1+ctx.cfg[\'adapt_tail_tolerance\']))\n                events.append({\'origin_index\':i,\'label_available_time\':update_time,\'first_new_prediction\':X.index[i+1],\n                               \'n_train\':len(trainX),\'n_validation\':len(validation),\'fit_seconds\':secs,\n                               \'warnings\':\' | \'.join(ws),\'trigger\':policy,\'accepted\':accepted,\n                               \'current_validation_RMSE\':base_val,\'candidate_validation_RMSE\':candidate_val,\n                               \'current_validation_AE95\':base_tail,\'candidate_validation_AE95\':candidate_tail})\n                last_attempt=i\n                if det:det.reset()\n                if accepted:\n                    current=fresh;last_update=i;cursor=i+1;did_update=True;break\n        if not did_update:break\n    return {\'predictions\':out,\'events\':events}\n\n\ndef drift_suite(ctx,bundles,fitted):\n    cal_tables=[];rows=[];adapt=[];events=[];timelines=[];natural=[];disabled=[]\n    for ds in DATASETS:\n        b=bundles[ds];fit=fitted[ds];cal=calibrate(ctx,b,fit);cal_tables.append(cal[\'table\'])\n        idx=b[\'splits\'][\'test\'];y=b[\'y\'].loc[idx].to_numpy();model_name=fit[\'deployment_model\'];model=fit[\'models\'][model_name]\n        clean=model.predict(b[\'X\'].loc[idx]);clean_z=(abs(y-clean)-cal[\'mean\'])/cal[\'scale\']\n        selections={str(r.detector):float(r.parameter) for r in cal[\'table\'].itertuples() if r.provisional_selected}\n        supported_detectors={str(r.detector):bool(r.far_claim_supported) for r in cal[\'table\'].itertuples() if r.provisional_selected}\n        for method in GRIDS:\n            if method not in selections:disabled.append({\'dataset\':ds,\'detector\':method,\'status\':\'disabled: no feasible calibration point\'})\n        for method,param in selections.items():\n            for i in alarms(clean_z,method,param):natural.append({\'dataset\':ds,\'detector\':method,\'index\':i,\'time\':idx[i]+pd.Timedelta(b[\'spec\'][\'frequency\']),\n              \'interpretation\':\'unlabelled natural alarm; not verified environmental/concept drift\'})\n        for condition in [\'No injected shift\']+DRIFTS:\n            R=ctx.cfg[\'adaptation_reps\'] if condition==\'No injected shift\' else ctx.cfg[\'drift_reps\']\n            for rep in range(R):\n                if condition==\'No injected shift\':X=b[\'X\'].loc[idx];onset=None\n                else:X,_,meta=corrupt(b,\'drift\',condition,.75,rep);onset=meta[\'injection_onset\']\n                pred=model.predict(X);z=(abs(y-pred)-cal[\'mean\'])/cal[\'scale\']\n                if condition!=\'No injected shift\':\n                    for method in GRIDS:\n                        param=selections.get(method);aa=[] if param is None else alarms(z,method,param)\n                        pre=[i for i in aa if i<onset];post=[i for i in aa if i>=onset]\n                        control=[] if param is None else alarms(clean_z,method,param)\n                        control_post=[i for i in control if i>=onset]\n                        rows.append({\'dataset\':ds,\'model\':model_name,\'drift\':condition,\'replicate\':rep,\'detector\':method,\n                          \'enabled\':param is not None,\'far_target_supported\':supported_detectors.get(method,False),\n                          \'pre_injection_alarms\':len(pre),\'pre_injection_far\':len(pre)*1000/onset,\n                          \'post_injection_alarm\':bool(post),\'alarm_without_prior_alarm\':bool(post) and not pre,\n                          \'matched_clean_post_alarm\':bool(control_post),\'delay_conditional\':post[0]-onset if post else np.nan,\n                          \'delay_or_censored\':post[0]-onset if post else len(idx)-onset,\'injection_index\':onset,\'n_clock\':len(idx)})\n                        if rep==0 and condition==\'Abrupt mean\':\n                            for i in aa:timelines.append({\'dataset\':ds,\'detector\':method,\'kind\':\'alarm\',\'index\':i,\'value\':np.nan,\'injection_index\':onset})\n                    if rep==0 and condition==\'Abrupt mean\':\n                        for i,val in enumerate(abs(y-pred)):timelines.append({\'dataset\':ds,\'detector\':\'Residual\',\'kind\':\'absolute_error\',\'index\':i,\'value\':val,\'injection_index\':onset})\n                if rep<ctx.cfg[\'adaptation_reps\']:\n                    policy_results={}\n                    for pol in [\'Frozen\',\'Periodic\',\'Alarm-triggered\',\'Guarded alarm\']:\n                        label=f\'RQ4 adaptation | {ds} | {condition} | {pol} | {rep+1}\'\n                        if pol==\'Frozen\':res={\'predictions\':pred,\'events\':[]}\n                        elif pol in {\'Alarm-triggered\',\'Guarded alarm\'} and cal[\'primary\'] is None:res={\'predictions\':pred,\'events\':[]}\n                        else:\n                            res=ctx.task(\'RQ4ADAPT\',label,{\'data\':b[\'data_key\'],\'condition\':condition,\'rep\':rep,\'policy\':pol,\'detector\':cal[\'primary\']},\n                              lambda pol=pol,X=X:prequential_adapt(ctx,b,fit,X,y,pol,(cal[\'mean\'],cal[\'scale\']),cal[\'primary\'] if pol in {\'Alarm-triggered\',\'Guarded alarm\'} else None))\n                        policy_results[pol]=res\n                        met=metrics(y,res[\'predictions\']);fmet=metrics(y,pred)\n                        accepted_events=sum(bool(e.get(\'accepted\',True)) for e in res[\'events\'])\n                        adapt.append(dict(dataset=ds,drift=condition,replicate=rep,policy=pol,**met,\n                          frozen_RMSE=fmet[\'RMSE\'],RMSE_difference=met[\'RMSE\']-fmet[\'RMSE\'],refits=accepted_events,refit_attempts=len(res[\'events\']),\n                          refit_seconds=sum(e[\'fit_seconds\'] for e in res[\'events\']),\n                          detector_disabled=cal[\'primary\'] is None and pol in {\'Alarm-triggered\',\'Guarded alarm\'}))\n                        for ev in res[\'events\']:events.append(dict(dataset=ds,drift=condition,replicate=rep,policy=pol,**ev))\n                    # Equal-compute negative control: update at random eligible times\n                    # using the same attempt budget as alarm-triggered adaptation.\n                    budget_n=len(policy_results[\'Alarm-triggered\'][\'events\'])\n                    eligible=np.arange(ctx.cfg[\'min_update_labels\'],max(ctx.cfg[\'min_update_labels\']+1,len(idx)-1),ctx.cfg[\'adapt_cooldown\'])\n                    rng=np.random.default_rng(seed(ds,condition,rep,\'random_budget\'))\n                    schedule=set(sorted(rng.choice(eligible,min(budget_n,len(eligible)),replace=False).tolist())) if budget_n else set()\n                    pol=\'Random budget-matched\';label=f\'RQ4 adaptation | {ds} | {condition} | {pol} | {rep+1}\'\n                    res=ctx.task(\'RQ4ADAPT\',label,{\'data\':b[\'data_key\'],\'condition\':condition,\'rep\':rep,\n                      \'policy\':pol,\'budget\':budget_n},lambda X=X,schedule=schedule:prequential_adapt(\n                        ctx,b,fit,X,y,pol,(cal[\'mean\'],cal[\'scale\']),None,scheduled_indices=schedule))\n                    met=metrics(y,res[\'predictions\']);fmet=metrics(y,pred)\n                    adapt.append(dict(dataset=ds,drift=condition,replicate=rep,policy=pol,**met,\n                      frozen_RMSE=fmet[\'RMSE\'],RMSE_difference=met[\'RMSE\']-fmet[\'RMSE\'],\n                      refits=len(res[\'events\']),refit_attempts=len(res[\'events\']),\n                      refit_seconds=sum(e[\'fit_seconds\'] for e in res[\'events\']),detector_disabled=False))\n                    for ev in res[\'events\']:events.append(dict(dataset=ds,drift=condition,replicate=rep,policy=pol,**ev))\n    adaptation_df=pd.DataFrame(adapt)\n    # Incremental value over scheduled updating is more relevant than comparison\n    # with a permanently frozen model alone.\n    periodic=(adaptation_df[adaptation_df.policy==\'Periodic\'][[\'dataset\',\'drift\',\'replicate\',\'RMSE\']]\n              .rename(columns={\'RMSE\':\'periodic_RMSE\'}))\n    adaptation_df=adaptation_df.merge(periodic,on=[\'dataset\',\'drift\',\'replicate\'],how=\'left\')\n    adaptation_df[\'RMSE_difference_vs_periodic\']=adaptation_df.RMSE-adaptation_df.periodic_RMSE\n    out={\'calibration\':pd.concat(cal_tables,ignore_index=True),\'operating_results\':pd.DataFrame(rows),\n         \'adaptation\':adaptation_df,\'update_events\':pd.DataFrame(events),\n         \'timelines\':pd.DataFrame(timelines),\'natural_events\':pd.DataFrame(natural),\'disabled\':pd.DataFrame(disabled)}\n    for k,v in out.items():ctx.table(\'RQ4_\'+k,v)\n    return out\n')

### Bundled source: Uncertainty

In [ ]:
write_module('primna_esd_uncertainty.py','"""RQ5: fair calibration samples, stable scores, tie-aware decision policies."""\nfrom __future__ import annotations\nimport math\nimport numpy as np\nimport pandas as pd\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nfrom sklearn.metrics import roc_auc_score\nfrom primna_esd_runtime import seed,metrics,ci,block_sums\nfrom primna_esd_data import DATASETS,scenario\nfrom primna_esd_models import ML,MissingAware,fit_logged\n\nCONDITIONS=[\'Clean\',\'MCAR 20%\',\'Gaussian 10%\',\'Abrupt drift\']\nMETHODS=[\'Global split conformal\',\'Locally scaled conformal\']\n\n\ndef conformal_quantile(scores,coverage):\n    x=np.asarray(scores,dtype=float);x=x[np.isfinite(x)];n=len(x)\n    if not n:raise ValueError(\'No finite calibration scores\')\n    k=math.ceil((n+1)*coverage)\n    return float(np.sort(x)[k-1]) if k<=n else float(\'inf\')\n\n\ndef canonical_score(score,step):\n    s=np.asarray(score,dtype=float)\n    if not np.isfinite(s).all():raise ValueError(\'Nonfinite width score; calibration sample too small or numerical error.\')\n    return np.round(s/step)*step\n\n\ndef interval_metrics(y,p,half,score,threshold,high_threshold):\n    y=np.asarray(y);p=np.asarray(p);half=np.asarray(half);score=np.asarray(score)\n    valid=np.isfinite(y);err=abs(y[valid]-p[valid]);hh=half[valid];ss=score[valid]\n    accepted=ss<=threshold;high=err>high_threshold\n    constant=np.unique(ss).size==1\n    auc=(.5 if constant else float(roc_auc_score(high,ss))) if len(np.unique(high))==2 else np.nan\n    return {\'n\':len(err),\'coverage\':float(np.mean(err<=hh)),\'mean_width\':float(np.mean(2*hh)),\n       \'acceptance\':float(accepted.mean()),\'accepted_RMSE\':float(np.sqrt(np.mean(err[accepted]**2))) if accepted.any() else np.nan,\n       \'joint_false_reassurance\':float(np.mean(high&accepted)),\n       \'high_error_given_accepted\':float(high[accepted].mean()) if accepted.any() else np.nan,\n       \'high_error_AUROC\':auc,\'score_constant\':constant,\'unique_scores\':int(np.unique(ss).size)}\n\n\ndef attainable_curve(y,p,score):\n    valid=np.isfinite(y);y=np.asarray(y)[valid];p=np.asarray(p)[valid];s=np.asarray(score)[valid]\n    thresholds=np.unique(np.quantile(s,np.linspace(0,1,21),method=\'higher\'))\n    rows=[]\n    for th in thresholds:\n        take=s<=th;rows.append({\'accepted_fraction\':take.mean(),\'selective_RMSE\':metrics(y[take],p[take])[\'RMSE\'],\n                               \'threshold\':th,\'n_accepted\':int(take.sum()),\'kind\':\'attainable tied-score threshold\'})\n    # Expected squared error of uniform random selection is the full-cohort MSE.\n    full=metrics(y,p)[\'RMSE\']\n    for f in np.linspace(.1,1,10):rows.append({\'accepted_fraction\':f,\'selective_RMSE\':full,\'threshold\':np.nan,\n       \'n_accepted\':int(round(f*len(y))),\'kind\':\'random selection: sqrt(expected MSE)\'})\n    return pd.DataFrame(rows)\n\n\ndef decision_metrics(y,p,half,event_threshold,fn_cost,abstain_cost,review_capacity=1.0):\n    """Three-action monitoring policy: alert, no alert, or abstain/review.\n\n    The event threshold is frozen from policy outcomes. Interval policies alert\n    only when the full interval exceeds the threshold and reject only when the\n    full interval is below it. Straddling intervals abstain. Costs are a declared\n    sensitivity grid, not estimated social preferences.\n    """\n    y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float);valid=np.isfinite(y)\n    truth=y>=event_threshold\n    if half is None:\n        action=(p>=event_threshold).astype(int);abstain=np.zeros(len(p),dtype=bool)\n    else:\n        h=np.asarray(half,dtype=float);action=np.full(len(p),-1,dtype=int)\n        action[p-h>=event_threshold]=1;action[p+h<event_threshold]=0;abstain=action<0\n        # Review capacity is a fraction of valid origins.  If more intervals\n        # straddle the threshold than can be reviewed, prioritize the largest\n        # threshold overlap and fall back to the point decision for the rest.\n        candidates=np.flatnonzero(abstain & valid);limit=int(np.floor(review_capacity*valid.sum()))\n        if len(candidates)>limit:\n            ambiguity=h[candidates]-np.abs(p[candidates]-event_threshold)\n            reviewed=set(candidates[np.argsort(-ambiguity)[:limit]].tolist())\n            overflow=np.array([i for i in candidates if i not in reviewed],dtype=int)\n            action[overflow]=(p[overflow]>=event_threshold).astype(int);abstain=action<0\n    fp=(action==1)&(~truth)&valid;fn=(action==0)&truth&valid;ab=abstain&valid\n    n=max(1,int(valid.sum()));npos=max(1,int((truth&valid).sum()));nneg=max(1,int((~truth&valid).sum()))\n    return {\'expected_cost\':float((fp.sum()+fn_cost*fn.sum()+abstain_cost*ab.sum())/n),\n      \'false_negative_rate\':float(fn.sum()/npos),\'false_positive_rate\':float(fp.sum()/nneg),\n      \'abstention_rate\':float(ab.sum()/n),\'review_capacity\':float(review_capacity),\n      \'capacity_binding\':bool((abstain&valid).sum()>=int(np.floor(review_capacity*valid.sum())) and review_capacity<1),\n      \'alert_rate\':float(((action==1)&valid).sum()/n)}\n\n\ndef mean_matrix_ci(V,mask,block,B,rng):\n    V=np.asarray(V);mask=np.asarray(mask,dtype=bool)\n    ss=np.array([block_sums(v,mask,block)[0] for v in V]);nn=block_sums(np.zeros(len(mask)),mask,block)[1]\n    vals=[]\n    for _ in range(B):\n        r=rng.integers(0,len(V),len(V));b=rng.integers(0,len(nn),len(nn));den=nn[b].sum()\n        vals.append(ss[r][:,b].sum(axis=1).mean()/den if den else np.nan)\n    return float(ss.sum(axis=1).mean()/nn.sum()),*ci(vals)\n\n\ndef calibrate_intervals(ctx,b,fit,model_name):\n    model=fit[\'models\'][model_name];ca=b[\'splits\'][\'calibration\'];gap=max(b[\'spec\'][\'lags\']+b[\'spec\'][\'windows\'])+1\n    half=len(ca)//2;ia=ca[:half-gap];ib=ca[half:]\n    ia=ia[b[\'y\'].loc[ia].notna()];ib=ib[b[\'y\'].loc[ib].notna()]\n    if min(len(ia),len(ib))<30:raise ValueError(\'Calibration subdivisions too short.\')\n    xa=b[\'X\'].loc[ia];ea=abs(b[\'y\'].loc[ia].to_numpy()-model.predict(xa))\n    scale=Pipeline([(\'impute\',MissingAware()),(\'model\',HistGradientBoostingRegressor(loss=\'absolute_error\',\n        max_iter=150 if ctx.cfg[\'profile\']==\'paper\' else 45,max_leaf_nodes=15,learning_rate=.05,\n        early_stopping=False,random_state=seed(b[\'dataset\'],model_name,\'scale\')))])\n    scale,ws,_=fit_logged(ctx,scale,xa,ea,\'uncertainty scale model\')\n    floor=max(.10*float(np.median(ea)),1e-6*float(np.std(b[\'y\'].loc[ia])),1e-8)\n    xb=b[\'X\'].loc[ib];eb=abs(b[\'y\'].loc[ib].to_numpy()-model.predict(xb));sb=np.maximum(scale.predict(xb),floor)\n    qg=conformal_quantile(eb,ctx.cfg[\'coverage\']);ql=conformal_quantile(eb/sb,ctx.cfg[\'coverage\'])\n    pol=b[\'splits\'][\'policy\'];xp=b[\'X\'].loc[pol];ep=abs(b[\'y\'].loc[pol].to_numpy()-model.predict(xp));pv=np.isfinite(ep)\n    high=float(np.quantile(ep[pv],.9));widths={\'Global split conformal\':np.full(len(pol),2*qg),\n        \'Locally scaled conformal\':2*ql*np.maximum(scale.predict(xp),floor)}\n    policies={}\n    for method,u in widths.items():\n        step=max(1,float(np.median(u)))*1e-10;s=canonical_score(u,step)\n        th=float(np.quantile(s,ctx.cfg[\'acceptance_target\'],method=\'higher\'))\n        edges=np.unique(np.quantile(s,np.linspace(0,1,9),method=\'higher\'))\n        policies[method]={\'score_step\':step,\'threshold\':th,\'policy_realized_acceptance\':float(np.mean(s<=th)),\n                         \'constant_score\':len(np.unique(s))==1,\'bin_edges\':edges[1:-1].tolist()}\n    return {\'scale_model\':scale,\'scale_floor\':floor,\'global_q\':qg,\'local_q\':ql,\'policies\':policies,\n       \'high_error_threshold\':high,\'scale_fit_n\':len(ia),\'common_conformal_n\':len(ib),\n       \'scale_fit_last\':str(ia[-1]),\'conformal_first\':str(ib[0]),\'warnings\':ws}\n\n\ndef uncertainty_suite(ctx,bundles,fitted):\n    parameter_rows=[];raw_metrics=[];summaries=[];curves=[];reliability=[];decisions=[]\n    for ds in DATASETS:\n        b=bundles[ds];idx=b[\'splits\'][\'test\'];y=b[\'y\'].loc[idx].to_numpy();valid=np.isfinite(y)\n        for model_name in ML:\n            fit=fitted[ds];model=fit[\'models\'][model_name]\n            ca=ctx.task(\'RQ5CAL\',f\'RQ5 calibration | {ds} | {model_name}\',{\'data\':b[\'data_key\'],\'model\':model_name},\n                        lambda model_name=model_name:calibrate_intervals(ctx,b,fit,model_name))\n            for method in METHODS:\n                parameter_rows.append(dict(dataset=ds,model=model_name,method=method,q=ca[\'global_q\'] if method==METHODS[0] else ca[\'local_q\'],\n                    scale_floor=ca[\'scale_floor\'],scale_fit_n=ca[\'scale_fit_n\'],common_conformal_n=ca[\'common_conformal_n\'],\n                    high_error_threshold=ca[\'high_error_threshold\'],**{k:v for k,v in ca[\'policies\'][method].items() if k!=\'bin_edges\'}))\n            for condition in CONDITIONS:\n                R=1 if condition==\'Clean\' else ctx.cfg[\'uncertainty_reps\'];collected={m:[] for m in METHODS}\n                for rep in range(R):\n                    label=f\'RQ5 | {ds} | {model_name} | {condition} | {rep+1}/{R}\'\n                    def work(condition=condition,rep=rep):\n                        X,_,_=scenario(b,condition,rep);p=model.predict(X);sc=np.maximum(ca[\'scale_model\'].predict(X),ca[\'scale_floor\'])\n                        return {\'prediction\':p,\'global_half\':np.full(len(p),ca[\'global_q\']),\'local_half\':ca[\'local_q\']*sc}\n                    res=ctx.task(\'RQ5\',label,{\'data\':b[\'data_key\'],\'model\':model_name,\'condition\':condition,\'rep\':rep},work)\n                    p=res[\'prediction\']\n                    policy_y=b[\'y\'].loc[b[\'splits\'][\'policy\']].dropna().to_numpy()\n                    for event_q in ctx.cfg[\'decision_event_quantiles\']:\n                        event_threshold=float(np.quantile(policy_y,event_q))\n                        for fn_cost in ctx.cfg[\'decision_fn_cost_ratios\']:\n                            for abstain_cost in ctx.cfg[\'decision_abstain_costs\']:\n                                for capacity in ctx.cfg[\'decision_review_capacities\']:\n                                    decisions.append(dict(dataset=ds,model=model_name,condition=condition,replicate=rep,\n                                      method=\'Point forecast\',event_quantile=event_q,event_threshold=event_threshold,\n                                      false_negative_cost_ratio=fn_cost,abstention_cost=abstain_cost,\n                                      **decision_metrics(y,p,None,event_threshold,fn_cost,abstain_cost,capacity)))\n                    for method,hkey in zip(METHODS,[\'global_half\',\'local_half\']):\n                        hh=res[hkey];policy=ca[\'policies\'][method];u=canonical_score(2*hh,policy[\'score_step\'])\n                        met=interval_metrics(y,p,hh,u,policy[\'threshold\'],ca[\'high_error_threshold\'])\n                        alpha=1-ctx.cfg[\'coverage\'];err=abs(y-p);wink=2*hh+2/alpha*np.maximum(0,err-hh)\n                        met[\'Winkler\']=float(np.mean(wink[valid]));met[\'RMSE\']=metrics(y,p)[\'RMSE\']\n                        raw_metrics.append(dict(dataset=ds,model=model_name,condition=condition,replicate=rep,method=method,**met))\n                        for event_q in ctx.cfg[\'decision_event_quantiles\']:\n                            event_threshold=float(np.quantile(policy_y,event_q))\n                            for fn_cost in ctx.cfg[\'decision_fn_cost_ratios\']:\n                                for abstain_cost in ctx.cfg[\'decision_abstain_costs\']:\n                                    for capacity in ctx.cfg[\'decision_review_capacities\']:\n                                        decisions.append(dict(dataset=ds,model=model_name,condition=condition,replicate=rep,\n                                          method=method,event_quantile=event_q,event_threshold=event_threshold,\n                                          false_negative_cost_ratio=fn_cost,abstention_cost=abstain_cost,\n                                          **decision_metrics(y,p,hh,event_threshold,fn_cost,abstain_cost,capacity)))\n                        cr=attainable_curve(y,p,u).assign(dataset=ds,model=model_name,condition=condition,replicate=rep,method=method);curves.append(cr)\n                        binid=np.searchsorted(np.asarray(policy[\'bin_edges\']),u,side=\'right\')\n                        for bn in np.unique(binid):\n                            take=(binid==bn)&valid\n                            if take.any():reliability.append(dict(dataset=ds,model=model_name,condition=condition,replicate=rep,method=method,\n                              policy_bin=int(bn),n=int(take.sum()),sum_width=float((2*hh[take]).sum()),sum_absolute_error=float(err[take].sum())))\n                        collected[method].append({\'coverage\':(err<=hh).astype(float),\'width\':2*hh,\'winkler\':wink,\n                            \'accepted\':(u<=policy[\'threshold\']).astype(float),\'error\':err,\'score\':u})\n                        # Portable row-level arrays, independently readable without unpickling models.\n                        outdir=ctx.root/\'results\'/\'RQ5_rows\';outdir.mkdir(exist_ok=True)\n                        fname=f\'{seed(ds,model_name,condition,rep,method):08x}.npz\'\n                        np.savez_compressed(outdir/fname,origin_ns=idx.asi8,y_true=y,prediction=p,half_width=hh,\n                            uncertainty_score=u,accepted=u<=policy[\'threshold\'],dataset=ds,model=model_name,condition=condition,method=method,replicate=rep)\n                for method,rec in collected.items():\n                    ans={\'dataset\':ds,\'model\':model_name,\'condition\':condition,\'method\':method,\'replicates\':len(rec),\'n_scored\':int(valid.sum())}\n                    for field in [\'coverage\',\'width\',\'winkler\',\'accepted\']:\n                        pt,lo,hi=mean_matrix_ci([r[field] for r in rec],valid,b[\'spec\'][\'period\'],ctx.cfg[\'bootstrap\'],np.random.default_rng(seed(ds,model_name,condition,method,field)))\n                        ans[field]=pt;ans[field+\'_low\']=lo;ans[field+\'_high\']=hi\n                    m=pd.DataFrame(raw_metrics);sub=m[(m.dataset==ds)&(m.model==model_name)&(m.condition==condition)&(m.method==method)]\n                    for f in [\'high_error_AUROC\',\'joint_false_reassurance\',\'high_error_given_accepted\',\'accepted_RMSE\',\'score_constant\']:\n                        ans[f]=sub[f].mean()\n                    summaries.append(ans)\n    result={\'parameters\':pd.DataFrame(parameter_rows),\'replicate_metrics\':pd.DataFrame(raw_metrics),\n      \'summary\':pd.DataFrame(summaries),\'curves\':pd.concat(curves,ignore_index=True),\n      \'reliability_bins\':pd.DataFrame(reliability),\'decision_utility\':pd.DataFrame(decisions)}\n    for k,v in result.items():ctx.table(\'RQ5_\'+k,v)\n    return result\n')

### Bundled source: Compute

In [ ]:
write_module('primna_esd_compute.py','"""RQ6: repeatable runtime/memory and explicit energy provenance; no manual watts."""\nfrom __future__ import annotations\nimport gc,os,platform,threading,time,warnings\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport psutil\nfrom threadpoolctl import threadpool_limits\nfrom primna_esd_runtime import seed,atomic_json,digest\nfrom primna_esd_data import DATASETS\nfrom primna_esd_models import ML,factory\n\n\ndef rapl_packages():\n    found={}\n    root=Path(\'/sys/class/powercap\')\n    if not root.exists():return []\n    for p in root.rglob(\'energy_uj\'):\n        try:\n            name=(p.parent/\'name\').read_text().strip()\n            if not name.startswith(\'package-\'):continue\n            value=float(p.read_text());maximum=float((p.parent/\'max_energy_range_uj\').read_text())\n            found[str(p.resolve())]={\'path\':p,\'max\':maximum,\'name\':name}\n        except (OSError,ValueError):continue\n    return list(found.values())\n\n\nclass Probe:\n    def __init__(self,ctx,label):\n        self.ctx=ctx;self.label=label;self.process=psutil.Process();self.start_rss=self.process.memory_info().rss\n        self.peak=self.start_rss;self.packages=rapl_packages();self.joules=0.;self.rapl_error=None;self.previous={}\n        self.stop_event=threading.Event();self.tracker=None;self.energy_kind=\'unavailable\';self.estimate=np.nan\n    def _sample(self):\n        try:self.peak=max(self.peak,self.process.memory_info().rss)\n        except psutil.Error:pass\n        for p in self.packages:\n            try:\n                key=str(p[\'path\']);v=float(p[\'path\'].read_text())\n                if key in self.previous:self.joules+=((v-self.previous[key])%p[\'max\'])/1e6\n                self.previous[key]=v\n            except (OSError,ValueError) as exc:self.rapl_error=str(exc)\n    def start(self):\n        if self.packages:self.energy_kind=\'counter: CPU packages (host scope, not whole machine)\'\n        elif self.ctx.cfg.get(\'try_codecarbon\',True):\n            try:\n                from codecarbon import OfflineEmissionsTracker\n                target=self.ctx.root/\'logs\'/\'energy\';target.mkdir(exist_ok=True)\n                self.tracker=OfflineEmissionsTracker(country_iso_code=\'USA\',project_name=self.label,\n                    output_dir=str(target),output_file=f\'{seed(self.label)}.csv\',measure_power_secs=1,\n                    save_to_file=True,save_to_api=False,log_level=\'error\',tracking_mode=\'process\')\n                self.tracker.start();self.energy_kind=\'CodeCarbon estimated/mixed CPU+RAM; not direct total measurement\'\n            except Exception as exc:\n                self.ctx.log(\'Energy telemetry unavailable: %s\',exc);self.tracker=None\n        self._sample()\n        def loop():\n            while not self.stop_event.wait(.2):self._sample()\n        self.thread=threading.Thread(target=loop,daemon=True);self.thread.start()\n    def stop(self):\n        self.stop_event.set();self.thread.join();self._sample();e=np.nan\n        if self.packages and self.rapl_error is None:e=self.joules/3.6e6\n        elif self.tracker is not None:\n            try:\n                self.tracker.stop();dat=self.tracker.final_emissions_data\n                e=float(dat.energy_consumed) # stop() returns emissions, NOT kWh\n            except Exception as exc:self.ctx.log(\'Could not read energy telemetry: %s\',exc)\n        if not np.isfinite(e) or e<=0:\n            e=np.nan;self.energy_kind=\'unavailable: no valid positive energy readout\'\n        return {\'energy_kWh_total\':e,\'energy_kind\':self.energy_kind,\'RSS_peak_MB\':self.peak/2**20,\n                \'RSS_delta_MB\':max(0,self.peak-self.start_rss)/2**20,\'RAPL_error\':self.rapl_error}\n\n\ndef benchmark(ctx,label,fn):\n    gc.collect();probe=Probe(ctx,label);probe.start();cost=[];warnings_seen=[];start=time.perf_counter()\n    try:\n        with warnings.catch_warnings(record=True) as ws,threadpool_limits(limits=ctx.cfg[\'threads\']):\n            warnings.simplefilter(\'always\')\n            while time.perf_counter()-start<ctx.cfg[\'benchmark_seconds\'] or not cost:\n                t=time.perf_counter();result=fn();cost.append(time.perf_counter()-t)\n                del result\n            warnings_seen=[str(w.category.__name__)+\': \'+str(w.message) for w in ws]\n    finally:measure=probe.stop()\n    for w in set(warnings_seen):ctx.log(\'BENCHMARK WARNING %s: %s\',label,w)\n    return dict(measure,seconds_per_operation=float(np.mean(cost)),operations=len(cost),\n      tracking_elapsed_seconds=time.perf_counter()-start,energy_kWh_per_operation=measure[\'energy_kWh_total\']/len(cost),\n      warnings=\' | \'.join(sorted(set(warnings_seen))))\n\n\ndef compute_suite(ctx,bundles,fitted):\n    rows=[];machine={\'platform\':platform.platform(),\'processor\':platform.processor(),\'logical_CPUs\':os.cpu_count(),\n      \'threads_used\':ctx.cfg[\'threads\'],\'physical_memory_bytes\':psutil.virtual_memory().total,\n      \'RAPL_packages\':[str(x[\'path\']) for x in rapl_packages()],\n      \'energy_scope_note\':\'CodeCarbon output is estimated/mixed unless proven otherwise; RAPL covers host CPU package domains only. No carbon-emissions or geographic claims.\',\n      \'comparison_scope\':\'one execution host; fixed model seed and parameters; randomized benchmark order; warmups excluded\'}\n    try:machine[\'host_boot_fingerprint\']=digest(Path(\'/proc/sys/kernel/random/boot_id\').read_text().strip())\n    except OSError:machine[\'host_boot_fingerprint\']=\'unavailable\'\n    atomic_json(ctx.root/\'manifests\'/\'hardware_and_energy_provenance.json\',machine)\n    for ds in DATASETS:\n        b=bundles[ds];fit=fitted[ds];ix=b[\'splits\'][\'development\'];ix=ix[b[\'y\'].loc[ix].notna()]\n        X=b[\'X\'].loc[ix];y=b[\'y\'].loc[ix];xt=b[\'X\'].loc[b[\'splits\'][\'test\']]\n        # Warm up each algorithm outside timed measurements once per session.\n        for m in ML:\n            with ctx.heartbeat(\'RQ6 untimed warmup \'+ds+\' \'+m),threadpool_limits(limits=ctx.cfg[\'threads\']):\n                warm=factory(m,fit[\'params\'][m],seed(ctx.cfg[\'seed\'],ds,m,\'cost\'),ctx.cfg[\'threads\']);warm.fit(X,y);warm.predict(xt.iloc[:100]);del warm\n        for rep in range(ctx.cfg[\'compute_reps\']):\n            order=np.random.default_rng(seed(ds,\'compute_order\',rep)).permutation([(m,s) for m in ML for s in [\'Training\',\'Inference / 1000\']])\n            for m,stage in order:\n                label=f\'RQ6 | {ds} | {m} | {stage} | repeat {rep+1}/{ctx.cfg["compute_reps"]}\'\n                def work(m=m,stage=stage,label=label):\n                    if stage==\'Training\':\n                        def fn():\n                            est=factory(m,fit[\'params\'][m],seed(ctx.cfg[\'seed\'],ds,m,\'cost\'),ctx.cfg[\'threads\']);est.fit(X,y);return est\n                    else:\n                        def fn():return fit[\'models\'][m].predict(xt)\n                    rr=benchmark(ctx,label,fn)\n                    factor=1000/len(xt) if stage!=\'Training\' else 1.\n                    rr[\'seconds_per_operation\']*=factor;rr[\'energy_kWh_per_operation\']*=factor\n                    return rr\n                # Hardware is part of this task identity; cannot reuse cost readings on another host.\n                out=ctx.task(\'RQ6\',label,{\'data\':b[\'data_key\'],\'model\':m,\'stage\':stage,\'rep\':rep,\'hardware\':machine},work)\n                rows.append(dict(dataset=ds,model=m,stage=stage,replicate=rep,**out))\n    result={\'measurements\':pd.DataFrame(rows),\'hardware\':machine};ctx.table(\'RQ6_measurements\',result[\'measurements\'])\n    return result\n')

### Bundled source: Figures

In [ ]:
write_module('primna_esd_figures.py','"""Publication figures: all plotted data and captions are exported alongside PDFs."""\nfrom __future__ import annotations\nimport math,textwrap\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.backends.backend_pdf import PdfPages\nfrom matplotlib.lines import Line2D\nfrom primna_esd_runtime import seed,ci,atomic_json\nfrom primna_esd_data import DATASETS,MISSING,FAULTS,SPECS\nfrom primna_esd_models import MODELS,ML,SHORT\nfrom primna_esd_uncertainty import METHODS,CONDITIONS\n\nCOLORS=dict(zip(MODELS,[\'#666666\',\'#A6A6A6\',\'#0072B2\',\'#009E73\',\'#D55E00\',\'#CC79A7\']))\nSHORT_DS={\'Air Quality\':\'Air\',\'Appliances Energy\':\'Energy\'}\nplt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':10,\'axes.titlesize\':11,\n \'axes.labelsize\':10,\'xtick.labelsize\':9,\'ytick.labelsize\':9,\'legend.fontsize\':8,\n \'pdf.fonttype\':42,\'ps.fonttype\':42,\'axes.spines.top\':False,\'axes.spines.right\':False})\n\n\ndef savefig(ctx,fig,stem,title,caption,data):\n    fig.suptitle(title,fontsize=12,fontweight=\'bold\')\n    if ctx.cfg[\'profile\']!=\'paper\':fig.text(.5,-.115,\'PILOT - engineering validation only\',ha=\'center\',fontsize=8)\n    paths={\'pdf\':ctx.root/\'figures_pdf\'/(stem+\'.pdf\'),\'png\':ctx.root/\'figures_png\'/(stem+\'.png\')}\n    dp=ctx.root/\'plot_data\'/(stem+\'.csv\');data.to_csv(dp,index=False,float_format=\'%.12g\')\n    try:\n        with ctx.substage(\'save PDF | \'+stem):\n            fig.savefig(paths[\'pdf\'],bbox_inches=\'tight\',pad_inches=.16,metadata={\'Title\':title,\'Subject\':caption})\n        with ctx.substage(\'save PNG | \'+stem,dpi=ctx.cfg[\'png_dpi\']):\n            fig.savefig(paths[\'png\'],bbox_inches=\'tight\',pad_inches=.16,dpi=ctx.cfg[\'png_dpi\'])\n    finally:plt.close(fig)\n    return {\'stem\':stem,\'type\':\'figure\',\'pdf\':str(paths[\'pdf\'].relative_to(ctx.root)),\'plot_data\':str(dp.relative_to(ctx.root)),\n            \'title\':title,\'caption\':caption}\n\n\ndef table(ctx,stem,title,df,caption):\n    df.to_csv(ctx.root/\'tables\'/(stem+\'.csv\'),index=False,float_format=\'%.12g\')\n    try:latex=df.to_latex(index=False,float_format=lambda x:f\'{x:.4g}\',escape=True)\n    except ImportError:\n        # Pandas delegates LaTeX to optional Jinja2 in recent releases. Keep the\n        # notebook self-contained when that presentation dependency is absent.\n        def esc(value):\n            s=\'NA\' if pd.isna(value) else f\'{value:.4g}\' if isinstance(value,(float,np.floating)) else str(value)\n            for a,b in [(\'\\\\\',\'\\\\textbackslash{}\'),(\'&\',\'\\\\&\'),(\'%\',\'\\\\%\'),(\'$\',\'\\\\$\'),(\'#\',\'\\\\#\'),(\'_\',\'\\\\_\'),(\'{\',\'\\\\{\'),(\'}\',\'\\\\}\')]:s=s.replace(a,b)\n            return s\n        cols=\'l\'*len(df.columns);lines=[\'\\\\begin{tabular}{\'+cols+\'}\',\'\\\\toprule\',\' & \'.join(map(esc,df.columns))+\' \\\\\\\\\',\'\\\\midrule\']\n        lines+=[\' & \'.join(esc(v) for v in row)+\' \\\\\\\\\' for row in df.itertuples(index=False,name=None)]\n        latex=\'\\n\'.join(lines+[\'\\\\bottomrule\',\'\\\\end{tabular}\'])\n    (ctx.root/\'tables\'/(stem+\'.tex\')).write_text(latex)\n    path=ctx.root/\'tables\'/(stem+\'.pdf\');pages=math.ceil(len(df)/12) or 1\n    with PdfPages(path) as pdf:\n        for page in range(pages):\n            d=df.iloc[page*12:(page+1)*12];fig,ax=plt.subplots(figsize=(11.69,8.27));ax.axis(\'off\')\n            fig.suptitle(title+f\'  |  {page+1}/{pages}\',fontsize=14,y=.96)\n            def fmt(v):\n                if pd.isna(v):return \'NA\'\n                if isinstance(v,(float,np.floating)):return f\'{v:.3g}\'\n                return \'\\n\'.join(textwrap.wrap(str(v),width=max(14,112//max(1,len(df.columns)))))\n            headers=[\'\\n\'.join(textwrap.wrap(str(c),width=13)) for c in df.columns]\n            cells=[[fmt(v) for v in row] for row in d.itertuples(index=False,name=None)]\n            tab=ax.table(cellText=cells,colLabels=headers,cellLoc=\'center\',loc=\'upper center\',bbox=[0,.20,1,.72])\n            tab.auto_set_font_size(False);tab.set_fontsize(9)\n            for (r,c),cell in tab.get_celld().items():\n                cell.set_linewidth(.3);cell.PAD=.08\n                if r==0:cell.set_facecolor(\'#E9EDF2\');cell.set_text_props(weight=\'bold\');cell.set_height(.075)\n                elif r%2==0:cell.set_facecolor(\'#F7F8FA\')\n            fig.text(.05,.085,textwrap.fill(caption,145),ha=\'left\',va=\'bottom\',fontsize=9)\n            status=(\'Core cohorts are retrospective; independent external confirmation is reported separately.\'\n                    if ctx.cfg.get(\'run_external_confirmation\') else \'Retrospective evaluation only; no untouched confirmatory claim.\')\n            fig.text(.05,.035,status+\' NA denotes undefined or unavailable, never zero.\',fontsize=8)\n            fig.subplots_adjust(left=.04,right=.96,top=.91,bottom=.04);pdf.savefig(fig);plt.close(fig)\n    return {\'stem\':stem,\'type\':\'table\',\'pdf\':str(path.relative_to(ctx.root)),\'plot_data\':\'tables/\'+stem+\'.csv\',\'title\':title,\'caption\':caption}\n\n\ndef newaxes(nrows=1,ncols=2,height=4.6):return plt.subplots(nrows,ncols,figsize=(10,height),squeeze=False,layout=\'constrained\')\n\n\ndef legend_models(fig,models=MODELS):\n    fig.legend(handles=[Line2D([0],[0],color=COLORS[m],marker=\'o\',label=SHORT[m]) for m in models],\n               loc=\'lower center\',bbox_to_anchor=(.5,-.035),ncol=len(models),frameon=False)\n\n\ndef heatmaps(df,value,rows,columns,title):\n    fig,axes=newaxes(height=4.4);v=df[value].to_numpy();v=v[np.isfinite(v)]\n    lo=min(0,float(v.min())) if len(v) else 0;hi=max(.01,float(v.max())) if len(v) else 1\n    for ax,ds in zip(axes.flat,DATASETS):\n        mat=df[df.dataset==ds].pivot_table(index=\'model\',columns=columns[0],values=value).reindex(index=rows,columns=columns[1])\n        im=ax.imshow(mat,aspect=\'auto\',cmap=\'viridis\',vmin=lo,vmax=hi)\n        ax.set_xticks(range(len(mat.columns)),mat.columns,rotation=35,ha=\'right\');ax.set_yticks(range(len(rows)),[SHORT[x] for x in rows]);ax.set_title(ds)\n        for i in range(len(rows)):\n            for j in range(len(mat.columns)):\n                val=mat.iloc[i,j];ax.text(j,i,\'NA\' if not np.isfinite(val) else f\'{val:.2f}\',ha=\'center\',va=\'center\',fontsize=8,\n                  color=\'white\' if np.isfinite(val) and val<(lo+hi)/2 else \'black\')\n    fig.colorbar(im,ax=list(axes.flat),shrink=.75,label=title)\n    return fig\n\n\ndef stress_figures(ctx,rq,result,fitted):\n    outputs=[];d=result[\'summary\'];kind=\'missingness\' if rq==1 else \'sensor_faults\'\n    mechanisms=MISSING if rq==1 else FAULTS;show=mechanisms[:3] if rq==1 else [\'Gaussian\',\'Calibration bias\',\'Spikes\']\n    fig,axes=newaxes(2,3,7.3)\n    for row,ds in enumerate(DATASETS):\n        for col,mech in enumerate(show):\n            ax=axes[row,col]\n            for m in MODELS:\n                g=d[(d.dataset==ds)&(d.mechanism==mech)&(d.model==m)].sort_values(\'severity\')\n                ax.plot(g.severity,g.relative_change*100,label=SHORT[m],color=COLORS[m],marker=\'o\',markersize=3)\n                ax.fill_between(g.severity,g.relative_low*100,g.relative_high*100,color=COLORS[m],alpha=.10)\n            ax.axhline(0,color=\'black\',linewidth=.6);ax.set_title(ds+\'\\n\'+mech);ax.set_xlabel(\'Nominal severity\');ax.set_ylabel(\'RMSE change (%)\');ax.grid(alpha=.2)\n    legend_models(fig)\n    outputs.append(savefig(ctx,fig,f\'RQ{rq}_F1_response_curves\',f\'RQ{rq}. Severity response, with paired uncertainty\',\n      \'Curves are root mean squared errors pooled over corruption replicas relative to the same clean model. Bands: 95% replicate plus clock-day block bootstrap on the fixed historical test period. Nominal severity has mechanism-specific meaning. Datasets are not pooled.\',d))\n    a=result[\'audc\'];fig=heatmaps(a[a.mechanism!=\'Descriptive equal-weight scenario mean\'],\'AUDC\',MODELS,(\'mechanism\',mechanisms),\'Normalized area under relative-degradation curve\')\n    outputs.append(savefig(ctx,fig,f\'RQ{rq}_F2_vulnerability_fingerprint\',f\'RQ{rq}. Mechanism-specific vulnerability\',\n      \'Both panels use an identical color normalization. Negative areas are retained and indicate lower error in this stress setting, not universal superiority. All models, including naive baselines, are shown.\',a))\n    if rq==1:\n        data=result[\'injection_audit\'].groupby([\'dataset\',\'mechanism\',\'severity\'],as_index=False).agg(achieved=(\'achieved_new_missing_fraction\',\'mean\'),low=(\'achieved_new_missing_fraction\',lambda x:x.quantile(.025)),high=(\'achieved_new_missing_fraction\',lambda x:x.quantile(.975)))\n        fig,axes=newaxes()\n        for ax,ds in zip(axes.flat,DATASETS):\n            for mech,g in data[data.dataset==ds].groupby(\'mechanism\'):\n                ax.plot(g.severity*100,g.achieved*100,\'o-\',label=mech);ax.fill_between(g.severity*100,g.low*100,g.high*100,alpha=.10)\n            ax.plot([0,30],[0,30],\':\',color=\'black\',label=\'Nominal = achieved\');ax.set_title(ds);ax.set_xlabel(\'Nominal missingness (%)\');ax.set_ylabel(\'Newly missing observed cells (%)\');ax.grid(alpha=.2)\n        axes[0,1].legend(loc=\'upper left\',bbox_to_anchor=(1.02,1),frameon=False)\n        outputs.append(savefig(ctx,fig,\'RQ1_F3_injection_manipulation_check\',\'RQ1. Was the requested missingness actually achieved?\',\n          \'Denominator is originally observed raw cells over every clock origin, not only origins with an observed future label. Channel rounding, protected MAR context, and conditional risk change realized rates. Bands are empirical corruption-replicate 2.5-97.5 percentiles, not confidence intervals.\',data))\n    else:\n        data=result[\'tails\'].groupby([\'dataset\',\'model\',\'mechanism\'],as_index=False).agg(tail95=(\'tail95\',\'mean\'),MAE=(\'MAE\',\'mean\'))\n        fig,axes=newaxes()\n        for ax,ds in zip(axes.flat,DATASETS):\n            for m in MODELS:\n                g=data[(data.dataset==ds)&(data.model==m)];ax.scatter(g.MAE,g.tail95,label=SHORT[m],color=COLORS[m],s=35)\n            ax.set_xlabel(\'Mean absolute error (\'+SPECS[ds][\'unit\']+\')\');ax.set_ylabel(\'95th-percentile absolute error\');ax.set_title(ds);ax.grid(alpha=.2)\n        legend_models(fig)\n        outputs.append(savefig(ctx,fig,\'RQ2_F3_tail_error_exposure\',\'RQ2. Typical error and tail-error exposure are different\',\n          \'Each point is a model/fault combination at the largest registered severity, averaged over replicas. This diagnostic distinguishes typical error from the tail of the error distribution; it is not an uncertainty interval or a worst-case guarantee.\',data))\n    clean=pd.read_csv(ctx.root/\'results\'/\'P2_clean_summary.csv\').query("split == \'test\'")\n    tab=a[a.mechanism==\'Descriptive equal-weight scenario mean\'].merge(clean[[\'dataset\',\'model\',\'RMSE\']],on=[\'dataset\',\'model\'])\n    worst=d.groupby([\'dataset\',\'model\'],as_index=False).relative_change.max().rename(columns={\'relative_change\':\'Worst scenario change\'})\n    tab=tab.merge(worst,on=[\'dataset\',\'model\'])[[\'dataset\',\'model\',\'RMSE\',\'AUDC\',\'AUDC_low\',\'AUDC_high\',\'Worst scenario change\']]\n    tab.columns=[\'Dataset\',\'Model\',\'Clean RMSE\',\'Descriptive mean AUDC\',\'AUDC 95% low\',\'AUDC 95% high\',\'Worst scenario change\']\n    tab[\'Model\']=tab.Model.replace(SHORT)\n    outputs.append(table(ctx,f\'RQ{rq}_T1_accuracy_and_degradation\',f\'RQ{rq}. Absolute accuracy and relative degradation\',tab,\n       \'Equal weights across the prespecified mechanisms, not learned weights. AUDC is normalized by the maximum severity. Worst scenario is the maximum pooled relative RMSE change. Clean RMSE units differ by dataset: Air mg/m3; Energy Wh per 10 minutes. Relative changes and AUDC are fractions (0.10 = 10%). Detailed paired block-sign tests and Holm adjustments are separate result files.\'))\n    return outputs\n\n\ndef explanation_figures(ctx,result,fitted):\n    out=[];d=result[\'stability\'];p=result[\'importance\']\n    conditions=list(dict.fromkeys(d.scenario));a=d.groupby([\'dataset\',\'model\',\'scenario\'],as_index=False).Spearman.mean()\n    fig,axes=newaxes(height=4.8)\n    for ax,ds in zip(axes.flat,DATASETS):\n        mat=a[a.dataset==ds].pivot(index=\'model\',columns=\'scenario\',values=\'Spearman\').reindex(index=ML,columns=conditions)\n        im=ax.imshow(mat,vmin=-1,vmax=1,aspect=\'auto\',cmap=\'coolwarm\');ax.set_title(ds)\n        ax.set_xticks(range(len(conditions)),conditions,rotation=40,ha=\'right\');ax.set_yticks(range(len(ML)),[SHORT[m] for m in ML])\n        for i in range(len(ML)):\n            for j in range(len(conditions)):ax.text(j,i,f\'{mat.iloc[i,j]:.2f}\',ha=\'center\',va=\'center\',fontsize=8)\n    fig.colorbar(im,ax=list(axes.flat),label=\'Signed-importance Spearman correlation\',shrink=.75)\n    out.append(savefig(ctx,fig,\'RQ3_F1_rank_stability\',\'RQ3. Stability compared with repeat-only explanation noise\',\n      \'Grouped permutation importance is signed. Clean repeat control changes permutation draws without corrupting the data. Other conditions use paired permutation draws. Both panels share [-1, 1]. Values are conditional on the fixed, contiguous explanation windows.\',a))\n    selected=[];fig,axes=newaxes(height=5)\n    for ax,ds in zip(axes.flat,DATASETS):\n        model=fitted[ds][\'deployment_model\'];g=p[(p.dataset==ds)&(p.model==model)&p.scenario.isin([\'Clean\',\'MCAR 20%\'])].groupby([\'scenario\',\'group\']).importance.mean().unstack(0)\n        top=g.Clean.abs().nlargest(min(8,len(g))).index;g=g.loc[top].sort_values(\'Clean\');g[\'dataset\']=ds;g[\'model\']=model;selected.append(g.reset_index())\n        yy=np.arange(len(g));ax.barh(yy-.18,g.Clean,height=.36,label=\'Clean\');ax.barh(yy+.18,g[\'MCAR 20%\'],height=.36,label=\'MCAR 20%\');ax.axvline(0,color=\'black\',lw=.6)\n        ax.set_yticks(yy,g.index);ax.set_xlabel(\'Signed increase in RMSE after permutation\');ax.set_title(ds+\' / \'+SHORT[model]);ax.legend(frameon=False)\n    out.append(savefig(ctx,fig,\'RQ3_F2_signed_importance_changes\',\'RQ3. Which evidence changes, and in which direction?\',\n      \'Deployment model is selected using development CV only. Displayed groups are selected by absolute clean importance; negative importance is preserved. All lags of a sensor are permuted together within contiguous evaluation windows. This is grouped permutation evidence, not SHAP or a causal explanation.\',pd.concat(selected)))\n    fig,axes=newaxes();markers=[\'o\',\'s\',\'^\',\'v\',\'D\',\'x\']\n    for ax,ds in zip(axes.flat,DATASETS):\n        for m in ML:\n            g=d[(d.dataset==ds)&(d.model==m)&(d.scenario!=\'Clean repeat control\')]\n            ax.scatter(g.relative_RMSE_change*100,g.Spearman,color=COLORS[m],label=SHORT[m],s=23,alpha=.6)\n        ax.axvline(0,color=\'black\',lw=.6);ax.set_ylim(-1.03,1.03);ax.set_title(ds);ax.set_xlabel(\'Prediction RMSE change (%)\');ax.set_ylabel(\'Explanation-rank stability\');ax.grid(alpha=.2)\n    legend_models(fig,ML)\n    out.append(savefig(ctx,fig,\'RQ3_F3_prediction_explanation_dissociation\',\'RQ3. Prediction and explanation can deteriorate differently\',\n       \'Each point is a prespecified corruption replica. The figure diagnoses dissociation; it is not a pooled independent-sample correlation test. Clean-repeat controls are reported separately.\',d))\n    contrasts=result[\'contrasts\'].copy();contrasts.dataset=contrasts.dataset.replace(SHORT_DS);contrasts.model=contrasts.model.replace(SHORT)\n    out.append(table(ctx,\'RQ3_T1_explanation_metrics\',\'RQ3. Degradation beyond clean-repeat explanation noise\',contrasts,\n      \'Every entry is a paired degraded-minus-clean-repeat contrast using the same model and replica. Negative rank or agreement contrasts indicate degradation beyond the explainer control. Intervals bootstrap paired replicas; p-values use paired sign flips and Holm correction within dataset/model/metric.\'))\n    return out\n\n\ndef drift_figures(ctx,result):\n    out=[];c=result[\'calibration\'];fig,axes=newaxes()\n    symbols={\'Page-Hinkley\':\'o\',\'CUSUM\':\'s\',\'EWMA\':\'^\',\'Windowed KS\':\'D\'}\n    for ax,ds in zip(axes.flat,DATASETS):\n        for method,g in c[c.dataset==ds].groupby(\'detector\'):\n            ax.scatter(g.observed_policy_far,g.observed_far_upper95,marker=symbols[method],s=32,label=method,alpha=.7)\n            z=g[g.provisional_selected];ax.scatter(z.observed_policy_far,z.observed_far_upper95,s=160,facecolors=\'none\',edgecolors=\'black\')\n            z=g[g.selected];ax.scatter(z.observed_policy_far,z.observed_far_upper95,s=45,facecolors=\'black\',edgecolors=\'black\')\n        ax.axvline(ctx.cfg[\'target_far\'],color=\'black\',ls=\'--\');ax.axhline(ctx.cfg[\'target_far\'],color=\'black\',ls=\'--\')\n        ax.set_xscale(\'symlog\',linthresh=1);ax.set_yscale(\'symlog\',linthresh=1);ax.set_title(ds)\n        ax.set_xlabel(\'Observed policy alarms / 1,000 steps\');ax.set_ylabel(\'One-sided 95% FAR upper bound / 1,000\');ax.grid(alpha=.2)\n    axes[0,1].legend(frameon=False,loc=\'upper left\',bbox_to_anchor=(1.02,1))\n    out.append(savefig(ctx,fig,\'RQ4_F1_calibration_feasibility\',\'RQ4. Feasibility is a constraint, not a best-effort label\',\n      \'Rings mark provisional candidates; filled markers require the exact one-sided event-rate upper bound to meet the target. Zero observed alarms are not treated as proof of a sufficiently low future false-alarm rate.\',c))\n    d=result[\'timelines\'];fig,axes=newaxes(2,1,6)\n    for ax,ds in zip(axes.flat,DATASETS):\n        g=d[(d.dataset==ds)&(d.kind==\'absolute_error\')].sort_values(\'index\');period=SPECS[ds][\'period\']\n        sm=g.value.rolling(period,min_periods=1).mean();ax.plot((g[\'index\']+1)/period,sm,label=\'Mean absolute error, one-day trailing window\')\n        point=g.injection_index.iloc[0]/period;ax.axvline(point,color=\'black\',ls=\'--\',label=\'Injected onset (evaluation only)\')\n        top=max(float(sm.max()),1e-6)\n        for j,(method,aa) in enumerate(d[(d.dataset==ds)&(d.kind==\'alarm\')].groupby(\'detector\')):\n            ax.scatter((aa[\'index\']+1)/period,np.full(len(aa),top*(1.06+.07*j)),marker=symbols[method],label=method,s=20)\n        ax.set_title(ds);ax.set_xlabel(\'Days from test start\');ax.set_ylabel(\'Absolute error (\'+SPECS[ds][\'unit\']+\')\');ax.grid(alpha=.2)\n        ax.legend(loc=\'upper left\',bbox_to_anchor=(1.02,1),frameon=False)\n    out.append(savefig(ctx,fig,\'RQ4_F2_alarm_timelines\',\'RQ4. Residual evolution and all alarms, including early alarms\',\n      \'Prespecified abrupt-mean replica 0. Alarm timestamps correspond to residual availability. Pre-injection alarms are not filtered out. Real chronological data have no verified natural-drift labels; these are alarms around a simulated measurement shift.\',d))\n    a=result[\'adaptation\'];fig,axes=newaxes()\n    conditions=list(dict.fromkeys(a.drift));plotrows=[]\n    for ax,ds in zip(axes.flat,DATASETS):\n        policies=[\'Periodic\',\'Random budget-matched\',\'Alarm-triggered\',\'Guarded alarm\']\n        offsets=np.linspace(-.22,.22,len(policies))\n        for j,policy in enumerate(policies):\n            vals=[];los=[];his=[]\n            for cond in conditions:\n                g=a[(a.dataset==ds)&(a.policy==policy)&(a.drift==cond)];x=100*g.RMSE_difference/g.frozen_RMSE\n                rng=np.random.default_rng(seed(ds,policy,cond,\'adapt_ci\'));bs=np.mean(rng.choice(x,(ctx.cfg[\'bootstrap\'],len(x))),axis=1);lo,hi=ci(bs)\n                pt=x.mean();vals.append(pt);los.append(max(0,pt-lo));his.append(max(0,hi-pt));plotrows.append({\'dataset\':ds,\'policy\':policy,\'condition\':cond,\'RMSE_change_percent\':pt,\'low\':lo,\'high\':hi,\'replicates\':len(g)})\n            ax.errorbar(np.arange(len(conditions))+offsets[j],vals,yerr=[los,his],fmt=\'o\',capsize=3,label=policy)\n        ax.axhline(0,color=\'black\',ls=\'--\');ax.set_xticks(range(len(conditions)),conditions,rotation=35,ha=\'right\');ax.set_title(ds);ax.set_ylabel(\'Full-stream RMSE change vs frozen (%)\');ax.grid(alpha=.2)\n    axes[0,1].legend(frameon=False)\n    out.append(savefig(ctx,fig,\'RQ4_F3_non_oracle_adaptation\',\'RQ4. Does deployable adaptation beat frozen and periodic baselines?\',\n       \'Every prediction precedes its outcome. The random policy receives the alarm policy\\\'s refit-attempt budget, separating timing value from compute value. Guarded candidates use trailing observed labels. Negative is better; comparison with periodic updating is also saved numerically.\',pd.DataFrame(plotrows)))\n    r=result[\'operating_results\'];tab=r.groupby([\'dataset\',\'detector\',\'drift\'],as_index=False).agg(enabled=(\'enabled\',\'first\'),pre_FAR=(\'pre_injection_far\',\'mean\'),post_alarm=(\'post_injection_alarm\',\'mean\'),strict_alarm=(\'alarm_without_prior_alarm\',\'mean\'),clean_post=(\'matched_clean_post_alarm\',\'mean\'),delay=(\'delay_conditional\',\'median\'))\n    gate=c[c.provisional_selected][[\'dataset\',\'detector\',\'observed_far_upper95\',\'far_claim_supported\']]\n    tab=tab.merge(gate,on=[\'dataset\',\'detector\'],how=\'left\');tab.dataset=tab.dataset.replace(SHORT_DS)\n    tab.columns=[\'Dataset\',\'Detector\',\'Injected shift\',\'Enabled\',\'Pre alarms/1k\',\'Post alarm rate\',\'No-prior-alarm rate\',\'Matched clean rate\',\'Conditional residual lag\',\'FAR upper 95%\',\'FAR target supported\']\n    out.append(table(ctx,\'RQ4_T1_operating_characteristics\',\'RQ4. Operating characteristics and matched clean control\',tab,\n       \'Rates are averages over simulated corruptions on one historical test period. Post-onset alarm is not proof of causal detection; compare the same-window clean alarm rate. Residual lag counts updates after the first affected residual, not elapsed time from sensor onset; zero means the arrival of that residual. Misses have undefined lag, never zero. Full censoring information is saved separately.\'))\n    return out\n\n\ndef uncertainty_figures(ctx,result,fitted):\n    out=[];s=result[\'summary\'];fig,axes=newaxes();marker={\'Clean\':\'o\',\'MCAR 20%\':\'s\',\'Gaussian 10%\':\'^\',\'Abrupt drift\':\'D\'}\n    col={METHODS[0]:\'#0072B2\',METHODS[1]:\'#D55E00\'}\n    for ax,ds in zip(axes.flat,DATASETS):\n        mod=fitted[ds][\'deployment_model\'];g=s[(s.dataset==ds)&(s.model==mod)]\n        for row in g.itertuples():\n            ax.errorbar(row.width,row.coverage,xerr=[[max(0,row.width-row.width_low)],[max(0,row.width_high-row.width)]],\n                yerr=[[max(0,row.coverage-row.coverage_low)],[max(0,row.coverage_high-row.coverage)]],fmt=marker[row.condition],color=col[row.method],capsize=2)\n        ax.axhline(ctx.cfg[\'coverage\'],color=\'black\',ls=\'--\');ax.set_ylim(max(0,min(ctx.cfg[\'coverage\'],s.coverage_low.min())-.04),1.02);ax.set_title(ds+\' / \'+SHORT[mod]);ax.set_xlabel(\'Mean interval width (\'+SPECS[ds][\'unit\']+\')\');ax.set_ylabel(\'Observed coverage\');ax.grid(alpha=.2)\n    fig.legend([Line2D([0],[0],marker=\'o\',color=col[m]) for m in METHODS],METHODS,loc=\'lower center\',bbox_to_anchor=(.5,-.03),ncol=2,frameon=False)\n    fig.legend([Line2D([0],[0],marker=marker[c],color=\'black\',ls=\'\') for c in CONDITIONS],CONDITIONS,loc=\'lower center\',bbox_to_anchor=(.5,-.075),ncol=4,frameon=False)\n    out.append(savefig(ctx,fig,\'RQ5_F1_coverage_width\',\'RQ5. Empirical coverage versus interval width\',\n       \'Both methods use the identical calibration-B observations after a temporal gap from local-scale fitting. Bands use replicate plus clock-day resampling. Serial dependence and distribution shift preclude an unconditional exchangeability guarantee; the dashed line is nominal, not guaranteed.\',s))\n    d=result[\'curves\'];fig,axes=newaxes();used=[]\n    for ax,ds in zip(axes.flat,DATASETS):\n        mod=fitted[ds][\'deployment_model\'];g=d[(d.dataset==ds)&(d.model==mod)&(d.replicate==0)];used.append(g)\n        for k,cond in enumerate(CONDITIONS):\n            local=g[(g.condition==cond)&(g.method==METHODS[1])&g.kind.str.startswith(\'attainable\')].sort_values(\'accepted_fraction\')\n            line,=ax.plot(local.accepted_fraction,local.selective_RMSE,\'o-\',markersize=3,label=cond)\n            random=g[(g.condition==cond)&(g.method==METHODS[1])&g.kind.str.startswith(\'random\')]\n            ax.plot(random.accepted_fraction,random.selective_RMSE,\'--\',color=line.get_color(),alpha=.5)\n            glob=g[(g.condition==cond)&(g.method==METHODS[0])&g.kind.str.startswith(\'attainable\')]\n            ax.scatter(glob.accepted_fraction,glob.selective_RMSE,marker=\'x\',color=line.get_color(),s=70)\n        ax.set_title(ds+\' / \'+SHORT[mod]);ax.set_xlabel(\'Actually attainable accepted fraction\');ax.set_ylabel(\'Selective RMSE\');ax.grid(alpha=.2)\n    axes[0,1].legend(loc=\'upper left\',bbox_to_anchor=(1.02,1),frameon=False)\n    out.append(savefig(ctx,fig,\'RQ5_F2_tie_safe_risk_coverage\',\'RQ5. Selective prediction without arbitrary tie breaking\',\n       \'Prespecified replica 0. Solid dots: attainable local-width thresholds. Cross at 100%: constant global width has no individual ranking. Dashed lines: sqrt(expected MSE) for random selection. Operational thresholds are fixed on policy data; these curves are descriptive, not test-tuned policies.\',pd.concat(used)))\n    decision=result[\'decision_utility\'];fig,axes=newaxes();used=[]\n    event_q=float(min(decision.event_quantile.unique(),key=lambda x:abs(x-.90)))\n    fn_cost=float(min(decision.false_negative_cost_ratio.unique(),key=lambda x:abs(x-5.0)))\n    markers={\'Point forecast\':\'o\',METHODS[0]:\'s\',METHODS[1]:\'D\'}\n    for ax,ds in zip(axes.flat,DATASETS):\n        mod=fitted[ds][\'deployment_model\'];g=decision[(decision.dataset==ds)&(decision.model==mod)&\n          (decision.event_quantile==event_q)&(decision.false_negative_cost_ratio==fn_cost)&\n          (decision.abstention_cost==.25)]\n        a=g.groupby([\'dataset\',\'model\',\'condition\',\'method\',\'review_capacity\'],as_index=False).agg(cost=(\'expected_cost\',\'mean\'),abstention=(\'abstention_rate\',\'mean\'),\n          cost_low=(\'expected_cost\',lambda x:x.quantile(.025)),cost_high=(\'expected_cost\',lambda x:x.quantile(.975)));used.append(a)\n        a=a[(a.method!=\'Point forecast\') | (a.review_capacity==a.review_capacity.min())]\n        for k,(cond,q) in enumerate(a.groupby(\'condition\')):\n            for row in q.itertuples():\n                ax.errorbar(row.abstention,row.cost,yerr=[[max(0,row.cost-row.cost_low)],[max(0,row.cost_high-row.cost)]],\n                  fmt=markers[row.method],color=plt.cm.tab10(k),capsize=2,label=cond if row.method==\'Point forecast\' else None)\n        ax.set_title(ds+\' / \'+SHORT[mod]);ax.set_xlabel(\'Abstention/review fraction\');ax.set_ylabel(\'Expected decision cost per origin\');ax.grid(alpha=.2)\n    axes[0,1].legend(loc=\'upper left\',bbox_to_anchor=(1.02,1),frameon=False)\n    fig.legend([Line2D([0],[0],marker=markers[m],color=\'black\',ls=\'\',label=m) for m in markers],\n      list(markers),loc=\'lower center\',bbox_to_anchor=(.5,-.04),ncol=3,frameon=False)\n    out.append(savefig(ctx,fig,\'RQ5_F3_environmental_decision_tradeoff\',\'RQ5. Uncertainty changes environmental decision cost and review demand\',\n       f\'High-event thresholds are frozen at the policy {event_q:.0%} quantile. The panel fixes abstention cost at 0.25, varies feasible review capacity, and uses a false-negative cost ratio of {fn_cost:g}. The full cost/capacity grid is saved; these are scenarios, not stakeholder utilities.\',pd.concat(used)))\n    dep={ds:fitted[ds][\'deployment_model\'] for ds in DATASETS}\n    tab=s[s.apply(lambda r:r.model==dep[r.dataset],axis=1)][[\'dataset\',\'model\',\'condition\',\'method\',\'coverage\',\'width\',\'accepted\',\'high_error_given_accepted\']].copy()\n    du=pd.concat(used,ignore_index=True).rename(columns={\'cost\':\'Decision cost\',\'abstention\':\'Review fraction\'})\n    du=du.sort_values(\'review_capacity\').groupby([\'dataset\',\'model\',\'condition\',\'method\'],as_index=False).first()\n    du=du[[\'dataset\',\'model\',\'condition\',\'method\',\'Decision cost\',\'Review fraction\']]\n    tab=tab.merge(du,on=[\'dataset\',\'model\',\'condition\',\'method\'],how=\'left\')\n    tab.dataset=tab.dataset.replace(SHORT_DS);tab.model=tab.model.replace(SHORT);tab.method=tab.method.replace({METHODS[0]:\'Global\',METHODS[1]:\'Local-scale\'})\n    tab.columns=[\'Dataset\',\'Model\',\'Condition\',\'Method\',\'Coverage\',\'Mean width\',\'Acceptance\',\'High error / accepted\',\'Decision cost\',\'Review fraction\']\n    out.append(table(ctx,\'RQ5_T1_uncertainty_decision\',\'RQ5. Calibration, warning, and decision utility are distinct endpoints\',tab,\n       f\'Deployment models only. Decision columns use the policy {event_q:.0%} event threshold and false-negative cost ratio {fn_cost:g}; the complete prespecified sensitivity grid is in RQ5_decision_utility.csv. Costs are scenario assumptions, not inferred stakeholder preferences.\'))\n    return out\n\n\ndef compute_figures(ctx,result):\n    out=[];d=result[\'measurements\'];a=d.groupby([\'dataset\',\'model\',\'stage\'],as_index=False).agg(seconds=(\'seconds_per_operation\',\'mean\'),s_low=(\'seconds_per_operation\',lambda x:x.quantile(.025)),s_high=(\'seconds_per_operation\',lambda x:x.quantile(.975)),energy=(\'energy_kWh_per_operation\',\'mean\'),energy_kind=(\'energy_kind\',\'first\'),RSS=(\'RSS_peak_MB\',\'max\'),RSS_delta=(\'RSS_delta_MB\',\'max\'),repeats=(\'replicate\',\'nunique\'))\n    clean=pd.read_csv(ctx.root/\'results\'/\'P2_clean_summary.csv\').query("split == \'test\'")\n    ref=clean[clean.model==\'Persistence\'][[\'dataset\',\'RMSE\']].rename(columns={\'RMSE\':\'reference_RMSE\'})\n    use=a[a.stage==\'Training\'].merge(clean[[\'dataset\',\'model\',\'RMSE\']],on=[\'dataset\',\'model\']).merge(ref,on=\'dataset\');use[\'nRMSE\']=use.RMSE/use.reference_RMSE\n    fig,axes=newaxes()\n    for ax,ds in zip(axes.flat,DATASETS):\n        q=use[use.dataset==ds].copy();pareto=[]\n        for r in q.itertuples():\n            ax.errorbar(r.seconds,r.nRMSE,xerr=[[max(0,r.seconds-r.s_low)],[max(0,r.s_high-r.seconds)]],fmt=\'o\',color=COLORS[r.model],capsize=3)\n            ax.annotate(SHORT[r.model],(r.seconds,r.nRMSE),xytext=(5,4),textcoords=\'offset points\')\n            dominated=((q.seconds<=r.seconds)&(q.nRMSE<=r.nRMSE)&((q.seconds<r.seconds)|(q.nRMSE<r.nRMSE))).any()\n            if not dominated:pareto.append((r.seconds,r.nRMSE))\n        if pareto:\n            pp=np.array(sorted(pareto));ax.plot(pp[:,0],pp[:,1],\':\',color=\'black\',label=\'Mean-point nondominated set\')\n        ax.axhline(1,color=\'black\',lw=.6);ax.set_xscale(\'log\');ax.set_title(ds);ax.set_xlabel(\'Training seconds per fit\');ax.set_ylabel(\'RMSE / persistence RMSE\');ax.grid(alpha=.2)\n    out.append(savefig(ctx,fig,\'RQ6_F1_accuracy_runtime_frontier\',\'RQ6. Accuracy and measured training cost\',\n      \'Costs use one host, fixed model seeds, excluded warmups and randomized repeat order. Horizontal bars are the empirical 2.5-97.5 percentiles across timing repeats, not cross-hardware confidence intervals. The dotted nondominated set is computed from means only.\',use))\n    fig,axes=newaxes();energy_rows=a[a.stage==\'Training\'].copy()\n    have_energy=np.isfinite(energy_rows.energy).any()\n    for ax,ds in zip(axes.flat,DATASETS):\n        q=energy_rows[energy_rows.dataset==ds].set_index(\'model\').reindex(ML)\n        for j,m in enumerate(ML):\n            r=q.loc[m]\n            if have_energy:\n                if np.isfinite(r.energy):\n                    marker=\'o\' if str(r.energy_kind).startswith(\'counter:\') else \'^\'\n                    ax.scatter(j,r.energy*1000,s=70,marker=marker,color=COLORS[m])\n                else:ax.text(j,.5,\'Unavailable\',ha=\'center\',va=\'center\',rotation=90,transform=ax.get_xaxis_transform(),fontsize=9)\n            else:\n                ax.scatter(j,r.RSS,s=60,marker=\'o\',color=COLORS[m]);ax.annotate(f\'+{r.RSS_delta:.1f} MB\',(j,r.RSS),xytext=(0,8),textcoords=\'offset points\',ha=\'center\',fontsize=8)\n        ax.set_xticks(range(len(ML)),[SHORT[m] for m in ML]);ax.set_title(ds);ax.grid(axis=\'y\',alpha=.2)\n        ax.set_ylabel(\'Energy per fit (Wh)\' if have_energy else \'Measured process peak RSS (MB)\')\n        if not have_energy:ax.set_ylim(bottom=0);ax.margins(y=.15)\n    if have_energy:\n        fig.legend([Line2D([0],[0],marker=\'o\',color=\'black\',ls=\'\'),Line2D([0],[0],marker=\'^\',color=\'black\',ls=\'\')],\n          [\'CPU package counter (host scope)\',\'CodeCarbon estimated/mixed\'],loc=\'lower center\',bbox_to_anchor=(.5,-.03),ncol=2,frameon=False)\n    else:fig.text(.5,-.035,\'Energy unavailable: displaying measured memory; labels show peak increase over initial RSS.\',ha=\'center\',fontsize=9)\n    title=\'RQ6. Energy evidence is limited by exposed instrumentation\' if have_energy else \'RQ6. Memory remains measurable when energy telemetry is unavailable\'\n    out.append(savefig(ctx,fig,\'RQ6_F2_energy_evidence_and_provenance\',title,\n       \'Missing telemetry remains unavailable, never zero or assumed model-specific wattage. CPU package counters are host-scoped; CodeCarbon fallback values are estimates. When all energy readings are unavailable this panel displays measured process peak RSS and increments, not a fabricated energy comparison. Process RSS includes prior allocations.\',energy_rows))\n    fig,axes=newaxes()\n    for ax,ds in zip(axes.flat,DATASETS):\n        for j,stage in enumerate([\'Training\',\'Inference / 1000\']):\n            q=a[(a.dataset==ds)&(a.stage==stage)].set_index(\'model\').reindex(ML)\n            ax.errorbar(np.arange(len(ML))+(j-.5)*.18,q.seconds,yerr=[np.maximum(0,q.seconds-q.s_low),np.maximum(0,q.s_high-q.seconds)],fmt=\'o\',capsize=3,label=stage)\n        ax.set_yscale(\'log\');ax.set_xticks(range(len(ML)),[SHORT[m] for m in ML]);ax.set_title(ds);ax.set_ylabel(\'Seconds (log scale)\');ax.grid(axis=\'y\',alpha=.2)\n    axes[0,1].legend(frameon=False)\n    out.append(savefig(ctx,fig,\'RQ6_F3_training_inference_cost\',\'RQ6. Training and inference are distinct operational costs\',\n       \'Inference is normalized per 1,000 predictions. Time covers computation, whereas energy covers the tracked operation batch and may also include instrumentation overhead. No arbitrary robustness-per-watt composite is formed.\',a))\n    tab=a.copy();tab.dataset=tab.dataset.replace(SHORT_DS);tab.model=tab.model.replace(SHORT);tab.energy_kind=tab.energy_kind.map(lambda x:\'Counter CPU\' if x.startswith(\'counter:\') else \'Unavailable\' if x.startswith(\'unavailable\') else \'Estimated/mixed\')\n    tab=tab[[\'dataset\',\'model\',\'stage\',\'repeats\',\'seconds\',\'energy\',\'energy_kind\',\'RSS\',\'RSS_delta\']]\n    tab.columns=[\'Dataset\',\'Model\',\'Operation\',\'Repeats\',\'Time (s)\',\'Energy (kWh)\',\'Energy provenance\',\'Peak RSS MB\',\'RSS increase MB\']\n    out.append(table(ctx,\'RQ6_T1_compute_and_measurement_scope\',\'RQ6. Measured cost and energy qualification\',tab,\n       \'Peak RSS is process resident memory, not model-only allocation. A zero RSS increment can mean previously allocated memory was reused. Energy estimates are not converted into carbon claims. Detailed operation counts, warnings and hardware records are included.\'))\n    return out\n\n\ndef all_figures(ctx):\n    outputs=[]\n    outputs+=stress_figures(ctx,1,ctx.results[\'RQ1\'],ctx.results[\'P2\'])\n    outputs+=stress_figures(ctx,2,ctx.results[\'RQ2\'],ctx.results[\'P2\'])\n    outputs+=explanation_figures(ctx,ctx.results[\'RQ3\'],ctx.results[\'P2\'])\n    outputs+=drift_figures(ctx,ctx.results[\'RQ4\'])\n    outputs+=uncertainty_figures(ctx,ctx.results[\'RQ5\'],ctx.results[\'P2\'])\n    outputs+=compute_figures(ctx,ctx.results[\'RQ6\'])\n    for rq in range(1,7):assert sum(o[\'stem\'].startswith(\'RQ\'+str(rq)+\'_\') for o in outputs)==4\n    atomic_json(ctx.root/\'manifests\'/\'publication_artifacts.json\',outputs)\n    captions=[]\n    for o in outputs:captions.append(\'## \'+o[\'stem\']+\'\\n\'+o[\'title\']+\'\\n\\n\'+o[\'caption\']+\'\\n\')\n    (ctx.root/\'FIGURE_TABLE_CAPTIONS.md\').write_text(\'\\n\'.join(captions))\n    return outputs\n')


### Bundled source: Checks

In [ ]:
write_module('primna_esd_checks.py','"""Regression tests target the specific audit failures. Fixtures are unit tests only."""\nfrom __future__ import annotations\nimport inspect,json,tempfile\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom primna_esd_runtime import atomic_json,metrics,filehash\nfrom primna_esd_data import features,SPECS,corrupt\nfrom primna_esd_models import MissingAware,factory\nfrom primna_esd_drift import select_feasible,prequential_adapt\nfrom primna_esd_uncertainty import conformal_quantile,canonical_score,interval_metrics,attainable_curve,decision_metrics\n\n\ndef run_tests(ctx,bundles=None):\n    results=[]\n    def check(name,fn):\n        fn();results.append({\'test\':name,\'passed\':True});ctx.log(\'TEST PASS | %s\',name)\n    def constant_width():\n        y=np.array([0.,1.,9.,3.]);p=np.zeros(4);h=np.full(4,2.);s=canonical_score(2*h,1e-9)\n        m=interval_metrics(y,p,h,s,4.,4.)\n        assert m[\'acceptance\']==1 and m[\'high_error_AUROC\']==.5 and m[\'score_constant\']\n        curve=attainable_curve(y,p,s);a=curve[curve.kind.str.startswith(\'attainable\')]\n        assert len(a)==1 and a.accepted_fraction.iloc[0]==1\n    check(\'Constant-width intervals cannot create selective rankings\',constant_width)\n    def permutation_invariance():\n        y=np.arange(6.);p=np.ones(6);s=np.array([1.,1.,2.,2.,3.,3.]);h=s/2\n        a=interval_metrics(y,p,h,s,2.,2.);ii=np.array([3,0,5,1,4,2]);b=interval_metrics(y[ii],p[ii],h[ii],s[ii],2.,2.)\n        for k in a:assert np.isclose(a[k],b[k],equal_nan=True)\n    check(\'Tied-score decisions are row-order invariant\',permutation_invariance)\n    check(\'Finite-sample conformal rank returns infinity when required\',lambda: _assert(np.isinf(conformal_quantile([1,2],.99))))\n    def detector_infeasible():\n        t=pd.DataFrame({\'null_mean_far\':[.2,.3],\'null_far_q95\':[5.,6.],\'observed_policy_far\':[0.,0.],\n          \'observed_far_upper95\':[4.,4.],\n          \'policy_strict_detection\':[1.,.8],\'policy_median_delay\':[1.,2.]})\n        assert not select_feasible(t,1.).selected.any()\n    check(\'Zero alarms do not bypass the rare-FAR uncertainty gate\',detector_infeasible)\n    def no_oracle():assert all(\'onset\' not in n and \'drift\' not in n for n in inspect.signature(prequential_adapt).parameters)\n    check(\'Operational adaptation has no oracle-onset input\',no_oracle)\n    def causal_features():\n        raw=pd.DataFrame({\'CO(GT)\':np.arange(100.),\'T\':np.arange(100.)},index=pd.date_range(\'2020-01-01\',periods=100,freq=\'h\'))\n        a=features(raw,SPECS[\'Air Quality\'],list(raw));raw.iloc[70:]=999.;b=features(raw,SPECS[\'Air Quality\'],list(raw))\n        pd.testing.assert_frame_equal(a.iloc[:70],b.iloc[:70])\n        assert sum(c.startswith(\'lag\') for c in a)==len(raw.columns)*len(SPECS[\'Air Quality\'][\'lags\'])\n        # For target t+1, the same clock hour yesterday is t+1-24, i.e. lag 23 at origin t.\n        assert raw[\'CO(GT)\'].shift(23).iloc[50]==27\n    check(\'Feature construction is nonrecursive and future invariant; seasonal offset\',causal_features)\n    def imputer():\n        t=MissingAware().fit([[1.,np.nan],[3.,np.nan]]);z=t.transform([[np.nan,9.]])\n        assert np.allclose(z,[[2.,9.,1.,0.]])\n    check(\'All-feature missing indicators and training-only medians\',imputer)\n    def denominator():\n        try:metrics([1.,2.],[1.,np.nan])\n        except ValueError:return\n        raise AssertionError(\'Nonfinite scored prediction silently ignored\')\n    check(\'Failed predictions do not shrink the evaluation denominator\',denominator)\n    def decision_policy():\n        y=np.array([0.,10.,0.,10.]);p=np.array([1.,9.,6.,4.]);h=np.array([1.,1.,2.,2.])\n        r=decision_metrics(y,p,h,5.,5.,.25)\n        assert r[\'false_negative_rate\']==0 and r[\'false_positive_rate\']==0 and r[\'abstention_rate\']==.5\n    check(\'Interval decision policy separates alerts, rejection, and abstention\',decision_policy)\n    def capacity_policy():\n        y=np.array([0.,10.,0.,10.]);p=np.array([4.,6.,4.,6.]);h=np.full(4,3.)\n        r=decision_metrics(y,p,h,5.,5.,.25,review_capacity=.25)\n        assert r[\'abstention_rate\']<=.25 and r[\'capacity_binding\']\n    check(\'Decision review demand obeys the registered capacity\',capacity_policy)\n    def adaptation_future():\n        class Tiny:\n            cfg=dict(ctx.cfg,adapt_window=50,adapt_cooldown=20,max_refits=3,min_update_labels=10)\n            def log(self,*args):pass\n        oldidx=pd.date_range(\'2020-01-01\',periods=80,freq=\'h\');testidx=pd.date_range(oldidx[-1]+pd.Timedelta(\'2h\'),periods=100,freq=\'h\')\n        oldX=pd.DataFrame({\'x\':np.sin(np.arange(80)/6)},index=oldidx);oy=pd.Series(2*oldX.x+1,index=oldidx)\n        X=pd.DataFrame({\'x\':np.sin(np.arange(100)/6)},index=testidx);y=(2*X.x+2).to_numpy()\n        est=factory(\'Ridge\',{\'alpha\':1.},1,1).fit(oldX,oy)\n        b={\'dataset\':\'unit_fixture\',\'X\':oldX,\'y\':oy,\'spec\':{\'frequency\':\'1h\'},\'splits\':{\'development\':oldidx}}\n        fit={\'deployment_model\':\'Ridge\',\'models\':{\'Ridge\':est},\'params\':{\'Ridge\':{\'alpha\':1.}}}\n        a=prequential_adapt(Tiny(),b,fit,X,y,\'Periodic\',(0,1));yy=y.copy();yy[70:]+=1000\n        c=prequential_adapt(Tiny(),b,fit,X,yy,\'Periodic\',(0,1));assert np.allclose(a[\'predictions\'][:71],c[\'predictions\'][:71])\n        assert all(pd.Timestamp(e[\'label_available_time\'])<=pd.Timestamp(e[\'first_new_prediction\']) for e in a[\'events\'])\n    check(\'Prequential predictions precede future-label updates\',adaptation_future)\n    if bundles:\n        for ds,b in bundles.items():\n            def actual_data(b=b):\n                assert b[\'raw\'].index.is_monotonic_increasing and b[\'raw\'].index.is_unique\n                pd.testing.assert_series_equal(b[\'y\'],b[\'raw\'][b[\'spec\'][\'target\']].shift(-1))\n                X,N,m=corrupt(b,\'missing\',\'MAR\',.2,0)\n                assert len(X)==len(b[\'splits\'][\'test\']) and m[\'MAR_context_protected\'] is not None\n                assert m[\'scored_rows\']==b[\'y\'].loc[X.index].notna().sum()\n                assert N.index.equals(X.index)\n            check(ds+\': actual labels and full-clock corruption denominator\',actual_data)\n    atomic_json(ctx.root/\'manifests\'/\'regression_tests.json\',results)\n    return results\n\n\ndef _assert(value):\n    if not value:raise AssertionError\n')

### Bundled source: External

In [ ]:
write_module('primna_esd_external.py','"""Independent cross-domain replication on public environmental time series.\n\nThis phase is never labelled prospectively confirmatory by software. Genuine\nconfirmation requires an externally timestamped protocol and author attestation.\n"""\nfrom __future__ import annotations\nimport io, json, urllib.request, zipfile\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nfrom primna_esd_runtime import atomic_json, digest, filehash, seed, metrics, ci\nfrom primna_esd_data import features, corrupt, scenario, cv_folds\nfrom primna_esd_models import factory, grid, fit_logged\nfrom primna_esd_uncertainty import calibrate_intervals, conformal_quantile\nfrom primna_esd_figures import savefig, table, newaxes\n\nEXTERNAL_SPECS={\n \'Beijing Aotizhongxin\':{\n   \'target\':\'CO\',\'frequency\':\'1h\',\'period\':24,\'lags\':[1,2,3,6,12,23,24],\n   \'windows\':[3,6,24],\'unit\':\'ug/m3\',\'filename\':\'PRSA_Data_Aotizhongxin_20130301-20170228.csv\',\n   \'url\':\'https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip\',\n   \'doi\':\'10.24432/C5RK5G\',\'domain\':\'air quality\'},\n \'Household Power\':{\n   \'target\':\'Global_active_power\',\'frequency\':\'10min\',\'period\':144,\'lags\':[1,2,3,6,12,18,36,143,144],\n   \'windows\':[6,18,144],\'unit\':\'kW\',\'filename\':\'household_power_consumption.txt\',\n   \'url\':\'https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip\',\n   \'doi\':\'10.24432/C58K54\',\'domain\':\'energy demand\'}\n}\nEXTERNAL_NAMES=list(EXTERNAL_SPECS)\nEXT_MODELS=[\'Persistence\',\'Seasonal naive\',\'Ridge\',\'HistGradientBoosting\']\nEXT_SHORT={\'Persistence\':\'Persist.\',\'Seasonal naive\':\'Seasonal\',\'Ridge\':\'Ridge\',\'HistGradientBoosting\':\'HGB\'}\nEXT_COLORS={\'Persistence\':\'#666666\',\'Seasonal naive\':\'#A6A6A6\',\'Ridge\':\'#0072B2\',\'HistGradientBoosting\':\'#D55E00\'}\nEXT_CONDITIONS=[\'Clean\',\'MCAR 20%\',\'Block 20%\',\'Gaussian 10%\',\'Abrupt drift\']\n\n\ndef _download_or_find(ctx,name):\n    spec=EXTERNAL_SPECS[name];explicit=ctx.cfg.get(\'external_data_paths\',{}).get(name)\n    if explicit:\n        p=Path(explicit)\n        if not p.exists():raise FileNotFoundError(p)\n        return p\n    for root in [Path(\'/kaggle/input\'),ctx.root/\'data\']:\n        if root.exists():\n            hits=sorted(root.rglob(spec[\'filename\']))\n            if hits:return hits[0]\n    if not ctx.cfg[\'allow_download\']:\n        raise FileNotFoundError(\'Attach \'+spec[\'filename\']+\' as Kaggle input or enable Internet.\')\n    ctx.log(\'Downloading official UCI external-confirmation archive: %s\',spec[\'url\'])\n    archive=ctx.root/\'data\'/(\'external_\'+digest(name)[:10]+\'.zip\')\n    with urllib.request.urlopen(spec[\'url\'],timeout=180) as r:archive.write_bytes(r.read())\n    with zipfile.ZipFile(archive) as outer:\n        direct=[n for n in outer.namelist() if Path(n).name==spec[\'filename\']]\n        if direct:\n            dest=ctx.root/\'data\'/spec[\'filename\'];dest.write_bytes(outer.read(direct[0]));return dest\n        for member in outer.namelist():\n            if member.lower().endswith(\'.zip\'):\n                with zipfile.ZipFile(io.BytesIO(outer.read(member))) as inner:\n                    hits=[n for n in inner.namelist() if Path(n).name==spec[\'filename\']]\n                    if hits:\n                        dest=ctx.root/\'data\'/spec[\'filename\'];dest.write_bytes(inner.read(hits[0]));return dest\n    raise RuntimeError(\'Official archive did not contain \'+spec[\'filename\'])\n\n\ndef _parse(path,name):\n    if name==\'Beijing Aotizhongxin\':\n        d=pd.read_csv(path,na_values=[\'NA\',\'?\'])\n        t=pd.to_datetime(d[[\'year\',\'month\',\'day\',\'hour\']],errors=\'raise\')\n        d=d.drop(columns=[\'No\',\'year\',\'month\',\'day\',\'hour\',\'wd\',\'station\'],errors=\'ignore\')\n        d=d.apply(pd.to_numeric,errors=\'raise\')\n    else:\n        d=pd.read_csv(path,sep=\';\',na_values=[\'?\',\'\'],low_memory=False)\n        t=pd.to_datetime(d.pop(\'Date\').astype(str)+\' \'+d.pop(\'Time\').astype(str),format=\'%d/%m/%Y %H:%M:%S\',errors=\'raise\')\n        d=d.apply(pd.to_numeric,errors=\'coerce\');d.index=pd.DatetimeIndex(t)\n        d=d.resample(\'10min\').mean();t=d.index\n    d.index=pd.DatetimeIndex(t);d=d.sort_index()\n    if d.index.duplicated().any():raise ValueError(name+\': duplicate timestamps\')\n    spec=EXTERNAL_SPECS[name];grid_index=pd.date_range(d.index[0],d.index[-1],freq=spec[\'frequency\'])\n    if not d.index.isin(grid_index).all():raise ValueError(name+\': timestamps do not align with the declared grid\')\n    return d.reindex(grid_index)\n\n\ndef _splits(index,history):\n    n=len(index);cuts=[int(n*x) for x in (.55,.70,.85)];gap=history+1\n    out={\'development\':index[:cuts[0]],\'calibration\':index[cuts[0]+gap:cuts[1]],\n         \'policy\':index[cuts[1]+gap:cuts[2]],\'test\':index[cuts[2]+gap:-1]}\n    if min(map(len,out.values()))<200:raise ValueError(\'External series too short after temporal gaps.\')\n    return out\n\n\ndef _bundle(ctx,name,path):\n    raw=_parse(path,name);spec=dict(EXTERNAL_SPECS[name]);columns=list(raw.columns)\n    X=features(raw,spec,columns);y=raw[spec[\'target\']].shift(-1)\n    naive=pd.DataFrame({\'Persistence\':raw[spec[\'target\']],\n                        \'Seasonal naive\':raw[spec[\'target\']].shift(spec[\'period\']-1)},index=raw.index)\n    splits=_splits(raw.index,max(spec[\'lags\']+spec[\'windows\']))\n    dev=raw.loc[splits[\'development\'],columns];med=dev.median();std=dev.std().replace(0,np.nan)\n    iqr=(dev.quantile(.75)-dev.quantile(.25)).replace(0,np.nan)\n    st=pd.DataFrame({\'median\':med.fillna(0),\'std\':std.fillna(iqr).fillna(1),\'iqr\':iqr.fillna(std).fillna(1)})\n    groups={}\n    for col in X:\n        group=\'Calendar\' if col.startswith(\'time::\') else col.split(\'::\')[-1]\n        groups.setdefault(group,[]).append(col)\n    bundle={\'dataset\':name,\'raw\':raw,\'X\':X,\'y\':y,\'naive\':naive,\'spec\':spec,\'columns\':columns,\n            \'stats\':st,\'groups\':groups,\'splits\':splits,\'input_sha256\':filehash(path),\n            \'all_measurements_missing_rows\':int(raw.isna().all(axis=1).sum())}\n    bundle[\'data_key\']=digest({\'file\':bundle[\'input_sha256\'],\'split_boundaries\':{k:[str(v[0]),str(v[-1])] for k,v in splits.items()},\'features\':list(X)})\n    return bundle\n\n\ndef _fit_external(ctx,b):\n    dev=b[\'splits\'][\'development\'];observed=dev[b[\'y\'].loc[dev].notna()]\n    fit={\'models\':{},\'params\':{},\'cv_scores\':{},\'fallbacks\':{},\'clean\':{}}\n    for name in EXT_MODELS[:2]:\n        v=b[\'naive\'].loc[observed,name].median();fit[\'fallbacks\'][name]=float(v if np.isfinite(v) else b[\'y\'].loc[observed].median())\n    cvrows=[]\n    for name in EXT_MODELS[2:]:\n        configs=grid(name,ctx.cfg[\'profile\']);scores=[]\n        for config_id,params in enumerate(configs):\n            fold_scores=[]\n            for fold,tr,va in cv_folds(b,ctx.cfg[\'external_cv_folds\']):\n                label=f\'External | {b["dataset"]} | {name} | config {config_id+1} | fold {fold+1}\'\n                def work(name=name,params=params,tr=tr,va=va,fold=fold,label=label):\n                    est=factory(name,params,seed(ctx.cfg[\'seed\'],\'external\',b[\'dataset\'],name,fold),ctx.cfg[\'threads\'])\n                    est,ws,secs=fit_logged(ctx,est,b[\'X\'].loc[tr],b[\'y\'].loc[tr],label)\n                    return {\'RMSE\':metrics(b[\'y\'].loc[va],est.predict(b[\'X\'].loc[va]))[\'RMSE\'],\'warnings\':ws,\'seconds\':secs}\n                r=ctx.task(\'EXTERNAL_CV\',label,{\'data\':b[\'data_key\'],\'params\':params,\'fold\':fold},work)\n                fold_scores.append(r[\'RMSE\']);cvrows.append({\'dataset\':b[\'dataset\'],\'model\':name,\'config\':config_id,\'fold\':fold,\'params\':json.dumps(params),**r})\n            scores.append(float(np.mean(fold_scores)))\n        best=int(np.argmin(scores));params=configs[best];fit[\'params\'][name]=params;fit[\'cv_scores\'][name]=scores[best]\n        label=f\'External | {b["dataset"]} | {name} | final fit\'\n        def final(name=name,params=params,label=label):\n            est=factory(name,params,seed(ctx.cfg[\'seed\'],\'external\',b[\'dataset\'],name,\'final\'),ctx.cfg[\'threads\'])\n            est,ws,secs=fit_logged(ctx,est,b[\'X\'].loc[observed],b[\'y\'].loc[observed],label)\n            return est,ws,secs\n        est,ws,secs=ctx.task(\'EXTERNAL_FIT\',label,{\'data\':b[\'data_key\'],\'params\':params},final)\n        fit[\'models\'][name]=est\n    fit[\'deployment_model\']=min(EXT_MODELS[2:],key=lambda n:(fit[\'cv_scores\'][n],n))\n    return fit,pd.DataFrame(cvrows)\n\n\ndef _predict(b,fit,X,naive):\n    out=[]\n    for name in EXT_MODELS:\n        if name in fit[\'models\']:p=fit[\'models\'][name].predict(X)\n        else:p=naive[name].fillna(fit[\'fallbacks\'][name]).to_numpy(float)\n        if not np.isfinite(p).all():raise ValueError(\'Nonfinite external prediction: \'+name)\n        out.append(p)\n    return np.column_stack(out)\n\n\ndef _decision_rows(ctx,b,fit,ca,condition,rep,y,p,halves):\n    pol=b[\'splits\'][\'policy\'];policy_y=b[\'y\'].loc[pol].dropna().to_numpy();rows=[];valid=np.isfinite(y)\n    for event_q in ctx.cfg[\'decision_event_quantiles\']:\n        threshold=float(np.quantile(policy_y,event_q));truth=y>=threshold\n        for fn_ratio in ctx.cfg[\'decision_fn_cost_ratios\']:\n            for method,half in [(\'Point forecast\',None)]+list(halves.items()):\n                if half is None:action=(p>=threshold).astype(int);abstain=np.zeros(len(p),bool)\n                else:\n                    low=p-half;high=p+half;action=np.full(len(p),-1);action[low>=threshold]=1;action[high<threshold]=0;abstain=action<0\n                fp=(action==1)&(~truth)&valid;fn=(action==0)&truth&valid;ab=abstain&valid\n                cost=(fp.sum()+fn_ratio*fn.sum()+ctx.cfg[\'decision_abstain_cost\']*ab.sum())/max(1,valid.sum())\n                rows.append({\'dataset\':b[\'dataset\'],\'condition\':condition,\'replicate\':rep,\'method\':method,\n                  \'event_quantile\':event_q,\'event_threshold\':threshold,\'false_negative_cost_ratio\':fn_ratio,\n                  \'expected_cost\':cost,\'false_negative_rate\':fn.sum()/max(1,(truth&valid).sum()),\n                  \'false_positive_rate\':fp.sum()/max(1,((~truth)&valid).sum()),\'abstention_rate\':ab.sum()/max(1,valid.sum())})\n    return rows\n\n\ndef external_suite(ctx):\n    protocol={\'target_journal\':\'Environment Systems and Decisions\',\'status\':\'independent retrospective replication; not preregistered confirmation\',\n      \'author_declares_outcomes_previously_inspected\':bool(ctx.cfg[\'external_outcomes_previously_inspected\']),\n      \'datasets\':{k:{\'doi\':v[\'doi\'],\'domain\':v[\'domain\'],\'target\':v[\'target\']} for k,v in EXTERNAL_SPECS.items()},\n      \'primary_endpoints\':[\'test RMSE relative to persistence\',\'natural-missing versus complete-origin error\',\'condition-specific RMSE change\',\'empirical interval coverage\',\'decision cost and abstention\'],\n      \'decision_sensitivity\':{\'event_quantiles\':ctx.cfg[\'decision_event_quantiles\'],\'false_negative_cost_ratios\':ctx.cfg[\'decision_fn_cost_ratios\'],\'abstention_cost\':ctx.cfg[\'decision_abstain_cost\']}}\n    atomic_json(ctx.root/\'manifests\'/\'external_confirmation_protocol.json\',protocol)\n    bundles={};fits={};audit=[];cv=[];clean=[];stress=[];intervals=[];decisions=[];natural_missing=[]\n    for name in EXTERNAL_NAMES:\n        path=_download_or_find(ctx,name);b=ctx.task(\'EXTERNAL_DATA\',name+\' external bundle\',{\'sha256\':filehash(path),\'spec\':EXTERNAL_SPECS[name]},lambda name=name,path=path:_bundle(ctx,name,path))\n        bundles[name]=b;fit,cvr=_fit_external(ctx,b);fits[name]=fit;cv.append(cvr)\n        idx=b[\'splits\'][\'test\'];y=b[\'y\'].loc[idx].to_numpy();P=_predict(b,fit,b[\'X\'].loc[idx],b[\'naive\'].loc[idx]);fit[\'clean\'][\'test\']=P\n        for j,m in enumerate(EXT_MODELS):\n            met=metrics(y,P[:,j]);clean.append({\'dataset\':name,\'model\':m,\'deployment_model\':fit[\'deployment_model\'],**met,\'relative_to_persistence\':met[\'RMSE\']/metrics(y,P[:,0])[\'RMSE\']})\n        deploy_clean=P[:,EXT_MODELS.index(fit[\'deployment_model\'])]\n        missing_origin=b[\'raw\'].loc[idx,b[\'columns\']].isna().any(axis=1).to_numpy();observed_y=np.isfinite(y)\n        for state,mask in [(\'Fully observed origin\',~missing_origin),(\'Natural missing origin\',missing_origin)]:\n            use=mask&observed_y\n            estimable=bool(use.sum()>=ctx.cfg[\'natural_missing_min_n\'])\n            natural_missing.append({\'dataset\':name,\'state\':state,\'n\':int(use.sum()),\'estimable\':estimable,\n              \'RMSE\':metrics(y[use],deploy_clean[use])[\'RMSE\'] if estimable else np.nan,\n              \'MAE\':metrics(y[use],deploy_clean[use])[\'MAE\'] if estimable else np.nan})\n        ca=ctx.task(\'EXTERNAL_CAL\',name+\' external interval calibration\',\n          {\'data\':b[\'data_key\'],\'model\':fit[\'deployment_model\'],\'coverage\':ctx.cfg[\'coverage\']},\n          lambda:calibrate_intervals(ctx,b,fit,fit[\'deployment_model\']))\n        for condition in EXT_CONDITIONS:\n            reps=1 if condition==\'Clean\' else ctx.cfg[\'external_reps\']\n            for rep in range(reps):\n                label=f\'External | {name} | {condition} | replicate {rep+1}/{reps}\'\n                def work(condition=condition,rep=rep):\n                    if condition==\'Clean\':X=b[\'X\'].loc[idx];N=b[\'naive\'].loc[idx]\n                    else:X,N,_=scenario(b,condition,rep)\n                    predall=_predict(b,fit,X,N)\n                    scale=np.maximum(ca[\'scale_model\'].predict(X),ca[\'scale_floor\'])\n                    return {\'predictions\':predall,\'scale\':scale}\n                result_rep=ctx.task(\'EXTERNAL_STRESS\',label,\n                  {\'data\':b[\'data_key\'],\'condition\':condition,\'replicate\':rep,\'model\':fit[\'deployment_model\']},work)\n                predall=result_rep[\'predictions\'];scale=result_rep[\'scale\']\n                deploy=predall[:,EXT_MODELS.index(fit[\'deployment_model\'])]\n                halves={\'Global split conformal\':np.full(len(deploy),ca[\'global_q\']),\n                        \'Locally scaled conformal\':ca[\'local_q\']*scale}\n                for j,m in enumerate(EXT_MODELS):\n                    met=metrics(y,predall[:,j]);base=metrics(y,P[:,j])[\'RMSE\']\n                    stress.append({\'dataset\':name,\'condition\':condition,\'replicate\':rep,\'model\':m,**met,\'clean_RMSE\':base,\'relative_change\':met[\'RMSE\']/base-1})\n                for method,half in halves.items():\n                    valid=np.isfinite(y);intervals.append({\'dataset\':name,\'condition\':condition,\'replicate\':rep,\'method\':method,\n                      \'coverage\':float(np.mean(np.abs(y[valid]-deploy[valid])<=half[valid])),\'mean_width\':float(np.mean(2*half[valid]))})\n                decisions.extend(_decision_rows(ctx,b,fit,ca,condition,rep,y,deploy,halves))\n        audit.append({\'dataset\':name,\'doi\':EXTERNAL_SPECS[name][\'doi\'],\'input_sha256\':b[\'input_sha256\'],\'rows\':len(b[\'raw\']),\n          \'development_start\':b[\'splits\'][\'development\'][0],\'test_end\':b[\'splits\'][\'test\'][-1],\'deployment_model\':fit[\'deployment_model\']})\n    result={\'protocol\':protocol,\'data_audit\':pd.DataFrame(audit),\'cv\':pd.concat(cv,ignore_index=True),\'clean\':pd.DataFrame(clean),\n      \'stress\':pd.DataFrame(stress),\'intervals\':pd.DataFrame(intervals),\'decisions\':pd.DataFrame(decisions),\n      \'natural_missing\':pd.DataFrame(natural_missing)}\n    for k,v in result.items():\n        if isinstance(v,pd.DataFrame):ctx.table(\'EXTERNAL_\'+k,v)\n    return result\n\n\ndef external_figures(ctx,result):\n    out=[];clean=result[\'clean\'];stress=result[\'stress\'];intervals=result[\'intervals\'];dec=result[\'decisions\'];natural=result[\'natural_missing\']\n    fig,axes=newaxes()\n    for ax,ds in zip(axes.flat,EXTERNAL_NAMES):\n        g=clean[clean.dataset==ds]\n        ax.bar(np.arange(len(g)),g.relative_to_persistence,color=[EXT_COLORS[m] for m in g.model])\n        ax.axhline(1,color=\'black\',lw=.8);ax.set_xticks(range(len(g)),[EXT_SHORT[m] for m in g.model],rotation=20)\n        ax.set_ylabel(\'Test RMSE / persistence RMSE\');ax.set_title(ds);ax.grid(axis=\'y\',alpha=.2)\n    out.append(savefig(ctx,fig,\'EV_F1_external_clean_skill\',\'External confirmation: clean forecasting skill\',\n      \'Independent UCI cohorts were specified before outcome inspection. Values below one outperform persistence. Model selection uses development-only temporal CV; no core-study test result selects the external model.\',clean))\n    agg=stress.groupby([\'dataset\',\'condition\',\'model\'],as_index=False).agg(change=(\'relative_change\',\'mean\'),low=(\'relative_change\',lambda x:x.quantile(.025)),high=(\'relative_change\',lambda x:x.quantile(.975)))\n    fig,axes=newaxes()\n    for ax,ds in zip(axes.flat,EXTERNAL_NAMES):\n        q=agg[(agg.dataset==ds)&(agg.condition!=\'Clean\')]\n        for m,g in q.groupby(\'model\'):\n            x=np.arange(len(g));ax.errorbar(x,g.change*100,yerr=[np.maximum(0,(g.change-g.low)*100),np.maximum(0,(g.high-g.change)*100)],fmt=\'o-\',capsize=2,label=EXT_SHORT[m],color=EXT_COLORS[m])\n        ax.axhline(0,color=\'black\',lw=.7);ax.set_xticks(range(len(g)),g.condition,rotation=28,ha=\'right\');ax.set_ylabel(\'RMSE change from clean (%)\');ax.set_title(ds);ax.grid(alpha=.2)\n    axes[0,1].legend(frameon=False,bbox_to_anchor=(1.02,1),loc=\'upper left\')\n    out.append(savefig(ctx,fig,\'EV_F2_external_stress_replication\',\'External confirmation: mechanism-specific degradation\',\n      \'Points average registered perturbation replicas; bars are empirical 2.5–97.5 percentiles across replicas. Conditions retain distinct physical/statistical meanings and are not pooled into one universal severity score.\',agg))\n    q=dec[(dec.event_quantile==.9)&(dec.method==\'Locally scaled conformal\')]\n    agg=q.groupby([\'dataset\',\'condition\',\'false_negative_cost_ratio\'],as_index=False).agg(cost=(\'expected_cost\',\'mean\'),abstention=(\'abstention_rate\',\'mean\'))\n    fig,axes=newaxes()\n    for ax,ds in zip(axes.flat,EXTERNAL_NAMES):\n        for condition,g in agg[agg.dataset==ds].groupby(\'condition\'):\n            ax.plot(g.abstention,g.cost,\'o-\',label=condition)\n        ax.set_xlabel(\'Abstention rate\');ax.set_ylabel(\'Expected decision cost / observation\');ax.set_title(ds+\' (90th-percentile event)\');ax.grid(alpha=.2)\n    axes[0,1].legend(frameon=False,bbox_to_anchor=(1.02,1),loc=\'upper left\')\n    out.append(savefig(ctx,fig,\'EV_F3_external_decision_tradeoff\',\'External confirmation: uncertainty-aware decision trade-off\',\n      \'Event thresholds are frozen from policy outcomes. Curves vary the registered false-negative/false-positive cost ratio; abstention cost is fixed in configuration. This is a sensitivity analysis, not a claimed universal social valuation.\',agg))\n    im=intervals.groupby([\'dataset\',\'condition\',\'method\'],as_index=False).agg(coverage=(\'coverage\',\'mean\'),coverage_low=(\'coverage\',lambda x:x.quantile(.025)),coverage_high=(\'coverage\',lambda x:x.quantile(.975)),width=(\'mean_width\',\'mean\'))\n    dm=dec.groupby([\'dataset\',\'condition\',\'method\'],as_index=False).agg(cost=(\'expected_cost\',\'mean\'),abstention=(\'abstention_rate\',\'mean\'))\n    nm=natural.pivot(index=\'dataset\',columns=\'state\',values=[\'n\',\'RMSE\']).reset_index()\n    nm.columns=[\'dataset\']+[f\'{a}: {b}\' for a,b in nm.columns[1:]]\n    if {\'RMSE: Natural missing origin\',\'RMSE: Fully observed origin\'}<=set(nm.columns):\n        nm[\'Natural/complete RMSE ratio\']=nm[\'RMSE: Natural missing origin\']/nm[\'RMSE: Fully observed origin\']\n    tabdf=im.merge(dm,on=[\'dataset\',\'condition\',\'method\'],how=\'left\').merge(nm,on=\'dataset\',how=\'left\')\n    out.append(table(ctx,\'EV_T1_external_confirmation_endpoints\',\'External confirmation endpoints\',tabdf,\n      \'External cohorts are independent public time series, not additional corruption replicas of the core datasets. Coverage is empirical. Decision costs are averaged over the registered sensitivity grid. Natural-missingness columns compare real incomplete sensor origins with fully observed origins and are descriptive, because missingness is not randomized.\'))\n    return out\n')

### Bundled source: Multisite

In [ ]:
write_module('primna_esd_multisite.py','"""Independent multi-site Beijing evaluation for the failure-aware audit.\n\nAll twelve stations are retained.  Local and leave-one-station-out (LOSO)\nmodels use fixed, declared hyperparameters so the held-out station never enters\nmodel or parameter selection.  Stations, rather than synthetic perturbation\nreplicas, are the unit of external generalisation.\n"""\nfrom __future__ import annotations\nimport io, urllib.request, zipfile\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom primna_esd_runtime import digest,filehash,seed,metrics,ci\nfrom primna_esd_data import features,scenario\nfrom primna_esd_models import factory,fit_logged\nfrom primna_esd_figures import savefig,table\n\nURL=\'https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip\'\nDOI=\'10.24432/C5RK5G\'\nTARGET=\'CO\'\nMODELS=[\'Ridge\',\'HistGradientBoosting\']\nPARAMS={\n \'Ridge\':{\'alpha\':10.0},\n \'HistGradientBoosting\':{\'max_iter\':250,\'max_leaf_nodes\':31,\'learning_rate\':.05,\'l2_regularization\':.5}}\nSPEC={\'target\':TARGET,\'frequency\':\'1h\',\'period\':24,\'lags\':[1,2,3,6,12,23,24],\n      \'windows\':[3,6,24],\'unit\':\'ug/m3\'}\nCONDITIONS=[\'MCAR 20%\',\'Block 20%\',\'Gaussian 10%\',\'Abrupt drift\']\nCOLORS={\'Ridge\':\'#0072B2\',\'HistGradientBoosting\':\'#D55E00\'}\n\n\ndef station_paths(ctx):\n    explicit=ctx.cfg.get(\'beijing_multisite_root\')\n    roots=[Path(explicit)] if explicit else [Path(\'/kaggle/input\'),ctx.root/\'data\'/\'beijing_multisite\']\n    for root in roots:\n        if root.exists():\n            hits=sorted(root.rglob(\'PRSA_Data_*_20130301-20170228.csv\'))\n            if len(hits)==12:return hits\n    if not ctx.cfg[\'allow_download\']:\n        raise FileNotFoundError(\'Attach the official UCI Beijing Multi-Site archive or enable Kaggle Internet.\')\n    ctx.log(\'Downloading official UCI Beijing multi-site archive: %s\',URL)\n    target=ctx.root/\'data\'/\'beijing_multisite\';target.mkdir(parents=True,exist_ok=True)\n    with urllib.request.urlopen(URL,timeout=180) as response:payload=response.read()\n    with zipfile.ZipFile(io.BytesIO(payload)) as outer:\n        members=[n for n in outer.namelist() if Path(n).name.startswith(\'PRSA_Data_\') and n.endswith(\'.csv\')]\n        if not members:\n            nested=[n for n in outer.namelist() if n.lower().endswith(\'.zip\')]\n            for n in nested:\n                with zipfile.ZipFile(io.BytesIO(outer.read(n))) as inner:\n                    for m in inner.namelist():\n                        if Path(m).name.startswith(\'PRSA_Data_\') and m.endswith(\'.csv\'):\n                            (target/Path(m).name).write_bytes(inner.read(m))\n        else:\n            for m in members:(target/Path(m).name).write_bytes(outer.read(m))\n    hits=sorted(target.glob(\'PRSA_Data_*_20130301-20170228.csv\'))\n    if len(hits)!=12:raise RuntimeError(f\'Expected 12 Beijing station files, found {len(hits)}.\')\n    return hits\n\n\ndef _bundle(path):\n    station=path.name.split(\'PRSA_Data_\',1)[1].split(\'_2013\',1)[0]\n    d=pd.read_csv(path,na_values=[\'NA\',\'?\'])\n    t=pd.to_datetime(d[[\'year\',\'month\',\'day\',\'hour\']],errors=\'raise\')\n    d=d.drop(columns=[\'No\',\'year\',\'month\',\'day\',\'hour\',\'wd\',\'station\'],errors=\'ignore\').apply(pd.to_numeric,errors=\'raise\')\n    d.index=pd.DatetimeIndex(t);d=d.sort_index()\n    if d.index.duplicated().any():raise ValueError(station+\': duplicate timestamps\')\n    grid=pd.date_range(d.index[0],d.index[-1],freq=\'1h\');d=d.reindex(grid)\n    columns=list(d.columns);X=features(d,SPEC,columns);y=d[TARGET].shift(-1)\n    n=len(d);cuts=[int(n*x) for x in (.60,.80)];gap=max(SPEC[\'lags\']+SPEC[\'windows\'])+1\n    splits={\'development\':d.index[:cuts[0]],\'validation\':d.index[cuts[0]+gap:cuts[1]],\n            \'test\':d.index[cuts[1]+gap:-1]}\n    med=d.loc[splits[\'development\']].median();std=d.loc[splits[\'development\']].std().replace(0,np.nan)\n    iqr=(d.loc[splits[\'development\']].quantile(.75)-d.loc[splits[\'development\']].quantile(.25)).replace(0,np.nan)\n    stats=pd.DataFrame({\'median\':med.fillna(0),\'std\':std.fillna(iqr).fillna(1),\'iqr\':iqr.fillna(std).fillna(1)})\n    groups={}\n    for col in X:groups.setdefault(\'Calendar\' if col.startswith(\'time::\') else col.split(\'::\')[-1],[]).append(col)\n    b={\'dataset\':\'Beijing \'+station,\'station\':station,\'raw\':d,\'X\':X,\'y\':y,\'columns\':columns,\n       \'spec\':SPEC,\'stats\':stats,\'groups\':groups,\'splits\':splits,\'input_sha256\':filehash(path)}\n    b[\'data_key\']=digest({\'file\':b[\'input_sha256\'],\'station\':station,\'split\':{k:[str(v[0]),str(v[-1])] for k,v in splits.items()}})\n    return b\n\n\ndef _observed(b,split):\n    idx=b[\'splits\'][split];return idx[b[\'y\'].loc[idx].notna()]\n\n\ndef _scoreable_test(b,split):\n    """Rows where both the forecast target and persistence baseline are observed."""\n    idx=b[\'splits\'][split]\n    target_ok=b[\'y\'].loc[idx].notna()\n    persistence_ok=b[\'raw\'].loc[idx,TARGET].notna()\n    return idx[target_ok & persistence_ok]\n\n\ndef _fit(ctx,name,X,y,label,tag):\n    est=factory(name,PARAMS[name],seed(ctx.cfg[\'seed\'],\'Beijing\',name,tag),ctx.cfg[\'threads\'])\n    est,ws,secs=fit_logged(ctx,est,X,y,label);return est,ws,secs\n\n\ndef run_multisite(ctx):\n    paths=station_paths(ctx)\n    bundles={}\n    for path in paths:\n        raw_hash=filehash(path)\n        b=ctx.task(\'BEIJING_DATA\',\'Beijing \'+path.stem,{\'sha256\':raw_hash,\'spec\':SPEC},lambda path=path:_bundle(path))\n        bundles[b[\'station\']]=b\n    stations=sorted(bundles);fits={};audit=[];rows=[];stress=[]\n    for holdout in stations:\n        b=bundles[holdout]\n        target_observed=_observed(b,\'test\')\n        test=_scoreable_test(b,\'test\')\n        if len(test)==0:\n            raise RuntimeError(f\'{holdout}: no test rows have both an observed target and persistence baseline.\')\n        yt=b[\'y\'].loc[test].to_numpy()\n        persist=b[\'raw\'].loc[test,TARGET].to_numpy()\n        if not np.all(np.isfinite(yt)):\n            raise RuntimeError(f\'{holdout}: nonfinite test targets after scoreability filtering.\')\n        if not np.all(np.isfinite(persist)):\n            raise RuntimeError(f\'{holdout}: nonfinite persistence values after scoreability filtering.\')\n        excluded=len(target_observed)-len(test)\n        ctx.log(\'Beijing %s | target-observed=%d | persistence-scoreable=%d | excluded-missing-persistence=%d\',\n                holdout,len(target_observed),len(test),excluded)\n        base=metrics(yt,persist)[\'RMSE\']\n        if not np.isfinite(base) or base<=0:\n            raise RuntimeError(f\'{holdout}: invalid persistence RMSE {base!r}.\')\n        for design in [\'Local\',\'LOSO\']:\n            if design==\'Local\':\n                tr=_observed(b,\'development\');X=b[\'X\'].loc[tr];y=b[\'y\'].loc[tr]\n            else:\n                xs=[];ys=[]\n                for s in stations:\n                    if s==holdout:continue\n                    q=bundles[s];tr=_observed(q,\'development\');xs.append(q[\'X\'].loc[tr]);ys.append(q[\'y\'].loc[tr])\n                X=pd.concat(xs);y=pd.concat(ys)\n                # Equal-site cap prevents the largest station or memory limits from dominating.\n                max_n=ctx.cfg[\'multisite_train_cap\'];take=np.linspace(0,len(X)-1,min(max_n,len(X)),dtype=int);X=X.iloc[take];y=y.iloc[take]\n            for name in MODELS:\n                label=f\'Beijing {holdout} | {design} | {name}\'\n                est,ws,secs=ctx.task(\'BEIJING_FIT\',label,{\'holdout\':holdout,\'design\':design,\'model\':name,\n                  \'inputs\':{s:bundles[s][\'data_key\'] for s in stations},\'params\':PARAMS[name]},\n                  lambda name=name,X=X,y=y,label=label,design=design:_fit(ctx,name,X,y,label,holdout+design))\n                fits[(holdout,design,name)]=est;pred=est.predict(b[\'X\'].loc[test]);m=metrics(yt,pred)\n                rows.append({\'station\':holdout,\'design\':design,\'model\':name,\'n_test\':m[\'n\'],\'RMSE\':m[\'RMSE\'],\n                  \'MAE\':m[\'MAE\'],\'R2\':m[\'R2\'],\'AE_q95\':m[\'AE_q95\'],\'persistence_RMSE\':base,\n                  \'RMSE_ratio_to_persistence\':m[\'RMSE\']/base,\'fit_seconds\':secs,\'warnings\':\' | \'.join(ws)})\n        for name in MODELS:\n            est=fits[(holdout,\'LOSO\',name)];clean=metrics(yt,est.predict(b[\'X\'].loc[test]))[\'RMSE\']\n            for condition in CONDITIONS:\n                for rep in range(ctx.cfg[\'multisite_stress_reps\']):\n                    Xp,_,_=scenario(b,condition,rep,scope=\'test\');pred=est.predict(Xp.loc[test]);m=metrics(yt,pred)\n                    stress.append({\'station\':holdout,\'model\':name,\'condition\':condition,\'replicate\':rep,\n                      \'RMSE\':m[\'RMSE\'],\'relative_change\':m[\'RMSE\']/clean-1})\n        audit.append({\'station\':holdout,\'doi\':DOI,\'input_sha256\':b[\'input_sha256\'],\'rows\':len(b[\'raw\']),\n          \'test_start\':test[0],\'test_end\':test[-1],\n          \'test_observed_labels\':len(target_observed),\'test_scoreable_labels\':len(test),\n          \'test_excluded_missing_persistence\':excluded})\n    clean=pd.DataFrame(rows);stress=pd.DataFrame(stress);audit=pd.DataFrame(audit)\n    # The site bootstrap is descriptive uncertainty across the twelve observed\n    # stations; it is not treated as a random sample of all cities.\n    summary=[];rng=np.random.default_rng(seed(ctx.cfg[\'seed\'],\'site bootstrap\'))\n    for (design,name),g in clean.groupby([\'design\',\'model\']):\n        x=g.RMSE_ratio_to_persistence.to_numpy();draw=np.array([rng.choice(x,len(x),replace=True).mean() for _ in range(ctx.cfg[\'bootstrap\'])])\n        summary.append({\'design\':design,\'model\':name,\'stations\':len(x),\'mean_ratio\':x.mean(),\n          \'median_ratio\':np.median(x),\'fraction_better_than_persistence\':np.mean(x<1),\n          \'site_boot_low\':np.quantile(draw,.025),\'site_boot_high\':np.quantile(draw,.975),\n          \'worst_station_ratio\':x.max()})\n    summary=pd.DataFrame(summary)\n    for name,df in [(\'BEIJING_multisite_audit\',audit),(\'BEIJING_multisite_clean\',clean),\n                    (\'BEIJING_multisite_stress\',stress),(\'BEIJING_multisite_summary\',summary)]:ctx.table(name,df)\n    return {\'audit\':audit,\'clean\':clean,\'stress\':stress,\'summary\':summary}\n\n\ndef multisite_figures(ctx,result):\n    out=[];clean=result[\'clean\'];stress=result[\'stress\'];summary=result[\'summary\']\n    fig,axes=plt.subplots(1,2,figsize=(10.2,4.8),constrained_layout=True)\n    for ax,design in zip(axes,[\'Local\',\'LOSO\']):\n        q=clean[clean.design==design]\n        for name,g in q.groupby(\'model\'):\n            g=g.sort_values(\'station\');ax.plot(g.station,g.RMSE_ratio_to_persistence,\'o-\',label=name,color=COLORS[name])\n        ax.axhline(1,color=\'black\',ls=\'--\');ax.set_title(design+\' training\');ax.tick_params(axis=\'x\',rotation=55)\n        ax.set_ylabel(\'RMSE / persistence RMSE\');ax.grid(alpha=.2)\n    axes[1].legend(frameon=False)\n    out.append(savefig(ctx,fig,\'EV_F1_multisite_skill\',\'External replication across all 12 Beijing stations\',\n      \'Every point is a genuinely held-out station test result. LOSO models are trained on the other eleven stations with fixed hyperparameters. Values below one beat persistence; failures are retained.\',clean))\n    agg=stress.groupby([\'model\',\'condition\',\'station\'],as_index=False).relative_change.mean()\n    fig,axes=plt.subplots(1,2,figsize=(10.2,4.8),constrained_layout=True)\n    for ax,(name,g) in zip(axes,agg.groupby(\'model\')):\n        groups=[g[g.condition==c].relative_change.to_numpy()*100 for c in CONDITIONS]\n        ax.boxplot(groups,tick_labels=CONDITIONS,showfliers=True);ax.axhline(0,color=\'black\',lw=.8)\n        ax.set_title(name+\' LOSO\');ax.set_ylabel(\'RMSE change from clean (%)\');ax.tick_params(axis=\'x\',rotation=30);ax.grid(axis=\'y\',alpha=.2)\n    out.append(savefig(ctx,fig,\'EV_F2_multisite_stress\',\'Between-station distribution of degradation\',\n      \'Boxes summarize the twelve station-level mean effects; corruption replicates are averaged within station before cross-station display. Stations, not simulated replicas, are the generalisation unit.\',agg))\n    pivot=clean.pivot_table(index=[\'station\',\'model\'],columns=\'design\',values=\'RMSE_ratio_to_persistence\').reset_index()\n    pivot[\'LOSO_minus_local\']=pivot.LOSO-pivot.Local\n    fig,ax=plt.subplots(figsize=(10.2,4.8),constrained_layout=True)\n    for name,g in pivot.groupby(\'model\'):\n        g=g.sort_values(\'LOSO_minus_local\');ax.plot(g.LOSO_minus_local,g.station,\'o\',label=name,color=COLORS[name])\n    ax.axvline(0,color=\'black\',ls=\'--\');ax.set_xlabel(\'LOSO minus local RMSE ratio\');ax.set_ylabel(\'Held-out station\');ax.grid(alpha=.2);ax.legend(frameon=False)\n    out.append(savefig(ctx,fig,\'EV_F3_transfer_penalty\',\'Cross-site transfer penalty is heterogeneous\',\n      \'Positive values mean that training on other stations transfers worse than station-local training. This isolates transportability rather than mixing stations with corruption replicas.\',pivot))\n    tabdf=clean.merge(result[\'audit\'][[\'station\',\'test_observed_labels\',\'test_scoreable_labels\',\'test_excluded_missing_persistence\']],on=\'station\').sort_values([\'station\',\'design\',\'model\'])\n    out.append(table(ctx,\'EV_T1_multisite_endpoints\',\'External multi-site replication endpoints\',tabdf,\n      \'All twelve stations are shown. Hyperparameters are fixed before held-out-station testing. Model and persistence metrics use the same explicitly reported scoreable rows; targets lacking a persistence value are excluded only from this paired comparison and counted in the audit. Ratios below one beat persistence. The station bootstrap in BEIJING_multisite_summary.csv is descriptive and does not imply that Beijing stations represent all cities.\'))\n    return out\n')


### Bundled source: Driver

In [ ]:
write_module('primna_esd_driver.py','"""Explicit phases, verified caches, artifact checks, and final ZIP packaging."""\nfrom __future__ import annotations\nimport importlib.metadata\nfrom pathlib import Path\nimport pandas as pd\n\nfrom primna_esd_runtime import Context,atomic_json,filehash,digest\nfrom primna_esd_data import acquire\nfrom primna_esd_models import train\nfrom primna_esd_stress import stress_suite,explanation_suite\nfrom primna_esd_drift import drift_suite\nfrom primna_esd_uncertainty import uncertainty_suite\nfrom primna_esd_compute import compute_suite\nfrom primna_esd_figures import all_figures,stress_figures,explanation_figures,drift_figures,uncertainty_figures,compute_figures\nfrom primna_esd_checks import run_tests\nfrom primna_esd_external import external_suite\nfrom primna_esd_multisite import run_multisite,multisite_figures\n\nPHASES={1:\'P1\',2:\'P2\',3:\'RQ1\',4:\'RQ2\',5:\'RQ3\',6:\'RQ4\',7:\'RQ5\',8:\'RQ6\',9:\'EXTERNAL\'}\n\n\ndef initialize(config,source_dir):\n    if config[\'profile\'] not in {\'paper\',\'pilot\'}:raise ValueError(\'Use paper or explicitly labelled pilot profile.\')\n    if not config.get(\'acknowledge_retrospective\',False):\n        raise ValueError(\'Acknowledge that the two core historical test periods were previously inspected.\')\n    if config[\'profile\']==\'paper\' and not config.get(\'run_external_confirmation\',False):\n        raise ValueError(\'Paper profile requires independent external confirmation; use pilot only for engineering tests.\')\n    ctx=Context(config,source_dir)\n    required=[\'numpy\',\'pandas\',\'scipy\',\'scikit-learn\',\'matplotlib\',\'joblib\',\'psutil\',\'threadpoolctl\']\n    lines=[]\n    for package in required:\n        try:lines.append(package+\'==\'+importlib.metadata.version(package))\n        except importlib.metadata.PackageNotFoundError:pass\n    (ctx.root/\'source\'/\'requirements_minimal.txt\').write_text(\'\\n\'.join(lines)+\'\\n\')\n    return ctx\n\n\ndef stage_work(ctx,number):\n    if number==1:\n        out=acquire(ctx);run_tests(ctx,out);return out\n    if number==2:return train(ctx,ctx.results[\'P1\'])\n    b=ctx.results[\'P1\'];f=ctx.results[\'P2\']\n    if number in [3,4]:return stress_suite(ctx,b,f,number-2)\n    if number==5:return explanation_suite(ctx,b,f)\n    if number==6:return drift_suite(ctx,b,f)\n    if number==7:return uncertainty_suite(ctx,b,f)\n    if number==8:return compute_suite(ctx,b,f)\n    if number==9:return {\'cross_domain\':external_suite(ctx),\'multisite\':run_multisite(ctx)}\n    raise KeyError(number)\n\n\ndef _phase_figures(ctx,number,key):\n    if number in [3,4]:return stress_figures(ctx,number-2,ctx.results[key],ctx.results[\'P2\'])\n    if number==5:return explanation_figures(ctx,ctx.results[key],ctx.results[\'P2\'])\n    if number==6:return drift_figures(ctx,ctx.results[key])\n    if number==7:return uncertainty_figures(ctx,ctx.results[key],ctx.results[\'P2\'])\n    if number==8:return compute_figures(ctx,ctx.results[key])\n    if number==9:return multisite_figures(ctx,ctx.results[key][\'multisite\'])\n    return []\n\n\ndef run_phase(ctx,number):\n    if number>ctx.cfg[\'end_phase\']:\n        ctx.log(\'Phase %s not requested (END_PHASE=%s)\',number,ctx.cfg[\'end_phase\']);return\n    if number==10:return finalize(ctx)\n    if number<ctx.cfg.get(\'start_phase\',1):ctx.log(\'Loading prerequisite phase %d; missing compatible tasks will recompute.\',number)\n    key=PHASES[number];ctx.log(\'========== PHASE %d: %s ==========\',number,key)\n    if number in [1,9]:\n        ctx.results[key]=stage_work(ctx,number);ctx.phase_status[str(number)]=\'complete\'\n    else:\n        identities={ds:b[\'data_key\'] for ds,b in ctx.results[\'P1\'].items()}\n        if number==8:\n            try:host=Path(\'/proc/sys/kernel/random/boot_id\').read_text().strip()\n            except OSError:host=\'unknown host\'\n            identities[\'host_boot_fingerprint\']=digest(host)\n        def compute():\n            with ctx.substage(\'phase scientific computation\',phase=number,key=key):\n                result=stage_work(ctx,number)\n            artifacts=[]\n            ctx.log(\'ARTIFACT HASH START | phase=%d | key=%s\',number,key)\n            for p in (ctx.root/\'results\').rglob(\'*\'):\n                if p.is_file():\n                    rel=str(p.relative_to(ctx.root))\n                    if p.name.startswith(key+\'_\') or key+\'_\' in str(p.relative_to(ctx.root/\'results\')):\n                        artifacts.append({\'path\':rel,\'sha256\':filehash(p)})\n            ctx.log(\'ARTIFACT HASH COMPLETE | phase=%d | files=%d\',number,len(artifacts))\n            return {\'result\':result,\'artifacts\':artifacts}\n        try:\n            payload=ctx.task(\'PHASE\'+str(number),key+\' phase result\',identities,compute)\n            ctx.log(\'ARTIFACT VALIDATION START | phase=%d | files=%d\',number,len(payload[\'artifacts\']))\n            invalid=[a for a in payload[\'artifacts\'] if not (ctx.root/a[\'path\']).exists() or filehash(ctx.root/a[\'path\'])!=a[\'sha256\']]\n            if invalid:\n                ctx.log(\'ARTIFACT VALIDATION REBUILD | phase=%d | invalid=%s\',number,[a[\'path\'] for a in invalid])\n                with ctx.substage(\'phase artifact reconstruction\',phase=number,key=key,invalid=len(invalid)):\n                    ctx.results[key]=stage_work(ctx,number)\n            else:ctx.results[key]=payload[\'result\']\n            ctx.log(\'ARTIFACT VALIDATION COMPLETE | phase=%d | invalid=%d\',number,len(invalid))\n            ctx.phase_status[str(number)]=\'complete\'\n        except BaseException:\n            ctx.phase_status[str(number)]=\'failed_or_interrupted\';ctx.failure(\'phase \'+str(number));raise\n    atomic_json(ctx.root/\'manifests\'/\'phase_status.json\',ctx.phase_status)\n    try:\n        with ctx.substage(\'phase figures and tables\',phase=number,key=key), ctx.heartbeat(\'phase figures \'+str(number)):\n            arts=_phase_figures(ctx,number,key)\n        if arts:atomic_json(ctx.root/\'manifests\'/(key+\'_artifacts.json\'),arts)\n    except BaseException:\n        ctx.phase_status[str(number)]=\'numerics_complete_figure_failed\';ctx.failure(\'phase figures \'+str(number));raise\n    ctx.log(\'PHASE %d COMPLETE. Safe restart: Run All; valid tasks will LOAD.\',number)\n    if ctx.cfg.get(\'snapshot_each_phase\',True):ctx.snapshot(False)\n    return ctx.results[key]\n\n\ndef finalize(ctx):\n    missing=[PHASES[i] for i in range(1,10) if PHASES[i] not in ctx.results]\n    if missing:raise RuntimeError(\'Cannot finalize; required phase results not loaded: \'+\', \'.join(missing))\n    tests=run_tests(ctx,ctx.results[\'P1\']);artifacts=all_figures(ctx)\n    if len(artifacts)!=24:raise AssertionError(\'Expected exactly four distinct publication artifacts per RQ.\')\n    ext=multisite_figures(ctx,ctx.results[\'EXTERNAL\'][\'multisite\']);artifacts+=ext\n    if len(ext)!=4:raise AssertionError(\'Expected three external-confirmation figures and one table.\')\n    atomic_json(ctx.root/\'manifests\'/\'publication_artifacts.json\',artifacts)\n    (ctx.root/\'FIGURE_TABLE_CAPTIONS.md\').write_text(\'\\n\'.join(\n      \'## \'+a[\'stem\']+\'\\n\'+a[\'title\']+\'\\n\\n\'+a[\'caption\']+\'\\n\' for a in artifacts))\n    try:import fitz\n    except ImportError as exc:raise RuntimeError(\'PyMuPDF is required for final PDF rendering validation.\') from exc\n    pdf_checks=[]\n    for a in artifacts:\n        p=ctx.root/a[\'pdf\'];data=ctx.root/a[\'plot_data\']\n        assert p.read_bytes().startswith(b\'%PDF\') and p.stat().st_size>1000\n        assert data.exists() and len(pd.read_csv(data))>0\n        doc=fitz.open(p);pages=len(doc);fonts=sum(len(pg.get_fonts()) for pg in doc);doc.close()\n        if pages<1 or fonts<1:raise AssertionError(\'PDF renderer/font validation failed: \'+str(p))\n        pdf_checks.append({\'path\':a[\'pdf\'],\'bytes\':p.stat().st_size,\'plot_data\':a[\'plot_data\'],\'pages\':pages,\'font_records\':fonts})\n    assert ctx.results[\'RQ5\'][\'parameters\'].query("method == \'Global split conformal\'").constant_score.all()\n    pars=ctx.results[\'RQ4\'][\'calibration\']\n    far_supported=bool(pars[pars.provisional_selected].far_claim_supported.all()) if pars.provisional_selected.any() else False\n    ext_summary=ctx.results[\'EXTERNAL\'][\'multisite\'][\'summary\']\n    cross_site_supported=bool((ext_summary.query("design == \'LOSO\'").fraction_better_than_persistence>=ctx.cfg[\'cross_site_success_fraction\']).any())\n    atomic_json(ctx.root/\'manifests\'/\'final_QA.json\',{\n      \'regression_tests_passed\':len(tests),\'principal_artifacts\':pdf_checks,\'retrospective_core\':True,\n      \'external_evaluation_status\':\'independent retrospective replication; not preregistered confirmation\',\n      \'completed_profile\':ctx.cfg[\'profile\'],\'technical_package_complete\':True,\n      \'evidence_gates\':{\'rare_false_alarm_claim_supported\':far_supported,\n                        \'cross_site_replication_gate_supported\':cross_site_supported},\n      \'publication_ready_claim\':\'not determined by software; requires author and peer review\',\n      \'energy_status\':sorted(set(ctx.results[\'RQ6\'][\'measurements\'].energy_kind))})\n    readme=("# Failure-aware environmental forecasting audit (ESD v4)\\n\\n"\n      "Target journal: Environment Systems and Decisions (Springer Nature).\\n\\n"\n      "All cohorts are reported as retrospective or independent external replication. The software never upgrades a user declaration into confirmatory evidence. All twelve Beijing stations are evaluated, including leave-one-station-out transfer.\\n\\n"\n      "All fitted preprocessing uses development data. Six research questions are evaluated, each with three vector PDF figures and one publication table. External confirmation adds three figures and one table. Numeric plot sources, CSV/LaTeX tables, captions, source, configuration, checksums, and resumable checkpoints are included.\\n\\n"\n      "Detector feasibility requires both observed-policy FAR and the 95th percentile of dependent-null FAR to meet the target. Guarded alarm adaptation validates every candidate refit on already observed pre-update labels. Failed or disabled detectors remain visible. Coverage is empirical rather than an unconditional exchangeability guarantee.\\n\\n"\n      "RQ5 reports an environmental monitoring decision sensitivity analysis using frozen policy thresholds, false-negative/false-positive cost ratios, and an explicit abstention cost. These costs are transparent scenarios, not claimed stakeholder utilities.\\n\\n"\n      "Energy telemetry may be unavailable or estimated; no carbon claim is generated. Passing software QA establishes reproducibility and technical completeness, not journal acceptance or universal environmental validity.\\n\\n"\n      "Checkpoints are matched to source, scientific configuration, package versions, inputs and task parameters. The final dissemination ZIP contains exactly the payloads listed in its checksum manifest; resumable checkpoints remain in partial phase archives.\\n")\n    (ctx.root/\'README_RESULTS.md\').write_text(readme)\n    ctx.phase_status[\'10\']=\'complete\'\n    return ctx.snapshot(True)\n')


## Initialize or restore the audited run

In [ ]:
# Find the renamed checkpoint archive; its ZIP contents are unchanged.
if RESUME_FROM == 'AUTO_KEEP':
    keep_archives=[]
    for candidate in sorted(Path('/kaggle/input').rglob('*')):
        if candidate.is_file() and candidate.suffix.lower()=='.keep' and zipfile.is_zipfile(candidate):
            with zipfile.ZipFile(candidate) as z:
                if any(Path(name).parts[-2:]==('manifests','run_identity.json') for name in z.namelist()):
                    keep_archives.append(candidate)
    if len(keep_archives)!=1:
        raise RuntimeError('Expected exactly one checkpoint .keep archive under /kaggle/input; found '
                           +str(len(keep_archives))+'. Set RESUME_FROM to its exact file path. Candidates: '
                           +str([str(p) for p in keep_archives]))
    RESUME_FROM=str(keep_archives[0])
CFG['resume_root']=str(RESUME_FROM) if RESUME_FROM else None
if RESUME_FROM:
    print('Resume input:',RESUME_FROM)
    if not Path(RESUME_FROM).exists():
        raise FileNotFoundError('Resume path does not exist: '+str(RESUME_FROM))
if RESUME_FROM and Path(RESUME_FROM).is_file():
    if not zipfile.is_zipfile(RESUME_FROM):
        raise RuntimeError('Resume file is not a valid ZIP archive: '+str(RESUME_FROM))
    archive=Path(RESUME_FROM);h=hashlib.sha256()
    with archive.open('rb') as stream:
        for chunk in iter(lambda:stream.read(2**20),b''):h.update(chunk)
    destination=WORK_ROOT/('restored_esd_v4_'+h.hexdigest()[:12]);destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            target=(destination/item.filename).resolve()
            if not target.is_relative_to(destination.resolve()):raise RuntimeError('Unsafe archive member')
            if ((item.external_attr>>16)&0o170000)==0o120000:raise RuntimeError('Symlink archive members are rejected')
        z.extractall(destination)
    CFG['resume_root']=str(destination)
# Pick one run folder when the archive (or a folder) holds several copies.
if CFG['resume_root']:
    _root=Path(CFG['resume_root'])
    _runs=[p.parent.parent for p in _root.rglob('run_identity.json')
           if p.parent.name=='manifests']
    for _r in _runs:
        _n=len(list((_r/'checkpoints').glob('*.joblib')))
        _ok=(_r/'manifests'/'output_checksums.csv').exists()
        _key=json.loads((_r/'manifests'/'run_identity.json').read_text()).get('key','')[:12]
        print(f'  candidate: {_r} | key {_key} | checkpoints {_n} | checksum manifest {_ok}')
    _usable=[r for r in _runs if (r/'manifests'/'output_checksums.csv').exists()]
    if len(_usable)>1:
        _best=max(_usable,key=lambda r:(len(list((r/'checkpoints').glob('*.joblib'))),
                                        (r/'manifests'/'output_checksums.csv').stat().st_mtime))
        print('Multiple run copies found; resuming from:',_best)
        CFG['resume_root']=str(_best)
for name in list(sys.modules):
    if name.startswith('primna_esd_'):del sys.modules[name]
if str(CODE_DIR) not in sys.path:sys.path.insert(0,str(CODE_DIR))
importlib.invalidate_caches()
from primna_esd_driver import initialize,run_phase
CTX=initialize(CFG,CODE_DIR)
print('Run identity:',CTX.key)
print('Execution log:',CTX.root/'logs'/'execution.log')

In [ ]:
from IPython.display import display,FileLink,IFrame
import pandas as pd
def show_phase(prefix):
    files=sorted((CTX.root/'figures_pdf').glob(prefix+'*.pdf'))
    tables=sorted((CTX.root/'tables').glob(prefix+'*.csv'))
    if files:display(IFrame(src=str(files[0]),width='100%',height=520))
    if tables:
        d=pd.read_csv(tables[0]);display(d.head(20));print('Preview only; complete CSV/PDF/LaTeX is saved.')

## Phase 1 — Inputs, causal features, and regression tests

In [ ]:
print("ENTER PHASE 1",flush=True)
try:
    with threadpool_limits(limits=CFG['threads']):
        result_1=run_phase(CTX,1)
except BaseException:
    CTX.failure("notebook phase")
    raise
print("EXIT PHASE 1",flush=True)


## Phase 2 — Temporal model selection and clean baselines

In [ ]:
print("ENTER PHASE 2",flush=True)
try:
    with threadpool_limits(limits=CFG['threads']):
        result_2=run_phase(CTX,2)
except BaseException:
    CTX.failure("notebook phase")
    raise
print("EXIT PHASE 2",flush=True)


## Phase 3 — RQ1 — Missing-data robustness

In [ ]:
print("ENTER PHASE 3",flush=True)
try:
    with threadpool_limits(limits=CFG['threads']):
        result_3=run_phase(CTX,3)
except BaseException:
    CTX.failure("notebook phase")
    raise
show_phase('RQ1_')
print("EXIT PHASE 3",flush=True)


## Phase 4 — RQ2 — Sensor-fault robustness

In [ ]:
import faulthandler
_phase4_trace=open(WORK_ROOT/'phase4_stacks.log','a',buffering=1)
faulthandler.dump_traceback_later(120,repeat=True,file=_phase4_trace)
try:
    print("ENTER PHASE 4",flush=True)
    try:
        with threadpool_limits(limits=CFG['threads']):
            result_4=run_phase(CTX,4)
    except BaseException:
        CTX.failure("notebook phase")
        raise
    show_phase('RQ2_')
    print("EXIT PHASE 4",flush=True)
finally:
    faulthandler.cancel_dump_traceback_later()
    _phase4_trace.close()


## Phase 5 — RQ3 — Explanation degradation beyond control noise

In [ ]:
print("ENTER PHASE 5",flush=True)
try:
    with threadpool_limits(limits=CFG['threads']):
        result_5=run_phase(CTX,5)
except BaseException:
    CTX.failure("notebook phase")
    raise
show_phase('RQ3_')
print("EXIT PHASE 5",flush=True)


## Phase 6 — RQ4 — Rare-alarm uncertainty and budget-matched adaptation

In [ ]:
print("ENTER PHASE 6",flush=True)
try:
    with threadpool_limits(limits=CFG['threads']):
        result_6=run_phase(CTX,6)
except BaseException:
    CTX.failure("notebook phase")
    raise
show_phase('RQ4_')
print("EXIT PHASE 6",flush=True)


## Phase 7 — RQ5 — Uncertainty, review capacity, and decision sensitivity

In [ ]:
print("ENTER PHASE 7",flush=True)
try:
    with threadpool_limits(limits=CFG['threads']):
        result_7=run_phase(CTX,7)
except BaseException:
    CTX.failure("notebook phase")
    raise
show_phase('RQ5_')
print("EXIT PHASE 7",flush=True)


## Phase 8 — RQ6 — Computational burden and qualified energy evidence

In [ ]:
print("ENTER PHASE 8",flush=True)
try:
    with threadpool_limits(limits=CFG['threads']):
        result_8=run_phase(CTX,8)
except BaseException:
    CTX.failure("notebook phase")
    raise
show_phase('RQ6_')
print("EXIT PHASE 8",flush=True)


## Phase 9 — Independent cross-domain and 12-station Beijing replication

In [ ]:
print("ENTER PHASE 9",flush=True)
try:
    with threadpool_limits(limits=CFG['threads']):
        result_9=run_phase(CTX,9)
except BaseException:
    CTX.failure("notebook phase")
    raise
show_phase('EV_')
print("EXIT PHASE 9",flush=True)


## Phase 10 — Final QA, evidence gates, exact manifest, and dissemination ZIP

In [ ]:
print("ENTER PHASE 10",flush=True)
try:
    with threadpool_limits(limits=CFG['threads']):
        result_10=run_phase(CTX,10)
except BaseException:
    CTX.failure("notebook phase")
    raise
print("EXIT PHASE 10",flush=True)


## Download the final package

In [ ]:
# Run at the end of every staged session: writes the partial ZIP you must download.
if END_PHASE<10:
    CTX.snapshot(False)
else:
    print('END_PHASE=10: final ZIP is created by Phase 10.')

In [ ]:
if END_PHASE==10 and CTX.phase_status.get('10')=='complete':
    archive=WORK_ROOT/'Environment_Systems_and_Decisions_Failure_Aware_Audit_v4_Outputs.zip'
    print('RUN COMPLETE:',archive)
    print('Accelerator used: None (CPU only)')
    display(FileLink(str(archive)))
else:
    partial=WORK_ROOT/'Environment_Systems_and_Decisions_Failure_Aware_Audit_v4_Partial.zip'
    print('Staged run. Preserve and download:',partial)
    if partial.exists():display(FileLink(str(partial)))